Differential expression analysis of Bulk RNA-seq data of LMC cells


Date last updated : January 23,2026

Created by : Sayane Shome

https://www.cores.emory.edu/eicc/_includes/documents/sections/resources/RNAseq_Methodology.html

The workflow for the DESEQ2 analysis has been adapted from the tutorial :
https://rstudio-pubs-static.s3.amazonaws.com/329027_593046fb6d7a427da6b2c538caf601e1.html#example-3-two-conditions-two-genotypes-with-an-interaction-term


Bulk RNA-seq Differential Gene Expression Analysis
In differential gene expression analysis, the basic underlying task is the analysis of count data from RNA-seq experiments for the detection of differentially expressed genes between two sample groups. Count data is provided as a table containing the number of sequence fragments that have been uniquely assigned to each gene. For more information about the methodology and algorithms used to generate the count table, please see the Methods section of the EICC website.

For general bulk RNA-seq analyses, we prefer the use of DESeq2 for differential gene expression results. The original DESeq2 publication, by Anders and Huber, introduces DESeq2 and outlines its principle concepts. For a detailed guide on the installation and implementation of DESeq2, please refer to the DESeq2 Vignette provided by the authors of the package.

The Basics of DESeq2 Analysis
DESeq2 requires count data obtained from RNA-seq or another high-throughput sequencing process. The count matrix is a matrix of integer values where typically each row i is a unique gene and each column j is the number of uniquely assigned reads. DESeq2 internally corrects for the appropriate library size, and it is not recommended that the original count matrix be transformed or normalized prior to being given as input.

Pre-filtering Low Count Genes
Pre-filtering the count matrix for differential expression analysis is typically performed to reduce the memory size requirements of the analysis and to increase the speed of the transformation and testing results. Here at the EICC, our computational power is such that we do not need to perform pre-filtering to increase analysis speed. Strict filtering is performed within the standard DESeq2 pipeline to increase power and is automatically applied on the normalized counts prior to differential expression analysis.

Normalization within DESeq2
DESeq2 assumes that the count matrix follows a negative binomial distribution (also known as the gamma-Poisson distribution). A mean is computed proportionally to the concentration of cDNA fragments from the genes in a given sample and then scaled by a normalization factor. For most applications, the same normalization factor can be used for all the genes in a given sample which accounts for the differences in sequencing depth between samples [1]. Gene specific normalization can be used to account for sources of technical bias, but these methods are not implemented by default.

P-values and Adjusted p-values (q-values)
To test the hypothesis that a gene is differentially expressed between one or more groups with statistical significance, DESeq2 implements the Wald Test by default. In the case of differential gene expression, the Wald Test uses the precision of the log fold change value (LFC) as a weight to compute a test statistic based on the distance between the unrestricted LFC estimate and its hypothetical value under the null hypothesis. The null hypothesis for this statistical test is that there is no difference between the two groups being tested (e.g., case and control). Each gene is tested individually and a single p-value is produced as a result.

For every statistical test, there is an acceptable false positive rate in the determination of a significant difference (usually α=
 0.05). Due to the extremely high dimension of RNA-seq data and therefore large number of tests, a multiple comparison adjustment is recommended when determining statistical significance. By default, DESeq2 implements the Benjamini-Hochberg False Discovery Rate (FDR) multiple testing correction. The FDR is the expected ratio of the number of false positives over the total number of significant differences found across all tests being corrected for. The FDR correction retains high power at the expense of a slight increased allowance of false positives, and is therefore the most commonly used multiple hypothesis corrections for exploratory studies.

Family-wise Error Rate (FWER) Corrections
The class of Family-wise Error Rate (FWER) test corrections are more conservative than the FDR corrections typically applied in exploratory analysis results. If the goal of the analysis is to directly confirm the difference in expression between one or more groups for a specific gene or set of genes, the application of a FWER in publication is preferred. The FWER corrections typically have reduced power and a more strict control over false positive rates. We do not provide this multiple test correction as a default due to its highly conservative nature, but FWER can be applied to your analysis results upon request.

Effect Size Estimation
We perform effect size estimation using the R package apeglm. This package provides empirical Bayes shrinkage estimators which can be used to compute s-values (Stephens, 2017). In short, these s-values provide a confidence level in the direction (or sign) of the log base 2 fold-change value. It is recommended that the threshold be much lower for significance when using these outcomes (i.e., 0.005 instead of 0.05 for significance). We provide these values by default, but leave it up to the researchers to decide to include them in publication.

We also use the apeglm package to provide lfcShrink log2 fold-change values. The empirical Bayes shrinkage estimators (Zhu, Ibrahim, Love) use heavy-tailed prior distributions to remove noise and to help prevent extremely large differences that may appear due to technical artifacts. Often, when one sample group has an over-abundance of zeros this can lead to highly inflated fold changes that often give very high significant differences between samples. We use lfcShrink to help determine which log2 fold-change values are more in line with what we would expect biologically, and not what may often be due to technical artifacts from sample processing in the lab.

"Here we have two genotypes, wild-type (wt) and mutant (mut). Two conditions: control (Control) and treated (LPS treated). We are interested in the responses of both wild-type and mutant to treatment. We are also interested in the differences in response between genotypes, which is captured by the interaction term in linear models."

The order of the samples is WT +/- LPS and then Sp3 KO +/- LPS.

Update: Based on Lance's suggestion (Mar 21, 2024), entire images, data frame, the order should be WT -/+ LPS and KO -/+ LPS.

setwd('/Users/sshome/Desktop/Bioinformatics_projects/Bulk_RNA_seqdata_sp3/cluster_data/')


In [ ]:
version #to make sure which R version and computer architecture the code was run



In [ ]:
#Loading the required packages
library(EnsDb.Mmusculus.v79)
library(DESeq2)
library(EnhancedVolcano)
library(tidyr)
library(ashr)
library(tidyverse)
library(knitr)
library(Glimma)
library(DESeq2)
library(pheatmap)
library(ggplot2)
library(dplyr)
library(plotly)
library(VennDiagram)
library(clusterProfiler)
library(enrichplot)
library(ggplot2)
library(ggridges)
library(org.Mm.eg.db)
library(RColorBrewer)

Loading packages,Reading count matrices

In [ ]:
#loading count matrix from processing data.
s <- read.csv('LMC_previouscounts_minushet.csv')

In [ ]:
s4 <- s

In [ ]:
s4

Complete sets of LPS + Ctrl

468-4

685-2

697-3

697-1

697-5

697-2

685-1

468-6

In [ ]:
colnames(s4)[1] <- "Gene_Name"
colnames(s4)[2] <- "468.4_LPS" 
colnames(s4)[3] <- "685.2_LPS"
colnames(s4)[4] <- "697.3_LPS"
colnames(s4)[5] <- "697.1_LPS"
colnames(s4)[6] <- "697.5_LPS"
colnames(s4)[7] <- "697.2_LPS"
colnames(s4)[8] <- "685.1_LPS"
colnames(s4)[9] <- "468.6_LPS"
colnames(s4)[10] <- "685.2_CTRL"
colnames(s4)[11] <- "697.3_CTRL"
colnames(s4)[12] <- "697.1_CTRL"
colnames(s4)[13] <- "468.4_CTRL"
colnames(s4)[14] <- "685.1_CTRL"
colnames(s4)[15] <- "697.5_CTRL"
colnames(s4)[16] <- "697.2_CTRL"
colnames(s4)[17] <- "468.6_CTRL"


s4 <- s4[, c("Gene_Name","468.4_LPS" ,"685.2_LPS","697.3_LPS","697.1_LPS" ,"697.5_LPS" ,"697.2_LPS" ,"685.1_LPS" ,"468.6_LPS","685.2_CTRL","697.3_CTRL","697.1_CTRL","468.4_CTRL","685.1_CTRL","697.5_CTRL","697.2_CTRL","468.6_CTRL")]

In [ ]:
#reading metadata about the count matrix and assigning names to the columns.
metadata <- read.csv('coldata_minus_het_lmc_allsamples.csv',header = FALSE)
colnames(metadata)[1] <- "id"
colnames(metadata)[2] <- "genotype"
colnames(metadata)[3] <- "condition"

In [ ]:
colnames(s4)

In [ ]:
#Displaying the information about the samples.
metadata



In [ ]:
#rearranging the rows based on requirements ; (wt,control),(wt,LPS),(mut,control),(mut,LPS)
new_order <- c(9,10,11,12,1,2,3,4,13,14,15,16,5,6,7,8)
metadata <- metadata[new_order, ]

In [ ]:
new_order1 <- c(1,10,11,12,13,2,3,4,5,14,15,16,17,6,7,8,9)
s4 <- s4[,new_order1]

In [ ]:
# Proceessing the data and removing genes with low counts
zeros_count <- apply(s4 == 0, 1, sum)
filtered_df <- s4[zeros_count <= 4, ]
s4 <- filtered_df 
s4[,2:17] <- s4[,2:17]+1 
s4[,2:17] <- round(s4[,2:17])

In [ ]:
#To adhere to the same names as DESEQ2,storing in new dataframes countDat and metaData, respectively.
metaData <- metadata
countData <- s4

In [ ]:
countData

In [ ]:
write.csv(countData$Gene_Name,'Gene_LMC_v1.csv')

In [ ]:
countData$Gene_Name

In [ ]:
write.csv(countData,'Countdata_lmc_july1_2025.csv')

In [ ]:
#printing PCA plots of the raw counts.
pca_result <- prcomp(t(countData[,2:17]), scale. = TRUE)

# Convert PCA results to a DataFrame for ggplot2
pca_df <- as.data.frame(pca_result$x)
pca_df$condition <- metaData$condition  # adding condition information to PCA DataFrame
pca_df$genotype <- metaData$genotype  # adding genotype information to PCA DataFrame

# Create a ggplot2 plot of the first two principal components
pca_plot <- ggplot(pca_df, aes(x = PC1, y = PC2, color = condition, shape = genotype)) +
  geom_point(size = 4) +
  labs(title = "PCA Plot",
       x = "PC1",
       y = "PC2") +
  theme_minimal()

# Show the PCA plot
print(pca_plot)

In [ ]:
# plotMA(deseq2Results) #include this

Carrying out DE analysis using DESeq2


Next part of code should help answer this :


I.The effect of treatment in wild-type.[This is for WT, treated compared with untreated.]

1.Genes upregulated with LPS in wild-type samples

2.Genes downregulated with LPS in wild-type samples



II.The effect of treatment in mutant[This is for knockout,treated compared with untreated ]   


3.Genes upregulated with LPS in Sp3 knockout samples

4.Genes downregulated with LPS in Sp3 knockout samples



III.What is the difference between mutant and wild-type in control treatments ?[mutant ctrl compared with wild type ctrl]

5.Genes upregulated in wild-type samples in control treatment/Genes downregulated in wild-type samples in control treatment

6.Genes downregulated in wild-type samples in control treatment/Genes downregulated in wild-type samples in control treatment



IV.With LPS treatment, what is the difference between mutant and wild-type?[mutant LPS compared with wild type LPS]

8.Genes upregulated in mutant samples compared to wildtype with LPS treatment

9.Genes downregulated in mutant samples compared to wildtype with LPS treatment




In [ ]:
# Assuming numeric columns 2 to n contain expression data
countData_resolved <- countData %>%
  group_by(Gene_Name) %>%
  summarise(across(where(is.numeric), mean))

In [ ]:
countData1 <- as.data.frame(countData_resolved)
countData1[,2:17] <- round(countData1[,2:17])

In [ ]:
#reading count matrix into DESEq2 object
dds <- DESeqDataSetFromMatrix(countData1,colData=metaData,design=~ genotype + condition + genotype:condition, tidy = TRUE)

In [ ]:
dds <- DESeq(dds)

In [ ]:
rlog_data <- rlog(dds)

pca_data <- plotPCA(rlog_data, intgroup=c("condition", "genotype"), returnData=TRUE)
percentVar <- round(100 * attr(pca_data, "percentVar"))

# Generate the PCA plot with condition and genotype
ggplot(pca_data, aes(x = PC1, y = PC2, color = condition, shape = genotype)) +
    geom_point(size = 3) +
    geom_text_repel(aes(label = row.names(pca_data))) +
    xlab(paste0("PC1: ", percentVar[1], "% variance")) +
    ylab(paste0("PC2: ", percentVar[2], "% variance")) +
    ggtitle("PCA of RNA-seq samples") +
    theme_bw() +
    scale_shape_manual(values = c(15, 17, 19)) # You may want to adjust the shapes or colors to your liking



In [ ]:
library(DESeq2)
library(ggplot2)
library(ggrepel) # Ensure you have ggrepel installed for geom_text_repel

# Assuming rlog_data and dds are already defined and correctly prepared
rlog_data <- rlog(dds)

pca_data <- plotPCA(rlog_data, intgroup=c("condition", "genotype"), returnData=TRUE)
percentVar <- round(100 * attr(pca_data, "percentVar"))

# Generate the PCA plot without sample names
ggplot(pca_data, aes(x = PC1, y = PC2, color = condition, shape = genotype)) +
    geom_point(size = 3) +
    xlab(paste0("PC1: ", percentVar[1], "% variance")) +
    ylab(paste0("PC2: ", percentVar[2], "% variance")) +
    ggtitle("PCA of RNA-seq samples") +
    theme_bw() +
    scale_shape_manual(values = c(15, 17, 19)) # Adjust shapes or colors as needed


In [ ]:
pca_data_onlyControl <- pca_data %>%
  filter(condition == "Control")

In [ ]:
ggplot(pca_data_onlyControl, aes(x = PC1, y = PC2, color = genotype)) +
    geom_point(size = 6, stroke = 1, shape = 16) + # Shape set to 16 for all points
    xlab(paste0("PC1: ", percentVar[1], "% variance")) +
    ylab(paste0("PC2: ", percentVar[2], "% variance")) +
    ggtitle("PCA of RNA-seq samples (Control condition)") +
    theme_bw() +
    scale_color_manual(values = c("wt" = "#96cb01", "mut" = "#fe9388")) # Adjust as per your genotypes

In [ ]:
library(ggplot2)

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc_Feb16_2026"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Create the plot
p <- ggplot(pca_data_onlyControl, aes(x = PC1, y = PC2, color = genotype)) +
  geom_point(size = 6, stroke = 1, shape = 16) +
  xlab(paste0("PC1: ", percentVar[1], "% variance")) +
  ylab(paste0("PC2: ", percentVar[2], "% variance")) +
  ggtitle("PCA of RNA-seq samples (Control condition)") +
  theme_bw() +
  scale_color_manual(values = c("wt" = "#96cb01", "mut" = "#fe9388"))

# Save as SVG
ggsave(
  filename = file.path(out_dir, "PCA_RNAseq_Control.svg"),
  plot = p,
  device = "svg",
  width = 7,
  height = 6,
  units = "in"
)


In [ ]:
dds$condition <- factor(dds$condition, levels = c("LPS","Control"))
dds$genotype <- factor(dds$genotype, levels = c("wt","mut"))

dds$condition <- relevel(dds$condition, ref = "Control")
dds$genotype <- relevel(dds$genotype, ref = "wt")
#making sure by above command that wild type and Control is considered as reference respectively

In [ ]:
colData(dds)

In [ ]:
#design(dds) <- ~ genotype + condition + genotype:condition
dds <- DESeq(dds) 
resultsNames(dds)

In [ ]:
glimmaMDS(dds,html = "MDS_plot_LMC_minushet_2020countsdata.html")

In [ ]:
groups <- c("wt_Control", "wt_Control", "wt_Control","wt_Control","wt_LPS","wt_LPS","wt_LPS","wt_LPS","mut_Control","mut_Control","mut_Control","mut_Control", "mut_LPS", "mut_LPS","mut_LPS", "mut_LPS")

In [ ]:
normalized_counts <- counts(dds, normalized=TRUE)
write.csv(normalized_counts,"LMC_Deseq2_countsdata_normalized.csv")

In [ ]:
manual_annotation <- data.frame(
  Genotype = factor(c("wt", "wt", "wt","wt","wt", "wt","wt","wt","mut","mut","mut","mut","mut","mut", "mut","mut")),
  Condition = factor(c("Control", "Control", "Control", "Control","LPS", "LPS", "LPS","LPS","Control","Control", "Control","Control","LPS","LPS","LPS","LPS"))
)

#annotation_col = as.data.frame(manual_annotation)

ann_colors <- list(
  Genotype = c("wt" = "#fe9388", "mut" = "#9bca1e"),
  Condition = c("Control" = "#05d8e1", "LPS" = "#e298fe")
)


# Ensure row names of annotations match the sample names, if needed
rownames(manual_annotation) <- colnames(normalized_counts)


In [ ]:
row_means_norm_counts <- apply(normalized_counts, 1, mean)

ordered_indices_norm_counts <- order(-row_means_norm_counts)
ordered_norm_counts <- normalized_counts[ordered_indices_norm_counts, ]
rows_to_keep <- !grepl("^(Gm|Mir|ENSMUS|Rpl.*|.*Rik|mt-|Malat1|Rn45s)", rownames(ordered_norm_counts))

# Filter the dataframe to remove undesired rows
ordered_norm_counts <- ordered_norm_counts[rows_to_keep, ]
color_palette <- colorRampPalette(c("blue", "white", "red"))(101) # 101 colors for -3 to 3 scale
breaks <- seq(-3, 3, length.out = length(color_palette))

# Step 3: Plot the heatmap with ordered rows of normalized counts
pheatmap(ordered_norm_counts[1:49,],
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         scale = "row", 
         show_rownames = TRUE, 
         show_colnames = FALSE,
         annotation_col = manual_annotation,
         breaks = breaks,
         annotation_colors = ann_colors,color = color_palette) # Adjust color gradient as needed

In [ ]:
library(pheatmap)
library(grid)

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Compute row means and order
row_means_norm_counts <- rowMeans(normalized_counts, na.rm = TRUE)

ordered_indices_norm_counts <- order(-row_means_norm_counts)
ordered_norm_counts <- normalized_counts[ordered_indices_norm_counts, ]

# Filter out undesired gene patterns
rows_to_keep <- !grepl("^(Gm|Mir|ENSMUS|Rpl.*|.*Rik|mt-|Malat1|Rn45s)", rownames(ordered_norm_counts))
ordered_norm_counts <- ordered_norm_counts[rows_to_keep, , drop = FALSE]

# Colors + breaks for -3..3
color_palette <- colorRampPalette(c("blue", "white", "red"))(101)
breaks <- seq(-3, 3, length.out = length(color_palette))

# Build heatmap (capture object!)
ph <- pheatmap(
  ordered_norm_counts[1:49, , drop = FALSE],
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  scale = "row",
  show_rownames = TRUE,
  show_colnames = FALSE,
  annotation_col = manual_annotation,
  annotation_colors = ann_colors,
  breaks = breaks,
  color = color_palette
)

# Save to SVG
svg_file <- file.path(out_dir, "heatmap_top49_ordered_norm_counts.svg")
svg(svg_file, width = 10, height = 8)  # adjust if you want bigger labels
grid.newpage()
grid.draw(ph$gtable)
dev.off()


In [ ]:
pheatmap(ordered_norm_counts[1:49,],
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         scale = "none", 
         show_rownames = TRUE, 
         show_colnames = FALSE,
         annotation_col = manual_annotation,
         annotation_colors = ann_colors,# Adjust based on your preference
         color = colorRampPalette(c("blue", "white", "red"))(255)) # Adjust color gradient as needed

In [ ]:
library(pheatmap)
library(grid)

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc_Feb16_2026"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Generate and capture the heatmap
ph <- pheatmap(
  ordered_norm_counts[1:49, , drop = FALSE],
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  scale = "none",
  show_rownames = TRUE,
  show_colnames = FALSE,
  annotation_col = manual_annotation,
  annotation_colors = ann_colors,
  color = colorRampPalette(c("blue", "white", "red"))(255)
)

# Save as SVG
svg(file.path(out_dir, "heatmap_top49_noScale.svg"),
    width = 10, height = 8)

grid.newpage()
grid.draw(ph$gtable)
dev.off()


In [ ]:
write.csv(rownames(ordered_norm_counts[1:50,]),"LMC_top50_meanvalues_genenames.csv")

In [ ]:
MEF_top50genes_bycounts <- read.csv('MEF_top50_meanvalues_genenames.csv')
MEF_top50genes_bycounts$X <- NULL
colnames(MEF_top50genes_bycounts)[1] <- "Gene_name"

In [ ]:
MEF_genes_data <- ordered_norm_counts[rownames(ordered_norm_counts) %in% MEF_top50genes_bycounts$Gene_name, ]
# Reorder MEF_genes_data to match the selected_genes order
# Make sure all selected_genes are in the rownames of MEF_genes_data
MEF_genes_data_ordered <- MEF_genes_data[MEF_top50genes_bycounts$Gene_name, ]
MEF_genes_data_ordered <- MEF_genes_data_ordered[-nrow(MEF_genes_data_ordered), ]


In [ ]:
pheatmap(MEF_genes_data_ordered,
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         scale = "none", 
         show_rownames = TRUE, 
         show_colnames = FALSE,
         annotation_col = manual_annotation,
         annotation_colors = ann_colors,# Adjust based on your preference
         color = colorRampPalette(c("blue", "white", "red"))(255)) # Adjust color gradient as needed

In [ ]:
library(pheatmap)
library(grid)

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Generate and capture the heatmap
ph <- pheatmap(
  MEF_genes_data_ordered,
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  scale = "none",
  show_rownames = TRUE,
  show_colnames = FALSE,
  annotation_col = manual_annotation,
  annotation_colors = ann_colors,
  color = colorRampPalette(c("blue", "white", "red"))(255)
)

# Save as SVG
svg(
  file.path(out_dir, "heatmap_MEF_genes_noScale.svg"),
  width = 10,
  height = 8
)

grid.newpage()
grid.draw(ph$gtable)
dev.off()


In [ ]:
color_palette <- colorRampPalette(c("blue", "white", "red"))(101) # 101 colors for -3 to 3 scale
breaks <- seq(-3, 3, length.out = length(color_palette))

# Plot the heatmap with the specified breaks and color scale
pheatmap(MEF_genes_data_ordered,
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         scale = "row", 
         show_rownames = TRUE, 
         show_colnames = FALSE,
         annotation_col = manual_annotation,
         annotation_colors = ann_colors,
         breaks = breaks,
         color = color_palette)

In [ ]:
library(pheatmap)
library(grid)

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Define colors and breaks
color_palette <- colorRampPalette(c("blue", "white", "red"))(101)
breaks <- seq(-3, 3, length.out = length(color_palette))

# Generate and capture the heatmap
ph <- pheatmap(
  MEF_genes_data_ordered,
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  scale = "row",
  show_rownames = TRUE,
  show_colnames = FALSE,
  annotation_col = manual_annotation,
  annotation_colors = ann_colors,
  breaks = breaks,
  color = color_palette
)

# Save as SVG
svg(
  file.path(out_dir, "heatmap_MEF_genes_rowScaled_fixedBreaks.svg"),
  width = 10,
  height = 8
)

grid.newpage()
grid.draw(ph$gtable)
dev.off()


In [ ]:
normalized_counts_1 <- data.frame(normalized_counts)

In [ ]:
normalized_counts_2 <- data.frame(rownames(normalized_counts_1),
                                  normalized_counts_1$X468.4_CTRL - normalized_counts_1$X468.4_LPS,
                                  normalized_counts_1$X468.6_CTRL - normalized_counts_1$X468.6_LPS,
                                  normalized_counts_1$X685.1_CTRL - normalized_counts_1$X685.1_LPS,
                                  normalized_counts_1$X685.2_CTRL - normalized_counts_1$X685.2_LPS,
                                  normalized_counts_1$X697.1_CTRL - normalized_counts_1$X697.1_LPS,
                                  normalized_counts_1$X697.2_CTRL - normalized_counts_1$X697.2_LPS,
                                  normalized_counts_1$X697.3_CTRL - normalized_counts_1$X697.3_LPS,
                                  normalized_counts_1$X697.5_CTRL - normalized_counts_1$X697.5_LPS)

In [ ]:
colnames(normalized_counts_2)[1] <- "Gene_name"
colnames(normalized_counts_2)[2] <- "468.4"
colnames(normalized_counts_2)[3] <- "468.6"
colnames(normalized_counts_2)[4] <- "685.1"
colnames(normalized_counts_2)[5] <- "685.2"
colnames(normalized_counts_2)[6] <- "697.1"
colnames(normalized_counts_2)[7] <- "697.2"
colnames(normalized_counts_2)[8] <- "697.3"
colnames(normalized_counts_2)[9] <- "697.5"


In [ ]:
write.csv(normalized_counts_2,'LMC_normalized_counts_differences.csv')

In [ ]:
#Plotting Glimma/Scatter plots for each gene seperately
countData2 <- countData1
rownames(countData2) <- toupper(countData2$Gene_Name)
countData2$Gene_Name <- NULL
gene_expression <- countData2["MEF2C",]  # Extracting expression levels for particular gene

plot_data <- data.frame(
  Sample = colnames(countData2),  
  Expression = as.numeric(gene_expression),  
  Group = groups  
)

gene_name <- "MEF2C"

In [ ]:
# Create Genotype and Condition columns based on the Group column
plot_data$Genotype <- ifelse(grepl("wt", plot_data$Group), "WT", "MUT")
plot_data$Condition <- ifelse(grepl("Control", plot_data$Group), "Control", "LPS")

# Create a new column for the combined labels
plot_data$Group_Combined <- with(plot_data, paste(Genotype, Condition, sep = "+"))

# Define the colors
genotype_colors <- c("WT" = "#fe9388", "MUT" = "#9bca1e")
condition_colors <- c("Control" = "#add8e6", "LPS" = "#e298fe")  # Lighter blue for Control

# Define the colors
genotype_colors <- c("WT" = "#EC3A91", "MUT" = "#71980E")  # Adjusted colors
condition_colors <- c("Control" = "#B0DBFF", "LPS" = "#CF98E2")  # Lightened inside colors

# Create the plot
ggplot(plot_data, aes(x = Group_Combined, y = Expression)) +
  geom_point(aes(fill = Condition, color = Genotype, shape = Genotype), size = 9, stroke = 2) +  # Use shape 21 for filled points with outline
  scale_fill_manual(name = "Condition", values = condition_colors, guide = guide_legend(override.aes = list(shape = 21, size = 9, stroke = 0))) +  # Inner circle colors and remove outline in legend
  scale_color_manual(name = "Genotype", values = genotype_colors, labels = c("WT" = "Wild-type", "MUT" = "Mutant")) +  # Outer circle colors
  scale_shape_manual(name = "Genotype", values = c("WT" = 21, "MUT" = 22), labels = c("WT" = "Wild-type", "MUT" = "Mutant")) +  # Shape 21 for circles, 22 for squares
  scale_x_discrete(limits = c("WT+Control", "WT+LPS", "MUT+Control", "MUT+LPS")) +  # Set the order of the x-axis labels
  labs(
    title = paste("Expression of", gene_name, "Across Groups"),
    x = "",
    y = "Expression Level"
  ) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(size = 14, color = "black"),  # Increase x-axis label size
    axis.text.y = element_text(size = 14, color = "black"),  # Increase y-axis label size
    axis.title.x = element_text(size = 16),  # Increase x-axis title size
    axis.title.y = element_text(size = 16),  # Increase y-axis title size
    axis.ticks = element_line(color = "black"),
    axis.line = element_line(color = "black"),
    panel.background = element_blank(),
    panel.border = element_rect(color = "black", fill = NA),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    legend.position = "right",
    legend.text = element_text(size = 12),  # Increase legend text size
    legend.title = element_text(size = 14),  # Increase legend title size
    legend.key = element_blank()  # Remove legend key outline
  )

In [ ]:
# -----------------------------
# Save SVG scatter plots for MEF2C and MEF2A (filled by Condition, outline by Genotype)
# Assumes you already have:
#   - countData1  (data.frame with a column named "Gene_Name" + sample columns)
#   - groups      (vector length == ncol(countData1)-1 OR == ncol(countData2))
# -----------------------------

suppressPackageStartupMessages({
  library(ggplot2)
})

# Ensure svglite is available for clean SVG output
if (!requireNamespace("svglite", quietly = TRUE)) {
  install.packages("svglite", repos = "https://cloud.r-project.org")
}

# ---- Prep count matrix with Gene_Name as rownames ----
stopifnot("Gene_Name" %in% colnames(countData1))

countData2 <- countData1
rownames(countData2) <- toupper(as.character(countData2$Gene_Name))
countData2$Gene_Name <- NULL

# Sanity checks
stopifnot(length(groups) == ncol(countData2))
if (anyDuplicated(rownames(countData2)) > 0) {
  warning("Duplicated Gene_Name after toupper(). Rowname duplicates exist; subsetting by gene may be ambiguous.")
}

# ---- Styling ----
genotype_colors  <- c("WT" = "#EC3A91", "MUT" = "#71980E")   # outline
condition_colors <- c("Control" = "#B0DBFF", "LPS" = "#CF98E2") # fill

make_gene_plot <- function(gene_name, count_mat, groups_vec) {
  gene_key <- toupper(gene_name)
  if (!gene_key %in% rownames(count_mat)) {
    stop(sprintf("Gene '%s' not found in rownames(countData2).", gene_key))
  }

  gene_expression <- as.numeric(count_mat[gene_key, ])

  plot_data <- data.frame(
    Sample     = colnames(count_mat),
    Expression = gene_expression,
    Group      = as.character(groups_vec),
    stringsAsFactors = FALSE
  )

  # Genotype/Condition parsing from Group
  plot_data$Genotype  <- ifelse(grepl("wt", plot_data$Group, ignore.case = TRUE), "WT", "MUT")
  plot_data$Condition <- ifelse(grepl("Control", plot_data$Group, ignore.case = TRUE), "Control", "LPS")
  plot_data$Group_Combined <- paste(plot_data$Genotype, plot_data$Condition, sep = "+")

  ggplot(plot_data, aes(x = Group_Combined, y = Expression)) +
    geom_point(
      aes(fill = Condition, color = Genotype, shape = Genotype),
      size = 9, stroke = 2
    ) +
    scale_fill_manual(
      name = "Condition",
      values = condition_colors,
      guide = guide_legend(override.aes = list(shape = 21, size = 9, stroke = 0))
    ) +
    scale_color_manual(
      name = "Genotype",
      values = genotype_colors,
      labels = c("WT" = "Wild-type", "MUT" = "Mutant")
    ) +
    scale_shape_manual(
      name = "Genotype",
      values = c("WT" = 21, "MUT" = 22),
      labels = c("WT" = "Wild-type", "MUT" = "Mutant")
    ) +
    scale_x_discrete(limits = c("WT+Control", "WT+LPS", "MUT+Control", "MUT+LPS")) +
    labs(
      title = paste("Expression of", gene_key, "Across Groups"),
      x = "",
      y = "Expression Level"
    ) +
    theme_minimal() +
    theme(
      axis.text.x = element_text(size = 14, color = "black"),
      axis.text.y = element_text(size = 14, color = "black"),
      axis.title.x = element_text(size = 16),
      axis.title.y = element_text(size = 16),
      axis.ticks = element_line(color = "black"),
      axis.line = element_line(color = "black"),
      panel.background = element_blank(),
      panel.border = element_rect(color = "black", fill = NA),
      panel.grid.major = element_blank(),
      panel.grid.minor = element_blank(),
      legend.position = "right",
      legend.text = element_text(size = 12),
      legend.title = element_text(size = 14),
      legend.key = element_blank()
    )
}

save_gene_svg <- function(gene_name, outdir = "gene_svgs", width = 10, height = 6) {
  dir.create(outdir, showWarnings = FALSE, recursive = TRUE)

  p <- make_gene_plot(gene_name, countData2, groups)
  outfile <- file.path(outdir, paste0(toupper(gene_name), "_lmc_expression.svg"))

  ggsave(
    filename = outfile,
    plot = p,
    device = svglite::svglite,
    width = width,
    height = height
  )

  message("Saved: ", normalizePath(outfile, winslash = "/", mustWork = FALSE))
  invisible(outfile)
}

# ---- Save requested genes ----
save_gene_svg("MEF2C")
save_gene_svg("MEF2A")


In [ ]:
library(dplyr)

# Calculate mean and sd for each group
summary_data <- plot_data %>%
  group_by(Group_Combined, Genotype, Condition) %>%
  summarise(
    mean_expression = mean(Expression, na.rm = TRUE),
    sd_expression = sd(Expression, na.rm = TRUE),
    n = n()
  )


In [ ]:
ggplot(summary_data, aes(x = Group_Combined, y = mean_expression)) +
  geom_point(
    aes(fill = Condition, color = Genotype, shape = Genotype),
    size = 9, stroke = 2
  ) +
  geom_errorbar(
    aes(ymin = mean_expression - sd_expression, ymax = mean_expression + sd_expression),
    width = 0.3, linewidth = 1
  ) +
  scale_fill_manual(
    name = "Condition",
    values = condition_colors,
    guide = guide_legend(override.aes = list(shape = 21, size = 9, stroke = 0))
  ) +
  scale_color_manual(
    name = "Genotype",
    values = genotype_colors,
    labels = c("WT" = "Wild-type", "MUT" = "Mutant")
  ) +
  scale_shape_manual(
    name = "Genotype",
    values = c("WT" = 21, "MUT" = 22),
    labels = c("WT" = "Wild-type", "MUT" = "Mutant")
  ) +
  scale_x_discrete(
    limits = c("WT+Control", "WT+LPS", "MUT+Control", "MUT+LPS")
  ) +
  labs(
    title = paste("Mean Expression of", gene_name, "Across Groups"),
    x = "",
    y = "Mean Expression ± SD"
  ) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(size = 14, color = "black"),
    axis.text.y = element_text(size = 14, color = "black"),
    axis.title.x = element_text(size = 16),
    axis.title.y = element_text(size = 16),
    axis.ticks = element_line(color = "black"),
    axis.line = element_line(color = "black"),
    panel.background = element_blank(),
    panel.border = element_rect(color = "black", fill = NA),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    legend.position = "right",
    legend.text = element_text(size = 12),
    legend.title = element_text(size = 14),
    legend.key = element_blank()
  )


In [ ]:
#Plotting Glimma/Scatter plots for each gene seperately
countData2 <- countData1
rownames(countData2) <- toupper(countData2$Gene_Name)
countData2$Gene_Name <- NULL
gene_expression <- countData2["C2",]  # Extracting expression levels for particular gene

plot_data <- data.frame(
  Sample = colnames(countData2),  
  Expression = as.numeric(gene_expression),  
  Group = groups  
)

gene_name <- "C2"

In [ ]:
# Load required libraries
library(ggplot2)

# Ensure gene names are in uppercase
countData2 <- countData1
rownames(countData2) <- toupper(countData2$Gene_Name)
countData2$Gene_Name <- NULL

# Define new gene list
gene_list <- c(
 "CCL4"
)

# Define colors
genotype_colors <- c("WT" = "#EC3A91", "MUT" = "#71980E")
condition_colors <- c("Control" = "#B0DBFF", "LPS" = "#CF98E2")

# Create output directory
output_dir <- "lmc_Gene_Expression_Plots"
if (!dir.exists(output_dir)) dir.create(output_dir)

# Loop through each gene and generate the plots
for (gene in gene_list) {
  if (gene %in% rownames(countData2)) {
    gene_expression <- countData2[gene,]  # Extract expression levels
    
    plot_data <- data.frame(
      Sample = colnames(countData2),
      Expression = as.numeric(gene_expression),
      Group = groups
    )

    # Assign Genotype and Condition
    plot_data$Genotype <- ifelse(grepl("wt", plot_data$Group, ignore.case = TRUE), "WT", "MUT")
    plot_data$Condition <- ifelse(grepl("Control", plot_data$Group, ignore.case = TRUE), "Control", "LPS")
    plot_data$Group_Combined <- with(plot_data, paste(Genotype, Condition, sep = "+"))

    # Generate plot
    p <- ggplot(plot_data, aes(x = Group_Combined, y = Expression)) +
      geom_point(aes(fill = Condition, color = Genotype, shape = Genotype), size = 9, stroke = 2) +
      scale_fill_manual(name = "Condition", values = condition_colors, guide = guide_legend(override.aes = list(shape = 21, size = 9, stroke = 0))) +
      scale_color_manual(name = "Genotype", values = genotype_colors, labels = c("WT" = "Wild-type", "MUT" = "Mutant")) +
      scale_shape_manual(name = "Genotype", values = c("WT" = 21, "MUT" = 22), labels = c("WT" = "Wild-type", "MUT" = "Mutant")) +
      scale_x_discrete(limits = c("WT+Control", "WT+LPS", "MUT+Control", "MUT+LPS")) +
      labs(
        title = paste("Expression of", gene, "Across Groups"),
        x = "",
        y = "Expression Level"
      ) +
      theme_minimal() +
      theme(
        axis.text.x = element_text(size = 14, color = "black"),
        axis.text.y = element_text(size = 14, color = "black"),
        axis.title.x = element_text(size = 16),
        axis.title.y = element_text(size = 16),
        axis.ticks = element_line(color = "black"),
        axis.line = element_line(color = "black"),
        panel.background = element_blank(),
        panel.border = element_rect(color = "black", fill = NA),
        panel.grid.major = element_blank(),
        panel.grid.minor = element_blank(),
        legend.position = "right",
        legend.text = element_text(size = 12),
        legend.title = element_text(size = 14),
        legend.key = element_blank()
      )

    # Save plot
    plot_filename <- file.path(output_dir, paste0(gene, "lmc_expression_plot.png"))
    ggsave(plot_filename, plot = p, width = 8, height = 6, dpi = 300)
    
    message("Saved plot for: ", gene)
  } else {
    message("Gene not found in data: ", gene)
  }
}

message("All plots saved in the directory: ", output_dir)


In [ ]:
# Load required libraries
library(ggplot2)

# Ensure gene names are in uppercase
countData2 <- countData1
rownames(countData2) <- toupper(countData2$Gene_Name)
countData2$Gene_Name <- NULL

# Define new gene list
gene_list <- c(
  "SLPI", "C3", "CFI", "MARCO", "APOE", "WFDC17", "OAS1G", "GCH1", "SERPING1", "KYNU",
  "S100A8", "C2", "CLEC4D", "IRF5", "TREML4", "SLC11A1", "OAS1A", "LY86", "OAS2", "C1QA",
  "TLR13", "NLRC5", "HCK", "TLR7", "IFI204", "VAV1", "RSAD2", "NPY", "TLR9", "CCL3",
  "NAIP6", "OAS3", "C1QC", "LY9", "SLAMF7", "CCL4", "CX3CR1", "C1QB", "CD84", "IRF8",
  "TLR1", "AIM2", "CLEC4N", "CD180", "FCER1G", "NAIP5", "H60B", "SERPINB1A", "PTPN6",
  "SPI1", "MRC1", "CSF1R", "CLEC4A3", "CLEC7A", "WAS", "CASP1", "NAIP2", "SLC15A3",
  "C1S1", "UNC13D", "CCL5", "BTK", "CEACAM1", "DPP4", "MCLON2", "TREM1", "ADAM8",
  "PLD4", "IFIT1", "H2-M3", "SIRPA", "CARD9", "CCR1", "CD74", "FCGR1", "TNFAIP8L2",
  "MYO1F", "NLRP3", "GBP6", "TICAM2", "GBP9", "C4BP", "CXCL16", "SH2D1B1", "PRDX1",
  "PARP14", "TRIM29", "PYCARD", "MST1R", "NLRC3", "HAVCR2", "ARRB2", "PTK2B",
  "TMEM106A", "SP110", "CCL8", "ISG15", "CASP4", "NR1H4", "OASL2", "SP100",
  "IL12RB1", "IL18", "BST2", "SPON2", "RAB20", "H2-AB1",
  "C3", "AIF1", "FCGR2B", "TREML4", "BIN2", "CD300A", "NCKAP1L", "MSR1", "RHOH",
  "FCER1G", "CLEC7A", "ITGB2", "SIRPA", "FCGR1", "ELMO1"
)

# Define colors
genotype_colors <- c("WT" = "#EC3A91", "MUT" = "#71980E")
condition_colors <- c("Control" = "#B0DBFF")  # Only Control is used now

# Create output directory
output_dir <- "lmc_Gene_Expression_Plots"
if (!dir.exists(output_dir)) dir.create(output_dir)

# Loop through each gene and generate the plots
for (gene in gene_list) {
  if (gene %in% rownames(countData2)) {
    gene_expression <- countData2[gene,]  # Extract expression levels
    
    plot_data <- data.frame(
      Sample = colnames(countData2),
      Expression = as.numeric(gene_expression),
      Group = groups
    )

    # Assign Genotype and Condition
    plot_data$Genotype <- ifelse(grepl("wt", plot_data$Group, ignore.case = TRUE), "WT", "MUT")
    plot_data$Condition <- ifelse(grepl("Control", plot_data$Group, ignore.case = TRUE), "Control", "LPS")
    
    # Filter out LPS groups
    plot_data <- subset(plot_data, Condition == "Control")
    
    # Update group labels
    plot_data$Group_Combined <- with(plot_data, paste(Genotype, Condition, sep = "+"))

    # Generate plot
    p <- ggplot(plot_data, aes(x = Group_Combined, y = Expression)) +
      geom_point(aes(fill = Condition, color = Genotype, shape = Genotype), size = 9, stroke = 2) +
      scale_fill_manual(name = "Condition", values = condition_colors, guide = guide_legend(override.aes = list(shape = 21, size = 9, stroke = 0))) +
      scale_color_manual(name = "Genotype", values = genotype_colors, labels = c("WT" = "Wild-type", "MUT" = "Mutant")) +
      scale_shape_manual(name = "Genotype", values = c("WT" = 21, "MUT" = 22), labels = c("WT" = "Wild-type", "MUT" = "Mutant")) +
      scale_x_discrete(limits = c("WT+Control", "MUT+Control")) +  # Excluding LPS groups
      labs(
        title = paste("Expression of", gene, "Across Groups"),
        x = "",
        y = "Expression Level"
      ) +
      theme_minimal() +
      theme(
        axis.text.x = element_text(size = 14, color = "black"),
        axis.text.y = element_text(size = 14, color = "black"),
        axis.title.x = element_text(size = 16),
        axis.title.y = element_text(size = 16),
        axis.ticks = element_line(color = "black"),
        axis.line = element_line(color = "black"),
        panel.background = element_blank(),
        panel.border = element_rect(color = "black", fill = NA),
        panel.grid.major = element_blank(),
        panel.grid.minor = element_blank(),
        legend.position = "right",
        legend.text = element_text(size = 12),
        legend.title = element_text(size = 14),
        legend.key = element_blank()
      )

    # Save plot
    plot_filename <- file.path(output_dir, paste0(gene, "_lmc_expression_plot_withoutLPS.png"))
    ggsave(plot_filename, plot = p, width = 8, height = 6, dpi = 300)
    
    message("Saved plot for: ", gene)
  } else {
    message("Gene not found in data: ", gene)
  }
}

message("All plots saved in the directory: ", output_dir)


In [ ]:
countData2

In [ ]:
write.csv(gene_expression,'IL1B_expression.csv') 

In [ ]:
###########################
## Load packages
###########################
suppressPackageStartupMessages({
  library(pheatmap)
})

###########################
## 1. Define genes of interest
###########################
genes_of_interest <- c(
  "HOXA1","HOXA10","HOXA11","HOXA11OS","HOXA2","HOXA3","HOXA4","HOXA5","HOXA6","HOXA7","HOXA9",
  "HOXB1","HOXB2","HOXB3","HOXB3OS","HOXB4","HOXB5","HOXB5OS","HOXB6","HOXB7","HOXB8","HOXB9",
  "HOXC10","HOXC11","HOXC13","HOXC4","HOXC5","HOXC6","HOXC8","HOXC9",
  "HOXD10","HOXD11","HOXD3","HOXD3OS1","HOXD8","HOXD9",
  "PAX3","PAX6",
  "POU2F1","POU2F2","POU2F3",
  "POU3F1","POU3F2","POU3F3","POU3F4",
  "POU4F1","POU4F3",
  "POU5F2",
  "POU6F1"
)

###########################
## 2. Subset countData2 to only those genes
##    and keep them in the specified order
###########################

# countData2 is assumed to have rownames = gene names, columns = samples
if (is.null(rownames(countData2))) {
  stop("countData2 must have gene names as rownames before running this script.")
}

# match() preserves the explicit order in genes_of_interest
row_idx <- match(genes_of_interest, rownames(countData2))

# remove genes that aren't present (NA after match)
row_idx <- row_idx[!is.na(row_idx)]

expr_sub <- countData2[row_idx, , drop = FALSE]

###########################
## 3. Collapse every 4 columns by mean
##    e.g. columns 1-4 -> group1 mean,
##         columns 5-8 -> group2 mean, etc.
###########################

n_cols <- ncol(expr_sub)

# create a group label for each column:
group_labels <- rep(
  paste0("Group", seq_len(ceiling(n_cols / 4))),
  each = 4
)[1:n_cols]

# take rowMeans within each group of 4 columns
expr_collapsed <- sapply(
  split(seq_len(n_cols), group_labels),
  function(col_idx) {
    rowMeans(expr_sub[, col_idx, drop = FALSE], na.rm = TRUE)
  }
)

# ensure it's a matrix (sapply can return vector if only 1 group)
expr_collapsed <- as.matrix(expr_collapsed)

###########################
## 4. Build a simple annotation for columns (optional)
##    Here we just annotate which averaged group each column is.
##    You can replace this with your manual_annotation if you want.
###########################

annotation_col <- data.frame(
  Condition = colnames(expr_collapsed)
)
rownames(annotation_col) <- colnames(expr_collapsed)

###########################
## 5. Plot heatmap
###########################

pheatmap(
  expr_collapsed,
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  scale = "row",                    # z-score per gene
  annotation_col = annotation_col,  # shows the grouped-condition bar on top
  show_rownames = TRUE,
  fontsize_row = 6,
  fontsize_col = 8,
  border_color = NA
)

###########################
## 6. (optional) Check the collapsed matrix
###########################
print(dim(expr_sub))        # before collapsing
print(dim(expr_collapsed))  # after collapsing
print(head(expr_collapsed))


In [ ]:
###########################
## Load packages
###########################
suppressPackageStartupMessages({
  library(pheatmap)
  library(grid)
})

###########################
## 0. Output directory
###########################
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

###########################
## 1. Define genes of interest
###########################
genes_of_interest <- c(
  "HOXA1","HOXA10","HOXA11","HOXA11OS","HOXA2","HOXA3","HOXA4","HOXA5","HOXA6","HOXA7","HOXA9",
  "HOXB1","HOXB2","HOXB3","HOXB3OS","HOXB4","HOXB5","HOXB5OS","HOXB6","HOXB7","HOXB8","HOXB9",
  "HOXC10","HOXC11","HOXC13","HOXC4","HOXC5","HOXC6","HOXC8","HOXC9",
  "HOXD10","HOXD11","HOXD3","HOXD3OS1","HOXD8","HOXD9",
  "PAX3","PAX6",
  "POU2F1","POU2F2","POU2F3",
  "POU3F1","POU3F2","POU3F3","POU3F4",
  "POU4F1","POU4F3",
  "POU5F2",
  "POU6F1"
)

###########################
## 2. Subset countData2 to only those genes
##    and keep them in the specified order
###########################
if (is.null(rownames(countData2))) {
  stop("countData2 must have gene names as rownames before running this script.")
}

row_idx <- match(genes_of_interest, rownames(countData2))
row_idx <- row_idx[!is.na(row_idx)]
expr_sub <- countData2[row_idx, , drop = FALSE]

###########################
## 3. Collapse every 4 columns by mean
###########################
n_cols <- ncol(expr_sub)

group_labels <- rep(
  paste0("Group", seq_len(ceiling(n_cols / 4))),
  each = 4
)[1:n_cols]

expr_collapsed <- sapply(
  split(seq_len(n_cols), group_labels),
  function(col_idx) rowMeans(expr_sub[, col_idx, drop = FALSE], na.rm = TRUE)
)
expr_collapsed <- as.matrix(expr_collapsed)

###########################
## 4. Column annotation (optional)
###########################
annotation_col <- data.frame(
  Condition = colnames(expr_collapsed)
)
rownames(annotation_col) <- colnames(expr_collapsed)

###########################
## 5. Plot heatmap + SAVE AS SVG
###########################
ph <- pheatmap(
  expr_collapsed,
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  scale = "row",
  annotation_col = annotation_col,
  show_rownames = TRUE,
  fontsize_row = 6,
  fontsize_col = 8,
  border_color = NA
)

svg_file <- file.path(out_dir, "heatmap_HOX_PAX_POU_collapsedBy4_rowScaled.svg")
svg(svg_file, width = 10, height = 8)
grid.newpage()
grid.draw(ph$gtable)
dev.off()

###########################
## 6. (optional) Check the collapsed matrix
###########################
print(dim(expr_sub))        # before collapsing
print(dim(expr_collapsed))  # after collapsing
print(head(expr_collapsed))


In [ ]:
head(rownames(countData2))
head(colnames(countData2))  #

In [ ]:
countData2

In [ ]:
# Create Genotype and Condition columns based on the Group column
plot_data$Genotype <- ifelse(grepl("wt", plot_data$Group), "WT", "MUT")
plot_data$Condition <- ifelse(grepl("Control", plot_data$Group), "Control", "LPS")

# Create a new column for the combined labels
plot_data$Group_Combined <- with(plot_data, paste(Genotype, Condition, sep = "+"))

# Define the colors
genotype_colors <- c("WT" = "#fe9388", "MUT" = "#9bca1e")
condition_colors <- c("Control" = "#add8e6", "LPS" = "#e298fe")  # Lighter blue for Control

# Define the colors
genotype_colors <- c("WT" = "#EC3A91", "MUT" = "#71980E")  # Adjusted colors
condition_colors <- c("Control" = "#B0DBFF", "LPS" = "#CF98E2")  # Lightened inside colors

# Create the plot
ggplot(plot_data, aes(x = Group_Combined, y = Expression)) +
  geom_point(aes(fill = Condition, color = Genotype, shape = Genotype), size = 9, stroke = 2) +  # Use shape 21 for filled points with outline
  scale_fill_manual(name = "Condition", values = condition_colors, guide = guide_legend(override.aes = list(shape = 21, size = 9, stroke = 0))) +  # Inner circle colors and remove outline in legend
  scale_color_manual(name = "Genotype", values = genotype_colors, labels = c("WT" = "Wild-type", "MUT" = "Mutant")) +  # Outer circle colors
  scale_shape_manual(name = "Genotype", values = c("WT" = 21, "MUT" = 22), labels = c("WT" = "Wild-type", "MUT" = "Mutant")) +  # Shape 21 for circles, 22 for squares
  scale_x_discrete(limits = c("WT+Control", "WT+LPS", "MUT+Control", "MUT+LPS")) +  # Set the order of the x-axis labels
  labs(
    title = paste("Expression of", gene_name, "Across Groups"),
    x = "",
    y = "Expression Level"
  ) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(size = 14, color = "black"),  # Increase x-axis label size
    axis.text.y = element_text(size = 14, color = "black"),  # Increase y-axis label size
    axis.title.x = element_text(size = 16),  # Increase x-axis title size
    axis.title.y = element_text(size = 16),  # Increase y-axis title size
    axis.ticks = element_line(color = "black"),
    axis.line = element_line(color = "black"),
    panel.background = element_blank(),
    panel.border = element_rect(color = "black", fill = NA),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    legend.position = "right",
    legend.text = element_text(size = 12),  # Increase legend text size
    legend.title = element_text(size = 14),  # Increase legend title size
    legend.key = element_blank()  # Remove legend key outline
  )

In [ ]:
res <- results(dds, name="genotypemut.conditionLPS",alpha = 0.05)
topGenes <- head(order(res$padj), 50)
topGeneCounts <- countData[topGenes,]

In [ ]:
topGenes <- head(order(res$log2FoldChange, na.last=NA), 50)  # na.last=NA removes NAs
countData <- counts(dds, normalized=TRUE)
topGeneCounts <- countData[topGenes, ]
rownames(manual_annotation) <- colnames(topGeneCounts)

In [ ]:
# Remove rows where row name starts with 'NA'
filtered_topGeneCounts <- topGeneCounts[!grepl("^NA", rownames(topGeneCounts)), ]

In [ ]:
# Filter the rows of filtered_topGeneCounts based on your criteria
filtered_rows <- rownames(filtered_topGeneCounts)[!grepl("^Gm|^Mir|^ENSMUS|^Rpl|Rik$", rownames(filtered_topGeneCounts))]

# Subset the dataframe to include only the filtered rows
filtered_topGeneCounts_filtered <- filtered_topGeneCounts[filtered_rows, ]

# Create the heatmap with the filtered dataframe
pheatmap(filtered_topGeneCounts_filtered, 
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         annotation_col = as.data.frame(manual_annotation), 
         show_rownames = TRUE)


In [ ]:
suppressPackageStartupMessages({
  library(pheatmap)
  library(grid)
})

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Filter the rows of filtered_topGeneCounts based on your criteria
filtered_rows <- rownames(filtered_topGeneCounts)[
  !grepl("^Gm|^Mir|^ENSMUS|^Rpl|Rik$", rownames(filtered_topGeneCounts))
]

# Subset the dataframe to include only the filtered rows
filtered_topGeneCounts_filtered <- filtered_topGeneCounts[filtered_rows, , drop = FALSE]

# Create + capture the heatmap
ph <- pheatmap(
  filtered_topGeneCounts_filtered,
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  annotation_col = as.data.frame(manual_annotation),
  show_rownames = TRUE
)

# Save as SVG
svg_file <- file.path(out_dir, "heatmap_filtered_topGeneCounts.svg")
svg(svg_file, width = 10, height = 8)
grid.newpage()
grid.draw(ph$gtable)
dev.off()


In [ ]:
# Create the heatmap with the filtered dataframe
pheatmap(filtered_topGeneCounts_filtered, 
         cluster_rows = FALSE, 
         cluster_cols = FALSE,
         scale = "row",
         annotation_col = as.data.frame(manual_annotation), 
         show_rownames = TRUE)

In [ ]:
suppressPackageStartupMessages({
  library(pheatmap)
  library(grid)
})

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Create + capture the heatmap
ph <- pheatmap(
  filtered_topGeneCounts_filtered,
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  scale = "row",
  annotation_col = as.data.frame(manual_annotation),
  show_rownames = TRUE
)

# Save as SVG
svg(
  file.path(out_dir, "heatmap_filtered_topGeneCounts_rowScaled.svg"),
  width = 10,
  height = 8
)

grid.newpage()
grid.draw(ph$gtable)
dev.off()


In [ ]:
nf_kb <- read.table('nf_kb_sites.txt')
colnames(nf_kb)[1] <- "Gene_name"

In [ ]:
# Filter the countData to include only genes in the nf_kb list
rownames(countData) <- toupper(rownames(countData))
nf_kb_genes <- intersect(rownames(countData), nf_kb$Gene_name)
filteredCountData <- countData[nf_kb_genes, ]

# Rank genes based on their mean expression levels and select the top 100
meanExpressionLevels <- rowMeans(filteredCountData)
top100Genes <- names(sort(meanExpressionLevels, decreasing = TRUE))[1:50]
filteredCountDataTop100 <- filteredCountData[top100Genes, ]

# Apply further filtering if necessary (you might want to skip this if focusing on top 100 already)
# Example pattern shown in your request - adjust if using this additional filter step
filtered_rows <- grep("^(?!GM|MIR|ENSMUS|RPL|RIK).*$", rownames(filteredCountDataTop100), perl = TRUE)
filteredCountDataTop100Filtered <- filteredCountDataTop100[filtered_rows, ]


# Plot the heatmap for the top 100 genes
pheatmap(filteredCountDataTop100Filtered, 
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         annotation_col = as.data.frame(manual_annotation), 
         show_rownames = TRUE)

In [ ]:
suppressPackageStartupMessages({
  library(pheatmap)
  library(grid)
})

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Filter the countData to include only genes in the nf_kb list
rownames(countData) <- toupper(rownames(countData))
nf_kb_genes <- intersect(rownames(countData), nf_kb$Gene_name)
filteredCountData <- countData[nf_kb_genes, , drop = FALSE]

# Rank genes based on mean expression and select top 50 (your code selects 50)
meanExpressionLevels <- rowMeans(filteredCountData, na.rm = TRUE)
top50Genes <- names(sort(meanExpressionLevels, decreasing = TRUE))[1:50]
filteredCountDataTop50 <- filteredCountData[top50Genes, , drop = FALSE]

# Optional additional filtering: remove GM/MIR/ENSMUS/RPL/*RIK (case-insensitive due to toupper)
filtered_rows <- grep("^(?!GM|MIR|ENSMUS|RPL|.*RIK$).*$",
                      rownames(filteredCountDataTop50), perl = TRUE)
filteredCountDataTop50Filtered <- filteredCountDataTop50[filtered_rows, , drop = FALSE]

# Plot + capture heatmap
ph <- pheatmap(
  filteredCountDataTop50Filtered,
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  annotation_col = as.data.frame(manual_annotation),
  show_rownames = TRUE
)

# Save as SVG
svg_file <- file.path(out_dir, "heatmap_nfkb_top50_meanExpr.svg")
svg(svg_file, width = 10, height = 8)
grid.newpage()
grid.draw(ph$gtable)
dev.off()


In [ ]:
res <- results(dds, name="genotypemut.conditionLPS")

In [ ]:
# Correctly filtering genes, accounting for NAs in 'padj'
filtered_genes <- res[!is.na(res$log2FoldChange) & !is.na(res$padj) & res$padj < 0.05, ]

In [ ]:
# Sort the filtered genes by the absolute value of log2FoldChange in descending order
sorted_genes <- filtered_genes[order(-abs(filtered_genes$log2FoldChange)), ]

# Select the top 50 genes based on this sorting
top_50_genes <- head(sorted_genes, 50)

In [ ]:
# Extract log2FoldChange values for the top 50 genes
log2fc_values_top_50 <- top_50_genes$log2FoldChange

# Convert to a matrix for pheatmap
log2fc_matrix_top_50 <- matrix(log2fc_values_top_50, nrow = length(log2fc_values_top_50), ncol = 1, 
                               dimnames = list(rownames(top_50_genes), "Log2FoldChange"))
genes_to_keep <- !grepl("^(Gm|Mir|ENSMUS|Rpl.*|.*Rik|mt-|Malat1)", rownames(log2fc_matrix_top_50))

# Filter both the log2FoldChange values and update row names accordingly
log2fc_values_filtered <- log2fc_values_top_50[genes_to_keep]
row_names_filtered <- rownames(log2fc_matrix_top_50)[genes_to_keep]

# Update the matrix for pheatmap using filtered genes
log2fc_matrix_filtered <- matrix(log2fc_values_filtered, nrow = length(log2fc_values_filtered), ncol = 1, 
                                 dimnames = list(row_names_filtered, "Log2FoldChange"))

In [ ]:
# Invert the signs of log2FoldChange values
log2fc_values_inverted <- -log2fc_values_filtered

# Update the matrix for pheatmap using inverted log2FoldChange values
log2fc_matrix_inverted <- matrix(log2fc_values_inverted, nrow = length(log2fc_values_inverted), ncol = 1, 
                                 dimnames = list(row_names_filtered, "Log2FoldChange"))

# Plot the heatmap with a slimmer appearance by adjusting cellwidth and cellheight
pheatmap(log2fc_matrix_inverted,
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         show_rownames = TRUE,
         #color = color_palette_fc,
         cellwidth = 50,   # Adjust this value as needed to make the heatmap slimmer
         cellheight = 10)  # Adjust the cell height if necessary


In [ ]:
normalized_counts <- counts(dds, normalized=TRUE)
normalized_counts <- as.data.frame(normalized_counts)

# Calculate mean normalized counts for each gene
mean_normalized_counts <- rowMeans(normalized_counts)

# Create a data frame of mean normalized counts with row names
mean_normalized_counts_df <- data.frame(Gene=rownames(normalized_counts), MeanCount=mean_normalized_counts)

# Filter out rows based on specified patterns, including genes starting with "mt-"
filtered_rows <- mean_normalized_counts_df[!grepl("^(Gm|Mir|ENSMUS|Rpl.*|.*Rik|mt-|Malat1)", mean_normalized_counts_df$Gene), ]

# Sort the filtered genes by mean normalized count in descending order
sorted_filtered_genes <- filtered_rows[order(-filtered_rows$MeanCount), ]

# Get the top 50 genes from the filtered and sorted list
top_50_filtered_genes <- head(sorted_filtered_genes, 50)

# Print the gene names of the top 50 genes
print(top_50_filtered_genes$Gene)

top_50_counts <- normalized_counts[match(top_50_filtered_genes$Gene, rownames(normalized_counts)), ]

pheatmap(top_50_counts,
         show_rownames = TRUE,
         show_colnames = TRUE,
         cluster_rows = FALSE,
         cluster_cols = FALSE,
         color = color_palette,
        scale = "row")

In [ ]:
pheatmap(top_50_counts,
         show_rownames = TRUE,
         show_colnames = TRUE,
         cluster_rows = FALSE,
         cluster_cols = FALSE,
         color = color_palette)

In [ ]:
suppressPackageStartupMessages({
  library(pheatmap)
  library(grid)
})

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Create + capture the heatmap
ph <- pheatmap(
  top_50_counts,
  show_rownames = TRUE,
  show_colnames = TRUE,
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  color = color_palette
)

# Save as SVG
svg(
  file.path(out_dir, "heatmap_top50_counts.svg"),
  width = 10,
  height = 8
)

grid.newpage()
grid.draw(ph$gtable)
dev.off()


In [ ]:
dds

In [ ]:
res <- results(dds, name="genotypemut.conditionLPS")
# Correctly filtering genes, accounting for NAs in 'padj'
# Correctly filtering genes, accounting for NAs in 'padj'
filtered_genes <- res[!is.na(res$log2FoldChange) & !is.na(res$padj) & res$padj < 0.5, ]


In [ ]:

# Sort the filtered genes by the absolute value of log2FoldChange in descending order
sorted_genes <- filtered_genes[order(-abs(filtered_genes$log2FoldChange)), ]

# Select the top 50 genes based on this sorting
top_50_genes <- head(sorted_genes, 50)

# Extract log2FoldChange values for the top 50 genes
log2fc_values_top_50 <- top_50_genes$log2FoldChange

# Convert to a matrix for pheatmap
log2fc_matrix_top_50 <- matrix(log2fc_values_top_50, nrow = length(log2fc_values_top_50), ncol = 1, 
                               dimnames = list(rownames(top_50_genes), "Log2FoldChange"))
genes_to_keep <- !grepl("^(Gm|Mir|ENSMUS|Rpl.*|.*Rik|mt-|Malat1)", rownames(log2fc_matrix_top_50))

# Filter both the log2FoldChange values and update row names accordingly
log2fc_values_filtered <- log2fc_values_top_50[genes_to_keep]
row_names_filtered <- rownames(log2fc_matrix_top_50)[genes_to_keep]

# Update the matrix for pheatmap using filtered genes
log2fc_matrix_filtered <- matrix(log2fc_values_filtered, nrow = length(log2fc_values_filtered), ncol = 1, 
                                 dimnames = list(row_names_filtered, "Log2FoldChange"))

# Invert the signs of log2FoldChange values
log2fc_values_inverted <- -log2fc_values_filtered

# Update the matrix for pheatmap using inverted log2FoldChange values
log2fc_matrix_inverted <- matrix(log2fc_values_inverted, nrow = length(log2fc_values_inverted), ncol = 1, 
                                 dimnames = list(row_names_filtered, "Log2FoldChange"))


color_palette_fc <- colorRampPalette(c("blue", "white", "red"))(100)
# Plot the heatmap with a slimmer appearance by adjusting cellwidth and cellheight
pheatmap(log2fc_matrix_inverted,
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         show_rownames = TRUE,
         color = color_palette_fc,
         cellwidth = 50,   # Adjust this value as needed to make the heatmap slimmer
         cellheight = 10)  # Adjust the cell height if necessary

I.The effect of treatment in wild-type.[This is for WT, treated compared with untreated.]

1.Genes upregulated with LPS in wild-type samples

2.Genes downregulated with LPS in wild-type samples

In [ ]:
res = results(dds, contrast=c("condition","LPS","Control"))
ix = which.min(res$padj) # most significant
res <- res[order(res$padj),] # sort
kable(res[1:5,-(3:4)])

In [ ]:
p <- barplot(assay(dds)[ix,],las=2, main=rownames(dds)[ ix  ]  )
#the plot below shows which gene is most prominent and how much it is expressed in each sample.

In [ ]:
res2 <- res

In [ ]:
#reset par
par(mfrow=c(1,1))
# Make a basic volcano plot
with(res2, plot(log2FoldChange, -log10(padj), pch=20, main="Volcano plot", xlim=c(-5,10)))

# Add colored points: blue if padj<0.05, red if log2FC>1 and padj<0.05)
with(subset(res2, padj<.05 ), points(log2FoldChange, -log10(padj), pch=20, col="blue"))
with(subset(res2, padj<.05 & abs(log2FoldChange)>1), points(log2FoldChange, -log10(padj), pch=20, col="dark green"))

In [ ]:
write.csv(res2,"LMC_I.csv")
res2 <- na.omit(res2)
res2_I <- res2[res2$padj < 0.05, ]

In [ ]:
res2_I_df <- data.frame(res2_I)

In [ ]:
ggplot(res2_I_df, aes(x = baseMean, y = log2FoldChange)) +
  geom_point(alpha = 0.6) + # Set alpha for better visualization if points overlap
  scale_x_log10() + # Log-transform the baseMean axis for better visualization
  theme_minimal() +
  labs(x = "Base Mean (log scale)", 
       y = "Log2 Fold Change", 
       title = "Scatter Plot of baseMean vs. log2FoldChange") +
  geom_hline(yintercept = 0, linetype = "dashed", color = "red") # Add a horizontal line at y=0

In [ ]:
library(ggplot2)

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Create the plot
p <- ggplot(res2_I_df, aes(x = baseMean, y = log2FoldChange)) +
  geom_point(alpha = 0.6) +
  scale_x_log10() +
  theme_minimal() +
  labs(
    x = "Base Mean (log scale)",
    y = "Log2 Fold Change",
    title = "Scatter Plot of baseMean vs. log2FoldChange"
  ) +
  geom_hline(yintercept = 0, linetype = "dashed", color = "red")

# Save as SVG
ggsave(
  filename = file.path(out_dir, "scatter_baseMean_vs_log2FC.svg"),
  plot = p,
  device = "svg",
  width = 7,
  height = 6,
  units = "in"
)


In [ ]:
# Number of genes you want to label
N <- 10  

# Sort your data frame based on the criteria (e.g., padj)
sorted_indices <- order(res2$padj)

# Initialize labels to NA
res2$labels <- NA

# Select top N genes based on sorted indices
top_N_indices <- sorted_indices[1:N]

# Identify those among the top N that need to be filtered out
filtered_indices <- top_N_indices[grepl("^Gm|-ik$|Mir", res2$SYMBOL[top_N_indices])]

# Identify how many more genes we need to pick
num_more <- length(filtered_indices)

# Pick the next top-most genes that are valid
next_valid_indices <- sorted_indices[(N + 1):(N + num_more)]
next_valid_indices <- next_valid_indices[!grepl("^Gm|-ik$|Mir", res2$SYMBOL[next_valid_indices])]

# Combine the valid top indices and the next valid indices to get top N labels
final_top_N_indices <- c(setdiff(top_N_indices, filtered_indices), next_valid_indices[1:num_more])

# Filter out NA indices
final_top_N_indices <- final_top_N_indices[!is.na(final_top_N_indices)]
final_top_N_indices <- final_top_N_indices[!is.na(res2$SYMBOL[final_top_N_indices])]

# Check if any of the final_top_N_indices are NA or if corresponding SYMBOLs are NA
if (any(is.na(final_top_N_indices)) || any(is.na(res2$SYMBOL[final_top_N_indices]))) {
    stop("NA indices detected. Please resolve before proceeding.")
}

# Set the labels for these final top N indices
res2$labels[final_top_N_indices] <- res2$SYMBOL[final_top_N_indices]

# Create the EnhancedVolcano plot
EnhancedVolcano(res2,
                lab = rownames(res2),
                x = 'log2FoldChange',
                y = 'padj')


In [ ]:
suppressPackageStartupMessages({
  library(EnhancedVolcano)
})

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

###########################
## Label selection logic
###########################

N <- 10  

sorted_indices <- order(res2$padj)

res2$labels <- NA

top_N_indices <- sorted_indices[1:N]

filtered_indices <- top_N_indices[
  grepl("^Gm|-ik$|Mir", res2$SYMBOL[top_N_indices])
]

num_more <- length(filtered_indices)

next_valid_indices <- sorted_indices[(N + 1):(N + num_more)]
next_valid_indices <- next_valid_indices[
  !grepl("^Gm|-ik$|Mir", res2$SYMBOL[next_valid_indices])
]

final_top_N_indices <- c(
  setdiff(top_N_indices, filtered_indices),
  next_valid_indices[1:num_more]
)

final_top_N_indices <- final_top_N_indices[!is.na(final_top_N_indices)]
final_top_N_indices <- final_top_N_indices[
  !is.na(res2$SYMBOL[final_top_N_indices])
]

if (any(is.na(final_top_N_indices)) ||
    any(is.na(res2$SYMBOL[final_top_N_indices]))) {
  stop("NA indices detected. Please resolve before proceeding.")
}

res2$labels[final_top_N_indices] <- res2$SYMBOL[final_top_N_indices]

###########################
## SAVE EnhancedVolcano as SVG
###########################

svg(
  file.path(out_dir, "EnhancedVolcano_top10_labeled.svg"),
  width = 8,
  height = 8
)

EnhancedVolcano(
  res2,
  lab = res2$labels,            # IMPORTANT: use your custom labels
  x = 'log2FoldChange',
  y = 'padj'
)

dev.off()


Here above plot suggests upregulated genes(positive Foldchange values) and downregulated genes(negative Foldchange values) in wildtype samples when subjected to LPS treatment.

In [ ]:
#Finding DE genes with adjusted p-value < 0.05 and LFC threshold = 1;

In [ ]:
res2_I$abs_LFC <- abs(res2_I$log2FoldChange)
res2_I_sig <- res2_I[res2_I$abs_LFC > 1, ]
res2_I_sig

In [ ]:
res3 <- res2[complete.cases(res2),]
res3$abs_LFC <- abs(res3$log2FoldChange)
res4 <- res3[res3$abs_LFC > 1, ]
res5 <- res4[res4$padj < 0.05, ]  

res2_I_up <- res2[res2$log2FoldChange > 1 & res2$padj < 0.05, ]
res2_I_down <- res2[res2$log2FoldChange < -1 & res2$padj < 0.05, ]

write.csv(res2_I,'results_DEgenes_LMC_I_2020countsdata.csv')
write.csv(rownames(res2_I_up),'upregulated_genes_LMC_I.csv')
write.csv(rownames(res2_I_down),'downregulated_genes_LMC_I.csv')

write.csv(res2_I_up,'upregulated_genes_LMC_I_rf.csv')
write.csv(res2_I_down,'downregulated_genes_LMC_I_rf.csv')


#write.csv(res5_I,'results_DEgenes_wt_LPStreatment_LMC_I_2020counts.csv')


In [ ]:
# Filter the countData to include only genes in the nf_kb list
rownames(countData) <- toupper(rownames(countData))
nf_kb_genes_res <- intersect(rownames(countData), nf_kb$Gene_name)
filteredCountData <- countData[nf_kb_genes, ]

# Rank genes based on their mean expression levels and select the top 100
meanExpressionLevels <- rowMeans(filteredCountData)
top100Genes <- names(sort(meanExpressionLevels, decreasing = TRUE))[1:50]
filteredCountDataTop100 <- filteredCountData[top100Genes, ]

# Apply further filtering if necessary (you might want to skip this if focusing on top 100 already)
# Example pattern shown in your request - adjust if using this additional filter step
filtered_rows <- grep("^(?!GM|MIR|ENSMUS|RPL|RIK).*$", rownames(filteredCountDataTop100), perl = TRUE)
filteredCountDataTop100Filtered <- filteredCountDataTop100[filtered_rows, ]


# Plot the heatmap for the top 100 genes
pheatmap(filteredCountDataTop100Filtered, 
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         annotation_col = as.data.frame(manual_annotation), 
         show_rownames = TRUE)

In [ ]:
suppressPackageStartupMessages({
  library(pheatmap)
  library(grid)
})

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Filter the countData to include only genes in the nf_kb list
rownames(countData) <- toupper(rownames(countData))
nf_kb_genes_res <- intersect(rownames(countData), nf_kb$Gene_name)

filteredCountData <- countData[nf_kb_genes_res, , drop = FALSE]

# Rank genes based on their mean expression levels and select the top 50 (your code uses 50)
meanExpressionLevels <- rowMeans(filteredCountData, na.rm = TRUE)
top50Genes <- names(sort(meanExpressionLevels, decreasing = TRUE))[1:50]
filteredCountDataTop50 <- filteredCountData[top50Genes, , drop = FALSE]

# Further filtering (remove GM/MIR/ENSMUS/RPL and genes ending in RIK)
filtered_rows <- grep("^(?!GM|MIR|ENSMUS|RPL|.*RIK$).*$",
                      rownames(filteredCountDataTop50), perl = TRUE)
filteredCountDataTop50Filtered <- filteredCountDataTop50[filtered_rows, , drop = FALSE]

# Plot + capture heatmap
ph <- pheatmap(
  filteredCountDataTop50Filtered,
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  annotation_col = as.data.frame(manual_annotation),
  show_rownames = TRUE
)

# Save as SVG
svg_file <- file.path(out_dir, "heatmap_nfkb_top50_meanExpr_filtered.svg")
svg(svg_file, width = 10, height = 8)
grid.newpage()
grid.draw(ph$gtable)
dev.off()


In [ ]:
write.csv(nf_kb_genes_res,'nfkb_genes_lmc.csv')  

In [ ]:
res

In [ ]:
cell_cycle_lmc <- read.table('lmc_cellcyclegenes.txt')
cell_cycle_mef <- read.table('mef_cellcyclegenes.txt')

cell_cycle_lmc_combo <- rbind(cell_cycle_lmc,cell_cycle_mef)
colnames(cell_cycle_lmc_combo)[1] <- "Gene_name"
cell_cycle_lmc_combo$Gene_name <- toupper(cell_cycle_lmc_combo$Gene_name)

In [ ]:
# Define annotation colors for Genotype only
ann_colors <- list(
  Genotype = c("wt" = "#05d7df", "mut" = "#d6a3eb")
)

# Read the cell cycle gene data
cell_cycle_lmc <- read.table('lmc_cellcyclegenes.txt', header = TRUE)
cell_cycle_mef <- read.table('mef_cellcyclegenes.txt', header = TRUE)

# Ensure both data frames have the same column names
colnames(cell_cycle_lmc)[1] <- "Gene_name"
colnames(cell_cycle_mef)[1] <- "Gene_name"

# Combine the gene data and convert gene names to uppercase
cell_cycle_lmc_combo <- rbind(cell_cycle_lmc, cell_cycle_mef)
cell_cycle_lmc_combo$Gene_name <- toupper(cell_cycle_lmc_combo$Gene_name)

# Ensure rownames of countData are uppercase
rownames(countData) <- toupper(rownames(countData))

# Filter the countData to include only genes in the cell cycle list
cell_cycle_res <- intersect(rownames(countData), cell_cycle_lmc_combo$Gene_name)
filteredCountData_cellcycle <- countData[cell_cycle_res, ]

# Subset the manual_annotation and filteredCountData_cellcycle to exclude LPS samples
samples_to_keep <- manual_annotation$Condition != "LPS"
filteredCountData_cellcycle <- filteredCountData_cellcycle[, samples_to_keep]
manual_annotation_filtered <- manual_annotation[samples_to_keep, "Genotype", drop = FALSE]

# Set the row names of manual_annotation_filtered to match the column names of filteredCountData_cellcycle
rownames(manual_annotation_filtered) <- colnames(filteredCountData_cellcycle)

# Check for any NA or infinite values and remove them
filteredCountData_cellcycle <- filteredCountData_cellcycle[complete.cases(filteredCountData_cellcycle), ]

# Ensure the data is not empty after filtering
if (nrow(filteredCountData_cellcycle) == 0 || ncol(filteredCountData_cellcycle) == 0) {
  stop("Filtered data contains no rows or columns. Please check your data and filtering criteria.")
}

# Plot the heatmap for the filtered data using only Genotype annotations
pheatmap(filteredCountData_cellcycle, 
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         annotation_col = manual_annotation_filtered, 
         show_rownames = TRUE, 
         scale = 'row',
         annotation_colors = ann_colors)

In [ ]:
suppressPackageStartupMessages({
  library(pheatmap)
  library(grid)
})

###########################
## Output directory
###########################
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

###########################
## Annotation colors (Genotype only)
###########################
ann_colors <- list(
  Genotype = c("wt" = "#05d7df", "mut" = "#d6a3eb")
)

###########################
## Read cell-cycle gene lists
###########################
cell_cycle_lmc <- read.table("lmc_cellcyclegenes.txt", header = TRUE)
cell_cycle_mef <- read.table("mef_cellcyclegenes.txt", header = TRUE)

colnames(cell_cycle_lmc)[1] <- "Gene_name"
colnames(cell_cycle_mef)[1] <- "Gene_name"

cell_cycle_lmc_combo <- rbind(cell_cycle_lmc, cell_cycle_mef)
cell_cycle_lmc_combo$Gene_name <- toupper(cell_cycle_lmc_combo$Gene_name)

###########################
## Filter countData
###########################
rownames(countData) <- toupper(rownames(countData))

cell_cycle_res <- intersect(rownames(countData), cell_cycle_lmc_combo$Gene_name)
filteredCountData_cellcycle <- countData[cell_cycle_res, , drop = FALSE]

###########################
## Remove LPS samples
###########################
samples_to_keep <- manual_annotation$Condition != "LPS"

filteredCountData_cellcycle <- filteredCountData_cellcycle[, samples_to_keep, drop = FALSE]
manual_annotation_filtered <- manual_annotation[samples_to_keep, "Genotype", drop = FALSE]

rownames(manual_annotation_filtered) <- colnames(filteredCountData_cellcycle)

###########################
## Remove NA / infinite rows
###########################
filteredCountData_cellcycle <- filteredCountData_cellcycle[
  complete.cases(filteredCountData_cellcycle),
  ,
  drop = FALSE
]

if (nrow(filteredCountData_cellcycle) == 0 || ncol(filteredCountData_cellcycle) == 0) {
  stop("Filtered data contains no rows or columns. Please check filtering.")
}

###########################
## Plot + SAVE AS SVG
###########################
ph <- pheatmap(
  filteredCountData_cellcycle,
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  annotation_col = manual_annotation_filtered,
  show_rownames = TRUE,
  scale = "row",
  annotation_colors = ann_colors
)

svg(
  file.path(out_dir, "heatmap_cellcycle_GenotypeOnly_rowScaled.svg"),
  width = 10,
  height = 10
)

grid.newpage()
grid.draw(ph$gtable)
dev.off()


In [ ]:
#Labelling selected genes in the volcano plot
label_genes <- c('Nfkbia','Ccl7','Tnf')

EnhancedVolcano(res2_I,
                lab = rownames(res2_I),
                x = 'log2FoldChange',
                y = 'padj',
                #selectLab = label_genes, 
                labSize = 5.0, 
                xlim = c(-8, 8),
               ylim = c(0,150))

In [ ]:
suppressPackageStartupMessages({
  library(EnhancedVolcano)
})

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Labelling selected genes in the volcano plot
label_genes <- c("Nfkbia", "Ccl7", "Tnf")

svg(
  filename = file.path(out_dir, "EnhancedVolcano_res2_I_selectedLabels.svg"),
  width = 8,
  height = 8
)

EnhancedVolcano(
  res2_I,
  lab = rownames(res2_I),
  x = "log2FoldChange",
  y = "padj",
  selectLab = label_genes,   # <-- label only these
  labSize = 5.0,
  xlim = c(-8, 8),
  ylim = c(0, 150)
)

dev.off()


In [ ]:
# Convert row names to uppercase to match case insensitivity
rownames(countData) <- toupper(rownames(countData))
rownames(res2_I_sig) <- toupper(rownames(res2_I_sig))
nf_kb$Gene_name <- toupper(nf_kb$Gene_name)  # Ensuring case consistency

# Find common genes between countData and res2_I_sig
common_genes_sig <- intersect(rownames(countData), rownames(res2_I_sig))

# Also intersect with nf_kb gene names
common_genes_nf_kb <- intersect(rownames(countData), nf_kb$Gene_name)

# Get the intersection of the two lists to find genes common in both
common_genes_final <- intersect(common_genes_sig, common_genes_nf_kb)

# Filter countData to include only rows that have genes present in both res2_I_sig and nf_kb
filteredCountData <- countData[common_genes_final, ]

# Further filter rows to exclude certain gene name prefixes
filtered_rows <- grep("^(?!GM|MIR|ENSMUS|RPL|RIK).*$", rownames(filteredCountData), perl = TRUE)
filteredCountDataFiltered <- filteredCountData[filtered_rows, ]

meanExpressionLevels <- rowMeans(filteredCountDataFiltered)

# Sort genes by mean expression levels in decreasing order and select the top 50
top50Genes <- names(sort(meanExpressionLevels, decreasing = TRUE))[1:50]

# Filter the data to include only these top 50 genes
top50FilteredData <- filteredCountDataFiltered[top50Genes, ]

# Plot the heatmap of the top 50 filtered countData
pheatmap(top50FilteredData,
         cluster_rows = FALSE,
         cluster_cols = FALSE,
         annotation_col = as.data.frame(manual_annotation),  # Ensure this is defined or modify as needed
         show_rownames = TRUE,
         show_colnames = FALSE,  # Hiding column names
         scale = "row",
         fontsize_row = 8)



In [ ]:
suppressPackageStartupMessages({
  library(pheatmap)
  library(grid)
})

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Convert row names to uppercase to match case insensitivity
rownames(countData) <- toupper(rownames(countData))
rownames(res2_I_sig) <- toupper(rownames(res2_I_sig))
nf_kb$Gene_name <- toupper(nf_kb$Gene_name)

# Find common genes between countData and res2_I_sig
common_genes_sig <- intersect(rownames(countData), rownames(res2_I_sig))

# Also intersect with nf_kb gene names
common_genes_nf_kb <- intersect(rownames(countData), nf_kb$Gene_name)

# Genes common to both lists
common_genes_final <- intersect(common_genes_sig, common_genes_nf_kb)

# Filter countData to include only those genes
filteredCountData <- countData[common_genes_final, , drop = FALSE]

# Further filter rows to exclude certain gene name prefixes + genes ending in RIK
filtered_rows <- grep("^(?!GM|MIR|ENSMUS|RPL).*(?<!RIK)$", rownames(filteredCountData), perl = TRUE)
filteredCountDataFiltered <- filteredCountData[filtered_rows, , drop = FALSE]

# Mean expression + top 50
meanExpressionLevels <- rowMeans(filteredCountDataFiltered, na.rm = TRUE)
top50Genes <- names(sort(meanExpressionLevels, decreasing = TRUE))[1:50]

top50FilteredData <- filteredCountDataFiltered[top50Genes, , drop = FALSE]

# Plot + capture heatmap
ph <- pheatmap(
  top50FilteredData,
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  annotation_col = as.data.frame(manual_annotation),
  show_rownames = TRUE,
  show_colnames = FALSE,
  scale = "row",
  fontsize_row = 8
)

# Save as SVG
svg_file <- file.path(out_dir, "heatmap_nfkb_res2I_sig_top50_rowScaled.svg")
svg(svg_file, width = 10, height = 10)
grid.newpage()
grid.draw(ph$gtable)
dev.off()


II.The effect of treatment in mutant samples [This is for Sp3 knockout,treated compared with untreated ]   


3.Genes upregulated with LPS in Sp3 knockout samples

4.Genes downregulated with LPS in Sp3 knockout samples


In [ ]:
res <- results(dds,list(c("condition_LPS_vs_Control","genotypemut.conditionLPS") ))
ix = which.min(res$padj) # most significant
res <- res[order(res$padj),] # sort
kable(res[1:5,-(3:4)])

In [ ]:
barplot(assay(dds)[ix,],las=2, main=rownames(dds)[ ix  ]  )

In [ ]:
res2 <- res
write.csv(res2,"LMC_II.csv")
res2 <- na.omit(res2)


In [ ]:
res2_II <- res2[res2$padj < 0.05, ]
res2_II_df <- data.frame(res2_II)

In [ ]:
ggplot(res2_II_df, aes(x = baseMean, y = log2FoldChange)) +
  geom_point(alpha = 0.6) + # Set alpha for better visualization if points overlap
  scale_x_log10() + # Log-transform the baseMean axis for better visualization
  theme_minimal() +
  labs(x = "Base Mean (log scale)", 
       y = "Log2 Fold Change", 
       title = "Scatter Plot of baseMean vs. log2FoldChange") +
  geom_hline(yintercept = 0, linetype = "dashed", color = "red") # Add a horizontal line at y=0

In [ ]:
library(ggplot2)

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Create the plot
p <- ggplot(res2_II_df, aes(x = baseMean, y = log2FoldChange)) +
  geom_point(alpha = 0.6) +
  scale_x_log10() +
  theme_minimal() +
  labs(
    x = "Base Mean (log scale)",
    y = "Log2 Fold Change",
    title = "Scatter Plot of baseMean vs. log2FoldChange"
  ) +
  geom_hline(yintercept = 0, linetype = "dashed", color = "red")

# Save as SVG
ggsave(
  filename = file.path(out_dir, "scatter_baseMean_vs_log2FC_res2_II.svg"),
  plot = p,
  device = "svg",
  width = 7,
  height = 6,
  units = "in"
)


In [ ]:
res2_II$abs_LFC <- abs(res2_II$log2FoldChange)
res2_II_sig <- res2_II[res2_II$abs_LFC > 1, ]
res2_II_sig

In [ ]:
#reset par
par(mfrow=c(1,1))
# Make a basic volcano plot
with(res2, plot(log2FoldChange, -log10(padj), pch=20, main="Volcano plot", xlim=c(-5,10)))

# Add colored points: blue if padj<0.05, red if log2FC>1 and padj<0.05)
with(subset(res2, padj<.05 ), points(log2FoldChange, -log10(padj), pch=20, col="blue"))
with(subset(res2, padj<.05 & abs(log2FoldChange)>1), points(log2FoldChange, -log10(padj), pch=20, col="dark green"))

Here above plot suggests upregulated genes(positive Foldchange) and downregulated genes(negative Foldchange) in mutant samples when subjected to LPS treatment

In [ ]:
res3 <- res2[complete.cases(res2),]
res3$abs_LFC <- abs(res3$log2FoldChange)
res4 <- res3[res3$abs_LFC > 1, ]
res5 <- res4[res4$padj < 0.05, ] 

res2_II_up <- res2[res2$log2FoldChange > 1 & res2$padj < 0.05, ]
res2_II_down <- res2[res2$log2FoldChange < -1 & res2$padj < 0.05, ]




write.csv(res2_II,'results_DEgenes_LMC_II_2020countsdata.csv')
write.csv(rownames(res2_II_up),'upregulated_genes_LMC_II.csv')
write.csv(rownames(res2_II_down),'downregulated_genes_LMC_II.csv')

write.csv(res2_II_up,'upregulated_genes_LMC_II_rf.csv')
write.csv(res2_II_down,'downregulated_genes_LMC_II_rf.csv')


In [ ]:
#Labelling selected genes in the volcano plot
label_genes <- c('Nfkbia','Ccl7','Plscr1')

EnhancedVolcano(res2_II,
                lab = rownames(res2_II),
                x = 'log2FoldChange',
                y = 'padj',
              #  selectLab = label_genes, 
                labSize = 5.0, 
                xlim = c(-8, 8),ylim = c(0,150))

In [ ]:
suppressPackageStartupMessages({
  library(EnhancedVolcano)
})

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Labelling selected genes in the volcano plot
label_genes <- c("Nfkbia", "Ccl7", "Plscr1")

# Save as SVG
svg(
  filename = file.path(out_dir, "EnhancedVolcano_res2_II_selectedLabels.svg"),
  width = 8,
  height = 8
)

EnhancedVolcano(
  res2_II,
  lab = rownames(res2_II),
  x = "log2FoldChange",
  y = "padj",
  selectLab = label_genes,   # label only these genes
  labSize = 5.0,
  xlim = c(-8, 8),
  ylim = c(0, 150)
)

dev.off()


In [ ]:
# Assuming res2_I and res2_II are your data frames
# Convert row names to a column for merging
res2_I_df <- data.frame(res2_I)
res2_II_df <- data.frame(res2_II)

In [ ]:
filtered_res2_I_df <- res2_I_df[rownames(res2_I_df) == "Il1b", ]
filtered_res2_II_df <- res2_II_df[rownames(res2_II_df) == "Il1b", ]

In [ ]:
filtered_res2_I_df

In [ ]:
filtered_res2_II_df

In [ ]:
filtered_res2_II_df

In [ ]:
filtered_res2_I_df

In [ ]:
filtered_res2_I_df <- res2_I_df[rownames(res2_I_df) == "Il12b", ]
filtered_res2_II_df <- res2_II_df[rownames(res2_II_df) == "Il12b", ]

In [ ]:
# Convert row names to uppercase to match case insensitivity
rownames(countData) <- toupper(rownames(countData))
rownames(res2_II_sig) <- toupper(rownames(res2_II_sig))
nf_kb$Gene_name <- toupper(nf_kb$Gene_name)  # Ensuring case consistency

common_genes_sig_II <- intersect(rownames(countData), rownames(res2_II_sig))


# Also intersect with nf_kb gene names
common_genes_nf_kb <- intersect(rownames(countData), nf_kb$Gene_name)

# Get the intersection of the two lists to find genes common in both
common_genes_final <- intersect(common_genes_sig, common_genes_nf_kb)

# Filter countData to include only rows that have genes present in both res2_I_sig and nf_kb
filteredCountData <- countData[common_genes_final, ]

# Further filter rows to exclude certain gene name prefixes
filtered_rows <- grep("^(?!GM|MIR|ENSMUS|RPL|RIK).*$", rownames(filteredCountData), perl = TRUE)
filteredCountDataFiltered <- filteredCountData[filtered_rows, ]

meanExpressionLevels <- rowMeans(filteredCountDataFiltered)

# Sort genes by mean expression levels in decreasing order and select the top 50
top50Genes <- names(sort(meanExpressionLevels, decreasing = TRUE))[1:50]

# Filter the data to include only these top 50 genes
top50FilteredData <- filteredCountDataFiltered[top50Genes, ]

# Plot the heatmap of the top 50 filtered countData
pheatmap(top50FilteredData,
         cluster_rows = FALSE,
         cluster_cols = FALSE,
         annotation_col = as.data.frame(manual_annotation),  # Ensure this is defined or modify as needed
         show_rownames = TRUE,
         show_colnames = FALSE,  # Hiding column names
         scale = "row",
         fontsize_row = 8)

In [ ]:
length(common_genes_final)

In [ ]:
# Convert row names to uppercase to match case insensitivity
rownames(countData) <- toupper(rownames(countData))
rownames(res2_I_sig) <- toupper(rownames(res2_I_sig))
rownames(res2_II_sig) <- toupper(rownames(res2_II_sig))
nf_kb$Gene_name <- toupper(nf_kb$Gene_name)  # Ensuring case consistency

# Intersect gene names across different datasets
common_genes_I_sig <- intersect(rownames(countData), rownames(res2_I_sig))
common_genes_II_sig <- intersect(rownames(countData), rownames(res2_II_sig))
common_genes_nf_kb <- intersect(rownames(countData), nf_kb$Gene_name)

# Combine intersections to find genes common across all conditions
common_genes_final <- Reduce(intersect, list(common_genes_I_sig, common_genes_II_sig, common_genes_nf_kb))

# Filter countData to include only rows that have genes present in all intersections
filteredCountData <- countData[common_genes_final, ]

# Further filter rows to exclude certain gene name prefixes
filtered_rows <- grep("^(?!GM|MIR|ENSMUS|RPL|RIK).*$", rownames(filteredCountData), perl = TRUE)
filteredCountDataFiltered <- filteredCountData[filtered_rows, ]

# Calculate mean expression levels
meanExpressionLevels <- rowMeans(filteredCountDataFiltered)

# Sort genes by mean expression levels in decreasing order and select the top 50
top50Genes <- names(sort(meanExpressionLevels, decreasing = TRUE))#[1:50]

# Filter the data to include only these top 50 genes
top50FilteredData <- filteredCountDataFiltered[top50Genes, ]

# Optional: Define or calculate any specific annotation for columns if needed
# manual_annotation <- data.frame(Condition = c("Control", "Treatment"), row.names = colnames(top50FilteredData))

# Plot the heatmap of the top 50 filtered countData
pheatmap(top50FilteredData,
         cluster_rows = FALSE,
         cluster_cols = FALSE,
         annotation_col = as.data.frame(manual_annotation),  # Ensure this is defined or modify as needed
         show_rownames = TRUE,
         show_colnames = FALSE,  # Hiding column names
         scale = "row",
         fontsize_row = 6,
         main = "")

In [ ]:
# Extract log2FC for the top 50 genes from res2_I_sig and res2_II_sig
# Ensuring only matching rows are selected and handling potential NA values
log2FC_I <- res2_I_sig[match(rownames(top50FilteredData), rownames(res2_I_sig)), "log2FoldChange", drop = FALSE]
log2FC_II <- res2_II_sig[match(rownames(top50FilteredData), rownames(res2_II_sig)), "log2FoldChange", drop = FALSE]


In [ ]:
colnames(log2FC_I)[1] <- "log2FoldChange_I"
colnames(log2FC_II)[1] <- "log2FoldChange_II"
log2FC_I$Gene = rownames(log2FC_I)
log2FC_II$Gene = rownames(log2FC_II)

log2FC_II <- data.frame(log2FC_II)
log2FC_I <- data.frame(log2FC_I)

# Merging the data frames on the 'Gene' column
merged_data = merge(log2FC_I, log2FC_II, by = "Gene", all = TRUE)  # all = TRUE is like a full join



# View the merged data frame
print(merged_data)
rownames(merged_data) <- merged_data$Gene
merged_data$Gene <- NULL

In [ ]:
pheatmap(merged_data, cluster_cols = FALSE,cluster_rows = FALSE,fontsize_row = 7,cellwidth = 50,show_colnames = FALSE)

In [ ]:
merged_data$Gene_name <- rownames(merged_data)

In [ ]:
# Ensure ggplot2 is loaded
library(ggplot2)

# Convert data from wide to long format for easier plotting with ggplot2
merged_data_long <- reshape2::melt(merged_data, id.vars = "Gene_name", variable.name = "Condition", value.name = "Log2FoldChange")

# Plot using ggplot2
p <- ggplot(merged_data_long, aes(x = Gene_name, y = Log2FoldChange, fill = Condition)) +
    geom_bar(stat = "identity", position = position_dodge()) +
    theme_minimal() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
    labs(x = "Gene", y = "Log2 Fold Change", title = "Comparison of Log2 Fold Changes across Conditions")

# Display the plot
print(p)


In [ ]:
library(ggplot2)
library(reshape2)

# Convert the data from wide to long format
merged_data_long <- melt(merged_data, id.vars = "Gene_name", variable.name = "Condition", value.name = "Log2FoldChange")

# Create a connected scatter plot
p_connected <- ggplot(merged_data_long, aes(x = Condition, y = Log2FoldChange, group = Gene_name)) +
  geom_line(aes(color = Gene_name), size = 1) +  # Connect lines between conditions for each gene
  geom_point(size = 3, aes(color = Gene_name)) +  # Add points for each condition
  theme_minimal() +
  labs(x = "Condition", y = "Log2 Fold Change", title = "Connected Scatter Plot of Log2 Fold Changes")

# Display the plot
print(p_connected)


In [ ]:
# Convert to a suitable format for a lollipop chart
merged_data_wide <- dcast(merged_data_long, Gene_name ~ Condition, value.var = "Log2FoldChange")

# Create a lollipop chart
p_lollipop <- ggplot(merged_data_wide, aes(x = Gene_name)) +
  geom_segment(aes(xend = Gene_name, y = log2FoldChange_I, yend = log2FoldChange_II), col = "grey") +
  geom_point(aes(y = log2FoldChange_I), color = 'blue', size = 3) +
  geom_point(aes(y = log2FoldChange_II), color = 'red', size = 3) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(x = "Gene", y = "Log2 Fold Change", title = "Lollipop Chart of Log2 Fold Changes")

# Display the plot
print(p_lollipop)


In [ ]:
library(ggplot2)
library(reshape2)
library(dplyr)

# Assuming merged_data_wide is already created and contains columns: Gene, log2FoldChange_I, log2FoldChange_II

# Create subsets of the data
increased_genes <- merged_data_wide %>% 
  filter(log2FoldChange_I < log2FoldChange_II)

decreased_genes <- merged_data_wide %>% 
  filter(log2FoldChange_I > log2FoldChange_II)


In [ ]:
# Lollipop chart for increased genes
p_lollipop_increased <- ggplot(increased_genes, aes(x = Gene_name)) +
  geom_segment(aes(xend = Gene_name, y = log2FoldChange_I, yend = log2FoldChange_II), col = "grey") +
  geom_point(aes(y = log2FoldChange_I), color = 'blue', size = 3) +
  geom_point(aes(y = log2FoldChange_II), color = 'red', size = 3) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1)) +
  labs(x = "Gene", y = "Log2 Fold Change", title = "Increase in Log2 Fold Changes")

# Display the plot
print(p_lollipop_increased)


In [ ]:
library(ggplot2)

# Creating the lollipop chart for increased genes with specified modifications
p_lollipop_increased <- ggplot(increased_genes, aes(x = Gene_name)) +
  geom_segment(aes(xend = Gene_name, y = log2FoldChange_I, yend = log2FoldChange_II), col = "grey") +
  geom_point(aes(y = log2FoldChange_I), color = 'black', size = 3, stroke = 0.5, shape = 21, fill = '#ff9288') +  # Specify shape and fill for a border
  geom_point(aes(y = log2FoldChange_II), color = 'black', size = 3, stroke = 0.5, shape = 21, fill = '#96cb01') + # Specify shape and fill for a border
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 8),
        axis.text.y = element_text(size = 7),
        axis.line = element_line(size = 0.2),   # Makes axis lines thicker
        panel.grid.major = element_blank(),    # Removes major grid lines
        panel.grid.minor = element_blank(),    # Removes minor grid lines
        plot.title = element_text(size = 14, face = "bold")) +
  labs(x = "Gene", y = "Log2 Fold Change", title = "Increase in Log2 Fold Changes")

# Display the plot
print(p_lollipop_increased)

# Save the plot to a file with specified dimensions
ggsave("Increased_Lollipop_Plot.png", plot = p_lollipop_increased, width = 12, height = 8, dpi = 300)


In [ ]:
library(ggplot2)

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Creating the lollipop chart
p_lollipop_increased <- ggplot(increased_genes, aes(x = Gene_name)) +
  geom_segment(
    aes(xend = Gene_name, y = log2FoldChange_I, yend = log2FoldChange_II),
    color = "grey"
  ) +
  geom_point(
    aes(y = log2FoldChange_I),
    color = "black",
    size = 3,
    stroke = 0.5,
    shape = 21,
    fill = "#ff9288"
  ) +
  geom_point(
    aes(y = log2FoldChange_II),
    color = "black",
    size = 3,
    stroke = 0.5,
    shape = 21,
    fill = "#96cb01"
  ) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 8),
    axis.text.y = element_text(size = 7),
    axis.line = element_line(size = 0.2),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    plot.title = element_text(size = 14, face = "bold")
  ) +
  labs(
    x = "Gene",
    y = "Log2 Fold Change",
    title = "Increase in Log2 Fold Changes"
  )

# Display the plot
print(p_lollipop_increased)

# Save as SVG (vector)
ggsave(
  filename = file.path(out_dir, "Increased_Lollipop_Plot.svg"),
  plot = p_lollipop_increased,
  device = "svg",
  width = 12,
  height = 8,
  units = "in"
)


In [ ]:
decreased_genes

In [ ]:
increased_genes

In [ ]:
library(ggplot2)

# Creating the lollipop chart for increased genes with specified modifications
p_lollipop_decreased <- ggplot(decreased_genes, aes(x = Gene_name)) +
  geom_segment(aes(xend = Gene_name, y = log2FoldChange_I, yend = log2FoldChange_II), col = "grey") +
  geom_point(aes(y = log2FoldChange_I), color = 'black', size = 3, stroke = 0.5, shape = 21, fill = '#ff9288') +  # Specify shape and fill for a border
  geom_point(aes(y = log2FoldChange_II), color = 'black', size = 3, stroke = 0.5, shape = 21, fill = '#96cb01') + # Specify shape and fill for a border
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 12),
        axis.text.y = element_text(size = 7),
        axis.line = element_line(size = 0.2),   # Makes axis lines thicker
        panel.grid.major = element_blank(),    # Removes major grid lines
        panel.grid.minor = element_blank(),    # Removes minor grid lines
        plot.title = element_text(size = 14, face = "bold")) +
  labs(x = "Gene", y = "Log2 Fold Change", title = "Increase in Log2 Fold Changes")

# Display the plot
print(p_lollipop_decreased)

# Save the plot to a file with specified dimensions
ggsave("Decreased_Lollipop_Plot_lmc_v1.png", plot = p_lollipop_decreased, width = 12, height = 8, dpi = 300)


In [ ]:
library(ggplot2)

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Creating the lollipop chart for decreased genes
p_lollipop_decreased <- ggplot(decreased_genes, aes(x = Gene_name)) +
  geom_segment(
    aes(xend = Gene_name, y = log2FoldChange_I, yend = log2FoldChange_II),
    color = "grey"
  ) +
  geom_point(
    aes(y = log2FoldChange_I),
    color = "black",
    size = 3,
    stroke = 0.5,
    shape = 21,
    fill = "#ff9288"
  ) +
  geom_point(
    aes(y = log2FoldChange_II),
    color = "black",
    size = 3,
    stroke = 0.5,
    shape = 21,
    fill = "#96cb01"
  ) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 12),
    axis.text.y = element_text(size = 7),
    axis.line = element_line(size = 0.2),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    plot.title = element_text(size = 14, face = "bold")
  ) +
  labs(
    x = "Gene",
    y = "Log2 Fold Change",
    title = "Decrease in Log2 Fold Changes"
  )

# Display the plot
print(p_lollipop_decreased)

# Save as SVG (vector)
ggsave(
  filename = file.path(out_dir, "Decreased_Lollipop_Plot_lmc_v1.svg"),
  plot = p_lollipop_decreased,
  device = "svg",
  width = 12,
  height = 8,
  units = "in"
)


In [ ]:
library(ggplot2)

# Data Preparation (Assuming 'increased_genes' is your data frame)
# increased_genes <- your_data_frame

# Function to create the lollipop chart
create_lollipop_plot <- function(data, orientation = "vertical") {
  p <- ggplot(data, aes(x = Gene_name)) +
    geom_segment(aes(xend = Gene_name, y = log2FoldChange_I, yend = log2FoldChange_II), col = "grey") +
    geom_point(aes(y = log2FoldChange_I), color = 'black', size = 7, stroke = 0.5, shape = 21, fill = '#ff9288') +  # Specify shape and fill for a border
    geom_point(aes(y = log2FoldChange_II), color = 'black', size = 7, stroke = 0.5, shape = 21, fill = '#96cb01') + # Specify shape and fill for a border
    theme_minimal() +
    theme(
      axis.text.x = element_text(angle = ifelse(orientation == "vertical", 90, 0), hjust = 1, vjust = 0.5, size = 9),  # Adjust font size and angle for x-axis
      axis.text.y = element_text(size = ifelse(orientation == "vertical", 7, 10)),  # Adjust font size for y-axis
      axis.ticks.length = unit(0.05, "cm"), # Adjust the length of ticks to reduce space
      axis.line = element_line(size = 0.2),   # Makes axis lines thicker
      panel.grid.major = element_blank(),    # Removes major grid lines
      panel.grid.minor = element_blank(),    # Removes minor grid lines
      plot.title = element_text(size = 14, face = "bold"),
      plot.margin = if (orientation == "horizontal") margin(0.1, 0.1, 0.1, 0.1, "cm") else margin(2, 1, 2, 1, "cm") # Increase margins around the plot
    ) +
    labs(x = "Gene", y = "Log2 Fold Change", title = "Increase in Log2 Fold Changes") +
    scale_x_discrete(expand = expansion(mult = c(0.01, 0.01)))  # Reduce gaps between x-axis labels
  
  if (orientation == "horizontal") {
    p <- p + coord_flip()  # Flip coordinates for horizontal plot
  }
  
  return(p)
}

# Creating vertical lollipop plot
p_lollipop_increased_vertical <- create_lollipop_plot(increased_genes, orientation = "vertical")
print(p_lollipop_increased_vertical)

# Save the vertical plot to a file with specified dimensions
ggsave("Increased_Lollipop_Plot_Vertical_lmc.png", plot = p_lollipop_increased_vertical, width = 14, height = 8, dpi = 300)

# Creating horizontal lollipop plot
p_lollipop_increased_horizontal <- create_lollipop_plot(increased_genes, orientation = "horizontal")
print(p_lollipop_increased_horizontal)

# Save the horizontal plot to a file with specified dimensions
ggsave("Increased_Lollipop_Plot_Horizontal.png", plot = p_lollipop_increased_horizontal, width = 12, height = 8, dpi = 300)


In [ ]:
library(dplyr)

# Calculate the absolute differences and categorize them
categorized_genes <- decreased_genes %>%
  mutate(Difference = abs(log2FoldChange_I - log2FoldChange_II)) %>%
  mutate(Category = case_when(
    Difference >= 1.5   ~ "High Difference",
    Difference <= 0.5 ~ "Low Difference",
    TRUE              ~ "Moderate Difference"
  ))

categorized_genes <- categorized_genes %>%
  mutate(Gene_name = str_to_title(Gene_name))

write.csv(categorized_genes,"decreasing_genes_lmc_all_entries_lollipop_plot.csv")


In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(stringr)
  library(ggplot2)
})

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Calculate the absolute differences and categorize them
categorized_genes <- decreased_genes %>%
  mutate(
    Difference = abs(log2FoldChange_I - log2FoldChange_II),
    Category = case_when(
      Difference >= 1.5 ~ "High Difference",
      Difference <= 0.5 ~ "Low Difference",
      TRUE              ~ "Moderate Difference"
    ),
    Gene_name = str_to_title(Gene_name)
  )

# Write CSV
write.csv(
  categorized_genes,
  file = file.path(out_dir, "decreasing_genes_lmc_all_entries_lollipop_plot.csv"),
  row.names = FALSE
)

# (Optional but helpful) order genes by Difference so the plot is interpretable
categorized_genes <- categorized_genes %>%
  arrange(desc(Difference)) %>%
  mutate(Gene_name = factor(Gene_name, levels = Gene_name))

# Create lollipop plot (decreased genes), colored by Category
p_lollipop_decreased_categorized <- ggplot(categorized_genes, aes(x = Gene_name)) +
  geom_segment(aes(xend = Gene_name, y = log2FoldChange_I, yend = log2FoldChange_II), color = "grey") +
  geom_point(aes(y = log2FoldChange_I), color = "black", size = 3, stroke = 0.5, shape = 21, fill = "#ff9288") +
  geom_point(aes(y = log2FoldChange_II), color = "black", size = 3, stroke = 0.5, shape = 21, fill = "#96cb01") +
  geom_point(aes(y = (log2FoldChange_I + log2FoldChange_II) / 2, color = Category), size = 2) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 10),
    axis.text.y = element_text(size = 8),
    axis.line = element_line(size = 0.2),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    plot.title = element_text(size = 14, face = "bold")
  ) +
  labs(
    x = "Gene",
    y = "Log2 Fold Change",
    title = "Decreased genes: Log2FC (I vs II) with difference categories",
    color = "Category"
  )

print(p_lollipop_decreased_categorized)

# Save as SVG
ggsave(
  filename = file.path(out_dir, "Decreased_Lollipop_Categorized.svg"),
  plot = p_lollipop_decreased_categorized,
  device = "svg",
  width = 12,
  height = 8,
  units = "in"
)


In [ ]:
# Optionally, filter and view genes with high, low, or moderate differences
high_difference_genes <- filter(categorized_genes, Category == "High Difference")
low_difference_genes <- filter(categorized_genes, Category == "Low Difference")
moderate_difference_genes <- filter(categorized_genes, Category == "Moderate Difference")

In [ ]:
write.table(high_difference_genes$Gene_name,"decreasing_genes_lmc_high_difference_genes_lollipop_plot.txt", row.names = FALSE, col.names = FALSE, quote = FALSE)
write.table(low_difference_genes$Gene_name,"decreasing_genes_lmc_low_difference_genes_lollipop_plot.txt", row.names = FALSE, col.names = FALSE, quote = FALSE)
write.table(moderate_difference_genes$Gene_name,"decreasing_genes_lmc_moderate_difference_genes_lollipop_plot.txt", row.names = FALSE, col.names = FALSE, quote = FALSE)

In [ ]:
library(dplyr)

# Calculate the absolute differences and categorize them
categorized_genes <- increased_genes %>%
  mutate(Difference = abs(log2FoldChange_I - log2FoldChange_II)) %>%
  mutate(Category = case_when(
    Difference >= 1.5   ~ "High Difference",
    Difference <= 0.5 ~ "Low Difference",
    TRUE              ~ "Moderate Difference"
  ))

categorized_genes <- categorized_genes %>%
  mutate(Gene_name = str_to_title(Gene_name))

write.csv(categorized_genes,"increasing_genes_lmc_all_entries_lollipop_plot.csv")

# Optionally, filter and view genes with high, low, or moderate differences
high_difference_genes <- filter(categorized_genes, Category == "High Difference")
low_difference_genes <- filter(categorized_genes, Category == "Low Difference")
moderate_difference_genes <- filter(categorized_genes, Category == "Moderate Difference")



write.table(high_difference_genes$Gene_name,"increasing_genes_lmc_high_difference_genes_lollipop_plot.txt", row.names = FALSE, col.names = FALSE, quote = FALSE)
write.table(low_difference_genes$Gene_name,"increasing_genes_lmc_low_difference_genes_lollipop_plot.txt", row.names = FALSE, col.names = FALSE, quote = FALSE)
write.table(moderate_difference_genes$Gene_name,"increasing_genes_lmc_moderate_difference_genes_lollipop_plot.txt", row.names = FALSE, col.names = FALSE, quote = FALSE)

In [ ]:
# Load necessary libraries
library(ggplot2)

# Data preparation
countData2 <- countData1
rownames(countData2) <- toupper(countData2$Gene_Name)
countData2$Gene_Name <- NULL

# Define the gene list
genes_to_plot <- toupper(moderate_difference_genes$Gene_name)

# Function to create plots
create_gene_plot <- function(gene_name) {
  if (!(gene_name %in% rownames(countData2))) {
    message(paste("Gene", gene_name, "not found in countData2"))
    return(NULL)
  }
  
  gene_expression <- countData2[gene_name, ]  # Extracting expression levels for particular gene
  
  plot_data <- data.frame(
    Sample = colnames(countData2),
    Expression = as.numeric(gene_expression),
    Group = groups
  )
  
  # Create Genotype and Condition columns based on the Group column
  plot_data$Genotype <- ifelse(grepl("wt", plot_data$Group), "WT", "MUT")
  plot_data$Condition <- ifelse(grepl("Control", plot_data$Group), "Control", "LPS")
  
  # Create a new column for the combined labels
  plot_data$Group_Combined <- with(plot_data, paste(Genotype, Condition, sep = "+"))
  
  # Define the colors
  genotype_colors <- c("WT" = "#EC3A91", "MUT" = "#71980E")
  condition_colors <- c("Control" = "#B0DBFF", "LPS" = "#CF98E2")
  
  # Create the plot
  p <- ggplot(plot_data, aes(x = Group_Combined, y = Expression)) +
    geom_point(aes(fill = Condition, color = Genotype, shape = Genotype), size = 9, stroke = 2) +
    scale_fill_manual(name = "Condition", values = condition_colors, guide = guide_legend(override.aes = list(shape = 21, size = 9, stroke = 0))) +
    scale_color_manual(name = "Genotype", values = genotype_colors, labels = c("WT" = "Wild-type", "MUT" = "Mutant")) +
    scale_shape_manual(name = "Genotype", values = c("WT" = 21, "MUT" = 22), labels = c("WT" = "Wild-type", "MUT" = "Mutant")) +
    scale_x_discrete(limits = c("WT+Control", "WT+LPS", "MUT+Control", "MUT+LPS")) +
    labs(
      title = paste("Expression of", gene_name, "Across Groups"),
      x = "",
      y = "Expression Level"
    ) +
    theme_minimal() +
    theme(
      axis.text.x = element_text(size = 14, color = "black"),
      axis.text.y = element_text(size = 14, color = "black"),
      axis.title.x = element_text(size = 16),
      axis.title.y = element_text(size = 16),
      axis.ticks = element_line(color = "black"),
      axis.line = element_line(color = "black"),
      panel.background = element_blank(),
      panel.border = element_rect(color = "black", fill = NA),
      panel.grid.major = element_blank(),
      panel.grid.minor = element_blank(),
      legend.position = "right",
      legend.text = element_text(size = 12),
      legend.title = element_text(size = 14),
      legend.key = element_blank()
    )
  
  return(p)
}

# Loop through each gene and create a plot
plots <- list()
for (gene in genes_to_plot) {
  message(paste("Creating plot for gene:", gene))
  p <- create_gene_plot(gene)
  if (!is.null(p)) {
    print(p)
    plots[[gene]] <- p
  }
}

# Optionally, save plots to files
for (gene in names(plots)) {
  ggsave(filename = paste0("plot_", gene, ".png"), plot = plots[[gene]], width = 8, height = 6)
}



In [ ]:
countData2

In [ ]:
# Plot using ggplot2
p_line <- ggplot(merged_data_long, aes(x = Condition, y = Log2FoldChange, group = Gene_name, color = Gene_name)) +
    geom_line() +
    geom_point(size = 3) +
    theme_minimal() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
    labs(x = "Condition", y = "Log2 Fold Change", title = "Gene Expression Changes Across Conditions")

# Display the plot
print(p_line)


In [ ]:
library(ggplot2)
library(reshape2)  # Ensure this is loaded for data manipulation

# Convert the data from wide to long format
merged_data_long <- melt(merged_data, id.vars = "Gene_name", variable.name = "Condition", value.name = "Log2FoldChange")

# Create a violin plot
p_violin <- ggplot(merged_data_long, aes(x = Condition, y = Log2FoldChange, fill = Condition)) +
  geom_violin(trim = FALSE) +
  geom_dotplot(binaxis = 'y', stackdir = 'center', dotsize = 0.5, fill = "white") +  # Optional: add dotplot for more detail
  theme_minimal() +
  labs(x = "Condition", y = "Log2 Fold Change", title = "Distribution of Log2 Fold Changes Across Conditions")

# Display the plot
print(p_violin)


In [ ]:
# Create a dot plot
p_dot <- ggplot(merged_data_long, aes(x = Condition, y = Log2FoldChange, color = Condition)) +
  geom_dotplot(binaxis = 'y', stackdir = 'center', stackratio = 1.0, dotsize = 1) +
  theme_minimal() +
  labs(x = "Condition", y = "Log2 Fold Change", title = "Dot Plot of Log2 Fold Changes Across Conditions")

# Display the plot
print(p_dot)


In [ ]:
log2FC_I

In [ ]:
res = results(dds, contrast=c("genotype","mut","wt"))
ix = which.min(res$padj) # most significant
res <- res[order(res$padj),] # sort
kable(res[1:5,-(3:4)])

In [ ]:
barplot(assay(dds)[ix,],las=2, main=rownames(dds)[ ix  ]  )

In [ ]:
res2 <- res
write.csv(res2,"LMC_III.csv")
res2 <- na.omit(res2)


In [ ]:
res2_III <- res2[res2$padj < 0.05, ]

In [ ]:
#reset par
par(mfrow=c(1,1))
# Make a basic volcano plot
with(res2, plot(log2FoldChange, -log10(padj), pch=20, main="Volcano plot", xlim=c(-5,10)))

# Add colored points: blue if padj<0.05, red if log2FC>1 and padj<0.05)
with(subset(res2, padj<.05 ), points(log2FoldChange, -log10(padj), pch=20, col="blue"))
with(subset(res2, padj<.05 & abs(log2FoldChange)>1), points(log2FoldChange, -log10(padj), pch=20, col="dark green"))

In [ ]:
# Number of genes you want to label
N <- 10  

# Sort your data frame based on the criteria (e.g., padj)
sorted_indices <- order(res2$padj)

# Initialize labels to NA
res2$labels <- NA

# Select top N genes based on sorted indices
top_N_indices <- sorted_indices[1:N]

# Identify those among the top N that need to be filtered out
filtered_indices <- top_N_indices[grepl("^Gm|Rik$|Mir", res2$SYMBOL[top_N_indices])]

# Identify how many more genes we need to pick
num_more <- length(filtered_indices)

# Pick the next top-most genes that are valid
next_valid_indices <- sorted_indices[(N + 1):(N + num_more)]
next_valid_indices <- next_valid_indices[!grepl("^Gm|Rik$|Mir", res2$SYMBOL[next_valid_indices])]

# Combine the valid top indices and the next valid indices to get top N labels
final_top_N_indices <- c(setdiff(top_N_indices, filtered_indices), next_valid_indices[1:num_more])

# Filter out NA indices
final_top_N_indices <- final_top_N_indices[!is.na(final_top_N_indices)]
final_top_N_indices <- final_top_N_indices[!is.na(res2$SYMBOL[final_top_N_indices])]

# Check if any of the final_top_N_indices are NA or if corresponding SYMBOLs are NA
if (any(is.na(final_top_N_indices)) || any(is.na(res2$SYMBOL[final_top_N_indices]))) {
    stop("NA indices detected. Please resolve before proceeding.")
}

# Set the labels for these final top N indices
res2$labels[final_top_N_indices] <- res2$SYMBOL[final_top_N_indices]

# Create the EnhancedVolcano plot
EnhancedVolcano(res2,
                lab = rownames(res2),
                x = 'log2FoldChange',
                y = 'padj')


In [ ]:
suppressPackageStartupMessages({
  library(EnhancedVolcano)
})

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Number of genes you want to label
N <- 10  

# Sort based on padj (smallest first)
sorted_indices <- order(res2$padj)

# Initialize labels to NA
res2$labels <- NA

# Select top N genes
top_N_indices <- sorted_indices[1:N]

# Filter out undesired names among top N
filtered_indices <- top_N_indices[grepl("^Gm|Rik$|Mir", res2$SYMBOL[top_N_indices])]

# How many more do we need
num_more <- length(filtered_indices)

# Pick next valid ones
next_valid_indices <- sorted_indices[(N + 1):(N + num_more)]
next_valid_indices <- next_valid_indices[!grepl("^Gm|Rik$|Mir", res2$SYMBOL[next_valid_indices])]

# Final indices for labeling
final_top_N_indices <- c(setdiff(top_N_indices, filtered_indices), next_valid_indices[1:num_more])

# Remove NA indices / NA symbols
final_top_N_indices <- final_top_N_indices[!is.na(final_top_N_indices)]
final_top_N_indices <- final_top_N_indices[!is.na(res2$SYMBOL[final_top_N_indices])]

if (any(is.na(final_top_N_indices)) || any(is.na(res2$SYMBOL[final_top_N_indices]))) {
  stop("NA indices detected. Please resolve before proceeding.")
}

# Set the labels
res2$labels[final_top_N_indices] <- res2$SYMBOL[final_top_N_indices]

# Save as SVG
svg(file.path(out_dir, "EnhancedVolcano_res2_top10_filteredLabels.svg"), width = 8, height = 8)

EnhancedVolcano(
  res2,
  lab = res2$labels,           # <-- uses your selected labels
  x = "log2FoldChange",
  y = "padj"
)

dev.off()


Here above results suggest upregulated genes(positive Foldchange) and downregulated genes(negative Foldchange)in
Wildtype samples compared to mutant without considering control/LPS treatment.


In [ ]:
res3 <- res2[complete.cases(res2),]
res3$abs_LFC <- abs(res3$log2FoldChange)
res4 <- res3[res3$abs_LFC > 1, ]
res5 <- res4[res4$padj < 0.05, ]
res3_up <- res2[res2$log2FoldChange > 1 & res2$padj < 0.05, ]
res3_down <- res2[res2$log2FoldChange < -1 & res2$padj < 0.05, ]

#write.csv(res2_III,'results_DEgenes_LMC_III_2020countsdata.csv')
write.csv(rownames(res3_up),'upregulated_genes_LMC_III.csv')
write.csv(rownames(res3_down),'downregulated_genes_LMC_III.csv')

write.csv(res3_up,'upregulated_genes_LMC_III_rf.csv')
write.csv(res3_down,'downregulated_genes_LMC_III_rf.csv')

res3_up_v <- res2[res2$log2FoldChange > 0 & res2$padj < 0.05, ]
res3_down_v <- res2[res2$log2FoldChange < 0 & res2$padj < 0.05, ]




In [ ]:
countData2

In [ ]:
res2_III

In [ ]:
res3_up_MEF <- read.csv('upregulated_genes_MEF_III.csv')
res3_down_MEF <- read.csv('downregulated_genes_MEF_III.csv')
res3_up_LMC <- res3_up
res3_down_LMC <- res3_down

In [ ]:
res3_up_MEF$X <- NULL
rownames(res3_up_MEF) <- res3_up_MEF$x
res3_down_MEF$X <- NULL
rownames(res3_down_MEF) <- res3_down_MEF$x

In [ ]:
gene_list_I <- na.omit(rownames(res3_up_LMC))  
gene_list_II <- na.omit(rownames(res3_up_MEF))  


# Make a list of the gene lists
gene_lists <- list(gene_list_I, gene_list_II)

# Run the venn diagram function
venn.plot <- venn.diagram(
    x = gene_lists,
    filename = NULL,
    category.names = c("", ""),  # Set as empty strings
    output = TRUE,
    fill = c("red", "blue"),
    alpha = 0.5,
    fontface = "bold",
    fontsize = 20,
    cat.fontface = "bold",
    cat.fontsize = 20,
    lty = "solid",
    lwd = 2,
    label.col = "black",  # Set this to "black" to show numbers
    category.pos = NULL  # Set to NULL to avoid positioning non-existing labels
)
# Plot the diagram
grid.draw(venn.plot)

In [ ]:
gene_list_I <- na.omit(rownames(res3_down_LMC))  
gene_list_II <- na.omit(rownames(res3_down_MEF))  


# Make a list of the gene lists
gene_lists <- list(gene_list_I, gene_list_II)

# Run the venn diagram function
venn.plot <- venn.diagram(
    x = gene_lists,
    filename = NULL,
    category.names = c("", ""),  # Set as empty strings
    output = TRUE,
    fill = c("red", "blue"),
    alpha = 0.5,
    fontface = "bold",
    fontsize = 20,
    cat.fontface = "bold",
    cat.fontsize = 20,
    lty = "solid",
    lwd = 2,
    label.col = "black",  # Set this to "black" to show numbers
    category.pos = NULL  # Set to NULL to avoid positioning non-existing labels
)
# Plot the diagram
grid.draw(venn.plot)

In [ ]:
gene_list_I <- na.omit(rownames(res3_down_LMC))  
gene_list_II <- na.omit(rownames(res3_down_MEF))  


# Make a list of the gene lists
gene_lists <- list(gene_list_I, gene_list_II)

# Generate a Venn diagram without any labels or numbers
venn.plot <- venn.diagram(
    x = gene_lists,
    filename = NULL,
    category.names = c("list1", "list2"), 
    output = TRUE,
    fill = c("red", "blue"),
    alpha = 0.5,
    fontface = "bold",
    fontsize = 20,
    cat.fontface = "bold",
    cat.fontsize = 20,
    lty = "solid",
    lwd = 2,
    label.col = NA,  # Set this to NA to remove numbers
    category.pos = NULL  # Set to NULL to avoid positioning non-existing labels
)

# Plot the diagram
grid.draw(venn.plot)

In [ ]:
gene_list_I <- na.omit(rownames(res3_down_LMC))  
gene_list_II <- na.omit(rownames(res3_down_MEF))  


# Make a list of the gene lists
gene_lists <- list(gene_list_I, gene_list_II)

# Run the venn diagram function
venn.plot <- venn.diagram(
    x = gene_lists,
    filename = NULL,
    category.names = c("", ""),  # Set as empty strings
    output = TRUE,
    fill = c("red", "blue"),
    alpha = 0.5,
    fontface = "bold",
    fontsize = 20,
    cat.fontface = "bold",
    cat.fontsize = 20,
    lty = "solid",
    lwd = 2,
    label.col = "black",  # Set this to "black" to show numbers
    category.pos = NULL  # Set to NULL to avoid positioning non-existing labels
)
# Plot the diagram
grid.draw(venn.plot)

In [ ]:
###########################
## Load packages
###########################
suppressPackageStartupMessages({
  library(VennDiagram)
  library(grid)
})

###########################
## Prepare gene lists
###########################
gene_list_I  <- na.omit(rownames(res3_down_LMC))
gene_list_II <- na.omit(rownames(res3_down_MEF))

gene_lists <- list(
  LMC = gene_list_I,
  MEF = gene_list_II
)

###########################
## Output file
###########################
out_file <- "LMC_MEF_downregulated_venn1.svg"

###########################
## Open SVG device
###########################
svg(
  filename = out_file,
  width = 6,
  height = 6
)

grid.newpage()

###########################
## Create Venn diagram
###########################
venn.plot <- venn.diagram(
  x = gene_lists,
  filename = NULL,
  category.names = c("LMC", "MEF"),
  fill = c("red", "blue"),
  alpha = 0.5,
  lty = "solid",
  lwd = 2,

  # Remove counts
  label.col = NA,

  # Text styling
  cat.fontface = "bold",
  cat.fontsize = 18,

  # Prevent label positioning artifacts
  category.pos = NULL
)

###########################
## Draw + close
###########################
grid.draw(venn.plot)
dev.off()

cat("Saved SVG to:", out_file, "\n")


In [ ]:
gene_list_I <- na.omit(rownames(res3_up_LMC))  
gene_list_II <- na.omit(rownames(res3_up_MEF))  


# Make a list of the gene lists
gene_lists <- list(gene_list_I, gene_list_II)

# Generate a Venn diagram without any labels or numbers
venn.plot <- venn.diagram(
    x = gene_lists,
    filename = NULL,
    category.names = c("list1", "list2"), 
    output = TRUE,
    fill = c("red", "blue"),
    alpha = 0.5,
    fontface = "bold",
    fontsize = 20,
    cat.fontface = "bold",
    cat.fontsize = 20,
    lty = "solid",
    lwd = 2,
    label.col = NA,  # Set this to NA to remove numbers
    category.pos = NULL  # Set to NULL to avoid positioning non-existing labels
)

# Plot the diagram
grid.draw(venn.plot)

In [ ]:
###########################
## Load packages
###########################
suppressPackageStartupMessages({
  library(VennDiagram)
  library(grid)
})

###########################
## Prepare gene lists
###########################
gene_list_I  <- na.omit(rownames(res3_up_LMC))
gene_list_II <- na.omit(rownames(res3_up_MEF))

gene_lists <- list(
  LMC = gene_list_I,
  MEF = gene_list_II
)

###########################
## Output file
###########################
out_file <- "LMC_MEF_upregulated_venn1.svg"

###########################
## Open SVG device
###########################
svg(
  filename = out_file,
  width = 6,
  height = 6
)

grid.newpage()

###########################
## Create Venn diagram
###########################
venn.plot <- venn.diagram(
  x = gene_lists,
  filename = NULL,
  category.names = c("LMC", "MEF"),
  fill = c("red", "blue"),
  alpha = 0.5,
  lty = "solid",
  lwd = 2,

  # Remove counts
  label.col = NA,

  # Text styling
  cat.fontface = "bold",
  cat.fontsize = 18,

  # Avoid label positioning artifacts
  category.pos = NULL
)

###########################
## Draw + close
###########################
grid.draw(venn.plot)
dev.off()

cat("Saved SVG to:", out_file, "\n")


In [ ]:
suppressPackageStartupMessages({
  library(VennDiagram)
  library(grid)
})

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Gene lists
gene_list_I <- na.omit(rownames(res3_up_LMC))  
gene_list_II <- na.omit(rownames(res3_up_MEF))  

gene_lists <- list(gene_list_I, gene_list_II)

# Create Venn diagram object (do NOT write to file yet)
venn.plot <- venn.diagram(
  x = gene_lists,
  filename = NULL,
  category.names = c("LMC", "MEF"),
  fill = c("red", "blue"),
  alpha = 0.5,
  fontface = "bold",
  fontsize = 20,
  cat.fontface = "bold",
  cat.fontsize = 20,
  lty = "solid",
  lwd = 2,
  label.col = NA,        # remove numbers
  category.pos = NULL   # avoid repositioning labels
)

# Save as SVG
svg(
  filename = file.path(out_dir, "Venn_LMC_vs_MEF_upregulated.svg"),
  width = 6,
  height = 6
)
grid.newpage()
grid.draw(venn.plot)
dev.off()


In [ ]:
#Labelling selected genes
label_genes <- c('Eng','Olr1','Axin2','Col10a1')

# Plotting with EnhancedVolcano
EnhancedVolcano(res2_III,
                lab = rownames(res2_III),
                x = 'log2FoldChange',
                y = 'padj',
                #selectLab = label_genes, 
                labSize = 5.0, 
                xlim = c(-8, 8)) 


In [ ]:
suppressPackageStartupMessages({
  library(EnhancedVolcano)
})

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Labelling selected genes
label_genes <- c("Eng", "Olr1", "Axin2", "Col10a1")

# Save as SVG
svg(
  filename = file.path(out_dir, "EnhancedVolcano_res2_III_selectedGenes.svg"),
  width = 8,
  height = 8
)

EnhancedVolcano(
  res2_III,
  lab = rownames(res2_III),
  x = "log2FoldChange",
  y = "padj",
  selectLab = label_genes,   # label only these genes
  labSize = 5.0,
  xlim = c(-8, 8)
)

dev.off()


In [ ]:
write.csv(res2_III,'LMC_III_DEgenes.csv')

In [ ]:
res2_I_df$gene <- rownames(res2_I_df)
res2_II_df$gene <- rownames(res2_II_df)

# Merge the data frames based on gene names
merged_data_lmc <- merge(res2_I_df, res2_II_df, by="gene", suffixes = c("_I", "_II"))

In [ ]:
res2_III_df <- data.frame(res2_III)
res2_III_df$gene <- rownames(res2_III_df)

# Merging the third DataFrame with the merged_data_lmc based on gene names
merged_data_lmc_III <- merge(merged_data_lmc, res2_III_df, by="gene", suffixes = c("", "_III"))



In [ ]:
library(dplyr)

# Assuming df is your dataframe
merged_data_lmc_III <- merged_data_lmc_III %>%
  rename_with(.fn = ~paste0(., "_III"), .cols = tail(names(merged_data_lmc_III), 6))

# Check the result
print(names(merged_data_lmc_III))


In [ ]:
write.csv(merged_data_lmc_III, 'merged_LMC_I_II_III_deseq2results.csv')

In [ ]:
gene_list_I <- na.omit(rownames(res2_I))  
gene_list_II <- na.omit(rownames(res2_II))  
gene_list_III <- na.omit(rownames(res2_III))

write.csv(gene_list_I,'gene_list_I_LMC_2020countsdata.csv')
write.csv(gene_list_II,'gene_list_II_LMC_2020countsdata.csv')
write.csv(gene_list_III,'gene_list_III_LMC_2020countsdata.csv')


# Make a list of the gene lists
gene_lists <- list(gene_list_I, gene_list_II)

# Run the venn diagram function
venn.plot <- venn.diagram(
    x = gene_lists,
    filename = NULL,
    category.names = c("", ""),  # Set as empty strings
    output = TRUE,
    fill = c("red", "blue"),
    alpha = 0.5,
    fontface = "bold",
    fontsize = 20,
    cat.fontface = "bold",
    cat.fontsize = 20,
    lty = "solid",
    lwd = 2,
    label.col = "black",  # Set this to "black" to show numbers
    category.pos = NULL  # Set to NULL to avoid positioning non-existing labels
)
# Plot the diagram
grid.draw(venn.plot)

In [ ]:
gene_list_I <- na.omit(rownames(res2_I))  
gene_list_II <- na.omit(rownames(res2_II))
gene_list_III <- na.omit(rownames(res2_III))

# Make a list of the gene lists
gene_lists <- list(gene_list_I, gene_list_II)

# Generate a Venn diagram without any labels or numbers
venn.plot <- venn.diagram(
    x = gene_lists,
    filename = NULL,
    category.names = c("", ""), 
    output = TRUE,
    fill = c("red", "blue"),
    alpha = 0.5,
    fontface = "bold",
    fontsize = 20,
    cat.fontface = "bold",
    cat.fontsize = 20,
    lty = "solid",
    lwd = 2,
    label.col = NA,  # Set this to NA to remove numbers
    category.pos = NULL  # Set to NULL to avoid positioning non-existing labels
)

# Plot the diagram
grid.draw(venn.plot)

In [ ]:
suppressPackageStartupMessages({
  library(VennDiagram)
  library(grid)
})

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Gene lists
gene_list_I   <- na.omit(rownames(res2_I))
gene_list_II  <- na.omit(rownames(res2_II))
gene_list_III <- na.omit(rownames(res2_III))  # not used yet

# Make a list of the gene lists (I vs II)
gene_lists <- list(gene_list_I, gene_list_II)

# Create Venn diagram object (do NOT write to file yet)
venn.plot <- venn.diagram(
  x = gene_lists,
  filename = NULL,
  category.names = c("", ""),   # no labels
  fill = c("red", "blue"),
  alpha = 0.5,
  fontface = "bold",
  fontsize = 20,
  cat.fontface = "bold",
  cat.fontsize = 20,
  lty = "solid",
  lwd = 2,
  label.col = NA,               # remove numbers
  category.pos = NULL
)

# Save as SVG
svg(
  filename = file.path(out_dir, "Venn_res2_I_vs_II_noLabels.svg"),
  width = 6,
  height = 6
)
grid.newpage()
grid.draw(venn.plot)
dev.off()


In [ ]:
DE_I <- data.frame(rownames(res2_I))
DE_II <- data.frame(rownames(res2_II))
DE_III <- data.frame(rownames(res2_III))

colnames(DE_I)[1] <- "Gene_name"
colnames(DE_II)[1] <- "Gene_name"
colnames(DE_III)[1] <- "Gene_name"

combined_column <- unique(data.frame(c(DE_I$Gene_name,DE_II$Gene_name,DE_III$Gene_name)))

In [ ]:
colnames(combined_column)[1] <- "Gene_name"
combined_column$Gene_name <- toupper(combined_column$Gene_name)

In [ ]:
write.csv(combined_column, "LMC_DE_genes_2020countsdata_total.csv", row.names = FALSE)

In [ ]:
#Setting the organism to mouse here
organism = "org.Mm.eg.db"
library(organism, character.only = TRUE)

In [ ]:
# we want the log2 fold change 
original_gene_list1 <- res2_I$log2FoldChange
names(original_gene_list1) <- rownames(res2_I)
gene_list1<-na.omit(original_gene_list1)
gene_list1 = sort(gene_list1, decreasing = TRUE)

original_gene_list2 <- res2_II$log2FoldChange
names(original_gene_list2) <- rownames(res2_II)
gene_list2<-na.omit(original_gene_list2)
gene_list2 = sort(gene_list2, decreasing = TRUE)

original_gene_list3 <- res2_III$log2FoldChange
names(original_gene_list3) <- rownames(res2_III)
gene_list3<-na.omit(original_gene_list3)
gene_list3 = sort(gene_list3, decreasing = TRUE)

In [ ]:
gene_list_3 <- data.frame(gene_list3)

In [ ]:
colnames(gene_list_3)[1] <- "log2FoldChange"

In [ ]:
write.csv(gene_list_3,'genelist_III_LMC_gsea.csv')

In [ ]:
gse1 <- gseGO(geneList=gene_list1, 
             ont ="BP", 
             keyType = "SYMBOL", 
             nPerm = 10000, 
             minGSSize = 3, 
             maxGSSize = 800, 
             pvalueCutoff = 0.05, 
             verbose = TRUE, 
             OrgDb = organism, 
             pAdjustMethod = "BH")

gse2 <- gseGO(geneList=gene_list2, 
             ont ="BP", 
             keyType = "SYMBOL", 
             nPerm = 10000, 
             minGSSize = 3, 
             maxGSSize = 800, 
             pvalueCutoff = 0.05, 
             verbose = TRUE, 
             OrgDb = organism, 
             pAdjustMethod = "BH")

gse3 <- gseGO(geneList=gene_list3, 
             ont ="BP", 
             keyType = "SYMBOL", 
             nPerm = 10000, 
             minGSSize = 3, 
             maxGSSize = 800, 
             pvalueCutoff = 0.05, 
             verbose = TRUE, 
             OrgDb = organism, 
             pAdjustMethod = "BH")

In [ ]:
gse_termsim1 <- pairwise_termsim(gse1)
gse_termsim2 <- pairwise_termsim(gse2)
gse_termsim3 <- pairwise_termsim(gse3)

In [ ]:
genelist_gse1 <- data.frame(gse1@geneList)

In [ ]:
colnames(genelist_gse1)[1] <- "FoldChange"

In [ ]:
genelist_gse3 <- data.frame(gse3@geneList)
colnames(genelist_gse3)[1] <- "FoldChange"

write.csv(genelist_gse3,'genelist_III_LMC_gsea_results.csv')

In [ ]:
GO_I_lmc <- data.frame(gse1@result)
GO_II_lmc <- data.frame(gse2@result)
GO_III_lmc <- data.frame(gse3@result)

In [ ]:
neutrophil_migration <- data.frame(gse1@geneSets$`GO:1990266`)

In [ ]:
#write.csv(neutrophil_migration,'lmc_I_neutrophilmigration_genes.csv')
write.csv(GO_I_lmc,'lmc_GO_I_GSEA_clusterprofiler.csv')
write.csv(GO_II_lmc,'lmc_GO_II_GSEA_clusterprofiler.csv')
write.csv(GO_III_lmc,'lmc_GO_III_GSEA_clusterprofiler.csv')


In [ ]:
require(DOSE)
dotplot(gse1, showCategory=20, split=".sign") + facet_grid(.~.sign)
dotplot(gse2, showCategory=20, split=".sign") + facet_grid(.~.sign)
dotplot(gse3, showCategory=20, split=".sign") + facet_grid(.~.sign)

In [ ]:
View(as.data.frame(gse1))

In [ ]:
gse1@result$Description <- sub("^(\\w)", "\\U\\1", gse1@result$Description, perl = TRUE)
gse1@result$Description <- gsub("RRNA", "rRNA", gse1@result$Description)

In [ ]:
dotplot(gse1,showCategory=c("Neutrophil migration", "Action potential","Defense response to bacterium","Cellular response to interferon-beta","rRNA processing", "Response to biotic stimulus","Response to interleukin-1"),split=".sign")+ facet_grid(.~.sign)


In [ ]:
suppressPackageStartupMessages({
  library(enrichplot)
  library(ggplot2)
})

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Create the dotplot
p_dot <- dotplot(
  gse1,
  showCategory = c(
    "Neutrophil migration",
    "Action potential",
    "Defense response to bacterium",
    "Cellular response to interferon-beta",
    "rRNA processing",
    "Response to biotic stimulus",
    "Response to interleukin-1"
  ),
  split = ".sign"
) + 
  facet_grid(. ~ .sign)

# Display (optional)
print(p_dot)

# Save as SVG
ggsave(
  filename = file.path(out_dir, "GSEA_dotplot_selectedCategories.svg"),
  plot = p_dot,
  device = "svg",
  width = 10,
  height = 6,
  units = "in"
)


In [ ]:
gse2@result$Description <- sub("^(\\w)", "\\U\\1", gse2@result$Description, perl = TRUE)
#gse2@result$Description <- gsub("RRNA", "rRNA", gse1@result$Description)

In [ ]:
View(as.data.frame(gse2))

In [ ]:
dotplot(gse2,showCategory=c("Cellular response to interferon-beta", "Neutrophil migration","Synaptic vesicle endocytosis","Self proteolysis","Forebrain morphogenesis", "Defense response to other organism","Cellular response to lipopolysaccharide"),split=".sign")+ facet_grid(.~.sign)


In [ ]:
suppressPackageStartupMessages({
  library(enrichplot)
  library(ggplot2)
})

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Create the dotplot
p_dot <- dotplot(
  gse2,
  showCategory = c(
    "Cellular response to interferon-beta",
    "Neutrophil migration",
    "Synaptic vesicle endocytosis",
    "Self proteolysis",
    "Forebrain morphogenesis",
    "Defense response to other organism",
    "Cellular response to lipopolysaccharide"
  ),
  split = ".sign"
) +
  facet_grid(. ~ .sign)

# Display (optional)
print(p_dot)

# Save as SVG
ggsave(
  filename = file.path(out_dir, "GSEA_dotplot_selectedCategories_gse2.svg"),
  plot = p_dot,
  device = "svg",
  width = 10,
  height = 6,
  units = "in"
)


In [ ]:
homeobox <- read.csv('homebox_gene_list.csv',header = FALSE)
colnames(homeobox) <- "Gene_name"


irf <- read.csv('irf_gene_names.txt',header = FALSE)
colnames(irf) <- "Gene_name"

ap1 <- read.csv('ap1_gene_names.txt',header = FALSE)
colnames(ap1) <- "Gene_name"

In [ ]:
irf

In [ ]:
homeobox$Gene_name <- toupper(homeobox$Gene_name)
irf$Gene_name <- toupper(irf$Gene_name)
ap1$Gene_name <- toupper(ap1$Gene_name)


In [ ]:
genes_I <- rownames(res2_I)
genes_II <- rownames(res2_II)

# Find common genes
common_genes_I_II <- toupper(intersect(genes_I, genes_II))

In [ ]:
common_homeo <- intersect(common_genes_I_II,homeobox$Gene_name)
common_irf <- intersect(common_genes_I_II,irf$Gene_name)
common_ap1 <- intersect(common_genes_I_II,ap1$Gene_name)

In [ ]:
write.csv(common_irf,'irf_genes.csv')

In [ ]:
homeobox_interest <- grep("^HOX|^LIM|^POU|^PAX", homeobox$Gene_name, value = TRUE)

In [ ]:
homeobox_interest

In [ ]:
# Load necessary libraries
library(pheatmap)
library(dplyr)

# Ensure the gene names in homeobox are present in the row names of countData
#common_genes <- intersect(homeobox$Gene_name, rownames(countData))
common_genes_hi <- intersect(homeobox_interest, rownames(countData))
# Filter countData to include only rows that have genes present in common_genes
filteredCountData_homeo_interest <- countData[common_genes_hi, ]

# Define the width and height of the plot
# Increase the height to ensure all rows are visible
plot_width <- 20
plot_height <- 30

# Save the heatmap as a PDF
#pdf("Heatmap_of_Homeobox_Gene_Expression.pdf", width = plot_width, height = plot_height)

# Plot the heatmap of the filtered countData with clustering and improved row names visibility
pheatmap(filteredCountData_homeo_interest,
         cluster_rows = TRUE,  # Cluster by rows
         cluster_cols = FALSE,
         annotation_col = as.data.frame(manual_annotation),  # Ensure this is defined or modify as needed
         show_rownames = TRUE,
         show_colnames = FALSE,  # Hiding column names
         scale = "row",
     #    fontsize_row = 4,  # Adjust the font size for row names
         fontsize_col = 10,  # Adjust the font size for column names if shown
         main = "Heatmap of Homeobox Gene Expression(LMC)"
   #      fontsize = 6,  # Increase overall font size
       #  cellheight = 4
        )  # Set the height of each cell (row)

In [ ]:
suppressPackageStartupMessages({
  library(pheatmap)
  library(dplyr)
  library(grid)
})

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Ensure the gene names in homeobox are present in the row names of countData
common_genes_hi <- intersect(homeobox_interest, rownames(countData))

# Filter countData
filteredCountData_homeo_interest <- countData[common_genes_hi, , drop = FALSE]

# Create + capture the heatmap
ph <- pheatmap(
  filteredCountData_homeo_interest,
  cluster_rows = TRUE,
  cluster_cols = FALSE,
  annotation_col = as.data.frame(manual_annotation),
  show_rownames = TRUE,
  show_colnames = FALSE,
  scale = "row",
  fontsize_col = 10,
  main = "Heatmap of Homeobox Gene Expression (LMC)"
)

# Save as SVG (large canvas to avoid clipping)
svg(
  file.path(out_dir, "Heatmap_Homeobox_Gene_Expression_LMC.svg"),
  width = 20,
  height = 30
)

grid.newpage()
grid.draw(ph$gtable)
dev.off()


In [ ]:
# Load necessary libraries
library(pheatmap)
library(dplyr)

# Ensure the gene names in homeobox are present in the row names of countData
#common_genes <- intersect(homeobox$Gene_name, rownames(countData))
common_genes_homeo <- intersect(common_homeo, rownames(countData))
# Filter countData to include only rows that have genes present in common_genes
filteredCountData_homeo <- countData[common_genes_homeo, ]

# Define the width and height of the plot
# Increase the height to ensure all rows are visible
plot_width <- 20
plot_height <- 30

# Save the heatmap as a PDF
#pdf("Heatmap_of_Homeobox_Gene_Expression.pdf", width = plot_width, height = plot_height)

# Plot the heatmap of the filtered countData with clustering and improved row names visibility
pheatmap(filteredCountData_homeo,
         cluster_rows = TRUE,  # Cluster by rows
         cluster_cols = FALSE,
         annotation_col = as.data.frame(manual_annotation),  # Ensure this is defined or modify as needed
         show_rownames = TRUE,
         show_colnames = FALSE,  # Hiding column names
         scale = "row",
     #    fontsize_row = 4,  # Adjust the font size for row names
         fontsize_col = 10,  # Adjust the font size for column names if shown
         main = "Heatmap of Homeobox Gene Expression(LMC)"
   #      fontsize = 6,  # Increase overall font size
       #  cellheight = 4
        )  # Set the height of each cell (row)

In [ ]:
suppressPackageStartupMessages({
  library(pheatmap)
  library(dplyr)
  library(grid)
})

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Subset to common homeobox genes
common_genes_homeo <- intersect(common_homeo, rownames(countData))
filteredCountData_homeo <- countData[common_genes_homeo, , drop = FALSE]

# Create + capture heatmap
ph <- pheatmap(
  filteredCountData_homeo,
  cluster_rows = TRUE,
  cluster_cols = FALSE,
  annotation_col = as.data.frame(manual_annotation),
  show_rownames = TRUE,
  show_colnames = FALSE,
  scale = "row",
  fontsize_col = 10,
  main = "Heatmap of Homeobox Gene Expression (LMC)"
)

# Save as SVG (large canvas so rows/labels don’t get clipped)
svg(
  file.path(out_dir, "Heatmap_Homeobox_Gene_Expression_LMC_common.svg"),
  width = 20,
  height = 30
)
grid.newpage()
grid.draw(ph$gtable)
dev.off()


In [ ]:
suppressPackageStartupMessages({
  library(pheatmap)
  library(dplyr)
  library(grid)
})

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)

# Subset to common homeobox genes
common_genes_homeo <- intersect(common_homeo, rownames(countData))
filteredCountData_homeo <- countData[common_genes_homeo, , drop = FALSE]

# ---- 1) Build group labels: wt-ctrl, wt-lps, mut-ctrl, mut-lps ----
# manual_annotation must have rownames == sample names (colnames of countData)
stopifnot(all(colnames(filteredCountData_homeo) %in% rownames(manual_annotation)))

ann <- as.data.frame(manual_annotation)
ann <- ann[colnames(filteredCountData_homeo), , drop = FALSE]  # align to columns

# Adjust these column names if yours differ (common ones: Genotype + Condition)
group <- paste0(tolower(ann$Genotype), "-", tolower(ann$Condition))
group <- factor(group, levels = c("wt-control", "wt-lps", "mut-control", "mut-lps"))

# ---- 2) Collapse columns by group mean ----
collapsed_mat <- sapply(levels(group), function(g) {
  cols <- which(group == g)
  if (length(cols) == 0) {
    return(rep(NA_real_, nrow(filteredCountData_homeo)))
  }
  rowMeans(filteredCountData_homeo[, cols, drop = FALSE], na.rm = TRUE)
})
collapsed_mat <- as.matrix(collapsed_mat)
rownames(collapsed_mat) <- rownames(filteredCountData_homeo)

# Remove empty groups (all NA) if any group had zero samples
collapsed_mat <- collapsed_mat[, colSums(!is.na(collapsed_mat)) > 0, drop = FALSE]

# ---- 3) Annotation for the 4 collapsed columns ----
collapsed_annot <- data.frame(
  Genotype = sub("^(wt|mut)-.*$", "\\1", colnames(collapsed_mat)),
  Condition = sub("^.*-(control|lps)$", "\\1", colnames(collapsed_mat))
)
collapsed_annot$Genotype <- factor(collapsed_annot$Genotype, levels = c("wt", "mut"))
collapsed_annot$Condition <- factor(collapsed_annot$Condition, levels = c("control", "lps"))
rownames(collapsed_annot) <- colnames(collapsed_mat)

# ---- 4) Plot + capture heatmap ----
ph <- pheatmap(
  collapsed_mat,
  cluster_rows = TRUE,
  cluster_cols = FALSE,                 # keep wt-ctrl, wt-lps, mut-ctrl, mut-lps order
  annotation_col = collapsed_annot,
  show_rownames = TRUE,
  show_colnames = TRUE,
  scale = "row",
  fontsize_col = 12,
  main = "Homeobox Gene Expression (LMC) — collapsed by Genotype+Condition"
)

# ---- 5) Save as SVG ----
svg(file.path(out_dir, "Heatmap_Homeobox_LMC_collapsed_wtctrl_wtlps_mutctrl_mutlps.svg"),
    width = 12, height = 18)
grid.newpage()
grid.draw(ph$gtable)
dev.off()


In [ ]:
common_genes_homeo <- data.frame(common_genes_homeo)
write.csv(common_genes_homeo,'genes_homeo_lmc.csv')

In [ ]:
# Load necessary libraries
library(pheatmap)
library(dplyr)

# Ensure the gene names in homeobox are present in the row names of countData
#common_genes_irf <- intersect(irf$Gene_name, rownames(countData))

common_genes_irf <- intersect(common_irf, rownames(countData))
# Filter countData to include only rows that have genes present in common_genes
filteredCountData_irf <- countData[common_genes_irf, ]

# Define the width and height of the plot
# Increase the height to ensure all rows are visible

# Save the heatmap as a PDF
#pdf("Heatmap_of_Homeobox_Gene_Expression.pdf", width = plot_width, height = plot_height)

# Plot the heatmap of the filtered countData with clustering and improved row names visibility
pheatmap(filteredCountData_irf,
         cluster_rows = TRUE,  # Cluster by rows
         cluster_cols = FALSE,
         annotation_col = as.data.frame(manual_annotation),  # Ensure this is defined or modify as needed
         show_rownames = TRUE,
         show_colnames = FALSE,  # Hiding column names
         scale = "row",
        # fontsize_row = 4,  # Adjust the font size for row names
        # fontsize_col = 10,  # Adjust the font size for column names if shown
         main = "Heatmap of IRF Gene Expression(LMC)",
         fontsize = 6,  # Increase overall font size
         cellheight = 6)
         #)# Set the height of each cell (row)

In [ ]:
suppressPackageStartupMessages({
  library(pheatmap)
  library(dplyr)
  library(grid)
})

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Subset to common IRF genes
common_genes_irf <- intersect(common_irf, rownames(countData))
filteredCountData_irf <- countData[common_genes_irf, , drop = FALSE]

# Create + capture heatmap
ph <- pheatmap(
  filteredCountData_irf,
  cluster_rows = TRUE,
  cluster_cols = FALSE,
  annotation_col = as.data.frame(manual_annotation),
  show_rownames = TRUE,
  show_colnames = FALSE,
  scale = "row",
  main = "Heatmap of IRF Gene Expression (LMC)",
  fontsize = 6,
  cellheight = 6
)

# Save as SVG
svg(
  file.path(out_dir, "Heatmap_IRF_Gene_Expression_LMC.svg"),
  width = 12,
  height = 12
)
grid.newpage()
grid.draw(ph$gtable)
dev.off()


In [ ]:
common_genes_irf <- data.frame(common_genes_irf)

In [ ]:
write.csv(common_genes_irf,'genes_irf_lmc.csv')

In [ ]:
# Load necessary libraries
library(pheatmap)
library(dplyr)

# Ensure the gene names in homeobox are present in the row names of countData
#common_genes <- intersect(homeobox$Gene_name, rownames(countData))

common_genes_ap1 <- intersect(common_ap1, rownames(countData))


# Filter countData to include only rows that have genes present in common_genes
filteredCountData_ap1 <- countData[common_genes_ap1, ]

# Define the width and height of the plot
# Increase the height to ensure all rows are visible
plot_width <- 20
plot_height <- 30

# Save the heatmap as a PDF
#pdf("Heatmap_of_Homeobox_Gene_Expression.pdf", width = plot_width, height = plot_height)

# Plot the heatmap of the filtered countData with clustering and improved row names visibility
pheatmap(filteredCountData_ap1 ,
         cluster_rows = TRUE,  # Cluster by rows
         cluster_cols = FALSE,
         annotation_col = as.data.frame(manual_annotation),  # Ensure this is defined or modify as needed
         show_rownames = TRUE,
         show_colnames = FALSE,  # Hiding column names
         scale = "row",
        # fontsize_row = 4,  # Adjust the font size for row names
         #fontsize_col = 10,  # Adjust the font size for column names if shown
         main = "Heatmap of AP1 Gene Expression(LMC)",
         fontsize = 6,  # Increase overall font size
         cellheight = 6)  
   #      )# Set the height of each cell (row)



In [ ]:
suppressPackageStartupMessages({
  library(pheatmap)
  library(dplyr)
  library(grid)
})

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_lmc"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Subset to common AP1 genes
common_genes_ap1 <- intersect(common_ap1, rownames(countData))
filteredCountData_ap1 <- countData[common_genes_ap1, , drop = FALSE]

# Create + capture heatmap
ph <- pheatmap(
  filteredCountData_ap1,
  cluster_rows = TRUE,
  cluster_cols = FALSE,
  annotation_col = as.data.frame(manual_annotation),
  show_rownames = TRUE,
  show_colnames = FALSE,
  scale = "row",
  main = "Heatmap of AP1 Gene Expression (LMC)",
  fontsize = 6,
  cellheight = 6
)

# Save as SVG (large canvas to avoid clipping)
svg(
  file.path(out_dir, "Heatmap_AP1_Gene_Expression_LMC.svg"),
  width = 20,
  height = 30
)

grid.newpage()
grid.draw(ph$gtable)
dev.off()


In [ ]:
heatmap_result  <- pheatmap(filteredCountData_ap1 ,
         cluster_rows = TRUE,  # Cluster by rows
         cluster_cols = FALSE,
         annotation_col = as.data.frame(manual_annotation),  # Ensure this is defined or modify as needed
         show_rownames = TRUE,
         show_colnames = FALSE,  # Hiding column names
         scale = "row",
        # fontsize_row = 4,  # Adjust the font size for row names
         #fontsize_col = 10,  # Adjust the font size for column names if shown
         main = "Heatmap of AP1 Gene Expression(LMC)",
         fontsize = 6,  # Increase overall font size
         cellheight = 6)  
   #      )# Set the height of each cell (row)

In [ ]:
# Extract the ordered row names from the heatmap result
ordered_rownames <- rownames(filteredCountData_ap1)[heatmap_result$tree_row$order]

ap1_ordered_lmc <- read.csv('output_directory_ucsc/genes_ap1_lmc_tf_family_counts.csv')
ap1_ordered_mef <- read.csv('output_directory_ucsc/genes_ap1_mef_tf_family_counts.csv')


In [ ]:
df <- ap1_ordered_lmc

In [ ]:
filteredCountData_ap1 <- data.frame(filteredCountData_ap1)
filteredCountData_ap1$Gene <- rownames(filteredCountData_ap1)

In [ ]:
results1 <- data.frame(toupper(rownames(res)),res$log2FoldChange)

In [ ]:
colnames(results1)[1] <- "Gene_name"
colnames(results1)[2] <- "Log2FoldChange"

In [ ]:
head(results1)

In [ ]:
normalized_counts_1$Gene_name <- toupper(rownames(normalized_counts_1))

In [ ]:
normalized_counts_il1b <- normalized_counts_1[rownames(normalized_counts_1) == "Il1b", ]


In [ ]:
write.csv(normalized_counts_il1b,'normalized_counts_il1b.csv')

In [ ]:
#Plotting Glimma/Scatter plots for each gene seperately

gene_expression <- normalized_counts_il1b["Il1b",]  # Extracting expression levels for particular gene
gene_expression$Gene_name <- NULL


plot_data <- data.frame(
  Sample = colnames(gene_expression),  
  Expression = as.numeric(gene_expression),  
  Group = groups  
)

gene_name <- "Il1b"

In [ ]:
gene_expression <- normalized_counts_il1b["Il1b",]  # Extracting expression levels for particular gene
gene_expression$Gene_name <- NULL

In [ ]:
# Create Genotype and Condition columns based on the Group column
plot_data$Genotype <- ifelse(grepl("wt", plot_data$Group), "WT", "MUT")
plot_data$Condition <- ifelse(grepl("Control", plot_data$Group), "Control", "LPS")

# Create a new column for the combined labels
plot_data$Group_Combined <- with(plot_data, paste(Genotype, Condition, sep = "+"))

# Define the colors
genotype_colors <- c("WT" = "#fe9388", "MUT" = "#9bca1e")
condition_colors <- c("Control" = "#add8e6", "LPS" = "#e298fe")  # Lighter blue for Control

# Define the colors
genotype_colors <- c("WT" = "#EC3A91", "MUT" = "#71980E")  # Adjusted colors
condition_colors <- c("Control" = "#B0DBFF", "LPS" = "#CF98E2")  # Lightened inside colors

# Create the plot
ggplot(plot_data, aes(x = Group_Combined, y = Expression)) +
  geom_point(aes(fill = Condition, color = Genotype, shape = Genotype), size = 9, stroke = 2) +  # Use shape 21 for filled points with outline
  scale_fill_manual(name = "Condition", values = condition_colors, guide = guide_legend(override.aes = list(shape = 21, size = 9, stroke = 0))) +  # Inner circle colors and remove outline in legend
  scale_color_manual(name = "Genotype", values = genotype_colors, labels = c("WT" = "Wild-type", "MUT" = "Mutant")) +  # Outer circle colors
  scale_shape_manual(name = "Genotype", values = c("WT" = 21, "MUT" = 22), labels = c("WT" = "Wild-type", "MUT" = "Mutant")) +  # Shape 21 for circles, 22 for squares
  scale_x_discrete(limits = c("WT+Control", "WT+LPS", "MUT+Control", "MUT+LPS")) +  # Set the order of the x-axis labels
  labs(
    title = paste("Expression of", gene_name, "Across Groups"),
    x = "",
    y = "Expression Level"
  ) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(size = 14, color = "black"),  # Increase x-axis label size
    axis.text.y = element_text(size = 14, color = "black"),  # Increase y-axis label size
    axis.title.x = element_text(size = 16),  # Increase x-axis title size
    axis.title.y = element_text(size = 16),  # Increase y-axis title size
    axis.ticks = element_line(color = "black"),
    axis.line = element_line(color = "black"),
    panel.background = element_blank(),
    panel.border = element_rect(color = "black", fill = NA),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    legend.position = "right",
    legend.text = element_text(size = 12),  # Increase legend text size
    legend.title = element_text(size = 14),  # Increase legend title size
    legend.key = element_blank()  # Remove legend key outline
  )

In [ ]:
ap1_ordered_lmc

In [ ]:
library(tidyverse)
library(randomForest)
library(factoextra)
library(ggrepel)
library(corrplot)

# Load and prepare data
normalized_counts_1 <- as.data.frame(normalized_counts_1)
normalized_counts_1$Gene_name <- as.character(normalized_counts_1$Gene_name)
tf_binding_data <- ap1_ordered_lmc

expression_long <- normalized_counts_1 %>%
  pivot_longer(cols = -Gene_name, names_to = "sample", values_to = "expression") %>%
  mutate(condition = case_when(
    grepl("CTRL", sample) ~ "Control",
    grepl("LPS", sample) ~ "LPS",
    TRUE ~ NA_character_
  ))

expression_mean <- expression_long %>%
  group_by(Gene_name, condition) %>%
  summarize(mean_expression = mean(expression), .groups = "drop")

expression_fc <- expression_mean %>%
  pivot_wider(names_from = condition, values_from = mean_expression) %>%
  mutate(log2FC = log2(LPS / Control))

combined_data <- expression_fc %>%
  select(Gene_name, log2FC) %>%
  inner_join(tf_binding_data, by = c("Gene_name" = "Gene"))

rf_data <- combined_data %>%
  select(-Gene_name)

# Random Forest model
rf_model <- randomForest(log2FC ~ ., data = rf_data, importance = TRUE)

# Calculate performance metrics
predictions <- predict(rf_model, rf_data)
rmse <- sqrt(mean((rf_data$log2FC - predictions)^2))
rsq <- 1 - sum((rf_data$log2FC - predictions)^2) / sum((rf_data$log2FC - mean(rf_data$log2FC))^2)
mae <- mean(abs(rf_data$log2FC - predictions))

# Print the metrics
cat("Random Forest Model Performance:\n")
cat("RMSE:", rmse, "\n")
cat("R-squared:", rsq, "\n")
cat("MAE:", mae, "\n")

# Perform cross-validation
set.seed(123)
k <- 5  # number of folds
folds <- sample(1:k, nrow(rf_data), replace = TRUE)
cv_results <- data.frame(RMSE = numeric(k), Rsquared = numeric(k), MAE = numeric(k))

for(i in 1:k){
  # Split data
  train_data <- rf_data[folds != i, ]
  test_data <- rf_data[folds == i, ]
  
  # Train model
  rf_cv <- randomForest(log2FC ~ ., data = train_data)
  
  # Make predictions
  predictions <- predict(rf_cv, test_data)
  
  # Calculate metrics
  cv_results$RMSE[i] <- sqrt(mean((test_data$log2FC - predictions)^2))
  cv_results$Rsquared[i] <- 1 - sum((test_data$log2FC - predictions)^2) / sum((test_data$log2FC - mean(test_data$log2FC))^2)
  cv_results$MAE[i] <- mean(abs(test_data$log2FC - predictions))
}

# Print cross-validation results
cat("\nCross-validation results:\n")
print(colMeans(cv_results))

importance_df <- importance(rf_model) %>%
  as.data.frame() %>%
  rownames_to_column("feature") %>%
  arrange(desc(`%IncMSE`))

ggplot(importance_df, aes(x = reorder(feature, `%IncMSE`), y = `%IncMSE`)) +
  geom_bar(stat = "identity") +
  coord_flip() +
  labs(title = "Feature Importance", x = "Features", y = "%IncMSE")

ggsave("feature_importance.png", width = 10, height = 8)

cluster_data <- combined_data %>%
  select(-Gene_name, -log2FC) %>%
  scale()

wss <- sapply(1:10, function(k){kmeans(cluster_data, k, nstart=50,iter.max = 15)$tot.withinss})
plot(1:10, wss, type="b", xlab="Number of Clusters", ylab="Within groups sum of squares")

k <- 5  # Choose the optimal k based on the elbow plot
km_result <- kmeans(cluster_data, centers = k, nstart = 25)
combined_data$cluster <- km_result$cluster
pca_result <- prcomp(cluster_data, scale. = TRUE)
pca_data <- as.data.frame(pca_result$x[,1:2])
pca_data$cluster <- factor(km_result$cluster)
pca_data$Gene_name <- combined_data$Gene_name
pca_data$log2FC <- combined_data$log2FC

ggplot(pca_data, aes(x = PC1, y = PC2, color = cluster, label = Gene_name)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text_repel(size = 3, max.overlaps = 20) +
  theme_minimal() +
  labs(title = "Gene Clusters based on TF Binding Sites",
       x = "Principal Component 1",
       y = "Principal Component 2") +
  scale_color_brewer(palette = "Set1")

ggsave("gene_clusters_plot_all.png", width = 15, height = 12, dpi = 300)

top_genes <- pca_data %>%
  group_by(cluster) %>%
  top_n(5, abs(log2FC)) %>%
  ungroup()

ggplot(pca_data, aes(x = PC1, y = PC2, color = cluster)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text_repel(data = top_genes, aes(label = Gene_name), 
                  size = 3, max.overlaps = 20) +
  theme_minimal() +
  labs(title = "Gene Clusters based on TF Binding Sites (Top 5 genes labeled)",
       x = "Principal Component 1",
       y = "Principal Component 2") +
  scale_color_brewer(palette = "Set1")

ggsave("gene_clusters_plot_top5.png", width = 15, height = 12, dpi = 300)

cluster_summary <- combined_data %>%
  group_by(cluster) %>%
  summarize(
    n_genes = n(),
    mean_log2FC = mean(log2FC),
    across(-c(Gene_name, log2FC), mean),
    .groups = "drop"
  )

print(cluster_summary)

top_tf_sites <- cluster_summary %>%
  select(-n_genes, -mean_log2FC) %>%
  pivot_longer(cols = -cluster, names_to = "TF_site", values_to = "mean_count") %>%
  group_by(cluster) %>%
  top_n(5, mean_count) %>%
  arrange(cluster, desc(mean_count))

print(top_tf_sites)

cor_matrix <- cor(combined_data[, c("log2FC", setdiff(names(combined_data), c("Gene_name", "log2FC", "cluster")))])

png("tf_expression_correlation.png", width = 800, height = 800)
corrplot(cor_matrix, method = "circle", type = "upper", order = "hclust", 
         tl.col = "black", tl.srt = 45, insig = "blank")
dev.off()

# Save results
write.csv(cluster_summary, "cluster_summary.csv", row.names = FALSE)
write.csv(top_tf_sites, "top_tf_sites.csv", row.names = FALSE)
write.csv(importance_df, "feature_importance.csv", row.names = FALSE)
write.csv(data.frame(RMSE = rmse, R_squared = rsq, MAE = mae), "rf_model_metrics.csv", row.names = FALSE)
write.csv(cv_results, "rf_cv_results.csv", row.names = FALSE)

In [ ]:
normalized_counts_1

In [ ]:
countdata45 <- normalized_counts_1

In [ ]:
rownames(countdata45) <- toupper(rownames(countdata45))

write.csv(countdata45,'normalised_LMC_countdata.csv')

In [ ]:
###########################
## Load packages
###########################
suppressPackageStartupMessages({
  library(pheatmap)
  library(stringr)
})

###########################
## 1️⃣ Derive group information from column names
###########################
# Example assumption:
# Column names look like: "WT_Control_1", "WT_LPS_2", "MUT_Control_1", "MUT_LPS_3"
# Adjust the patterns below to match your real naming scheme.

colnames(expr_collapsed)

# Derive Genotype and Condition automatically
annotation_col <- data.frame(
  Genotype = ifelse(grepl("WT", colnames(expr_collapsed), ignore.case = TRUE), "WT",
                    ifelse(grepl("MUT", colnames(expr_collapsed), ignore.case = TRUE), "MUT", "Unknown")),
  Condition = ifelse(grepl("LPS", colnames(expr_collapsed), ignore.case = TRUE), "LPS",
                     ifelse(grepl("CTRL|CONTROL", colnames(expr_collapsed), ignore.case = TRUE), "Control", "Unknown"))
)

# Add row names (must match colnames of expr_collapsed)
rownames(annotation_col) <- colnames(expr_collapsed)

###########################
## 2️⃣ Define colors for annotation
###########################
ann_colors <- list(
  Genotype = c(WT = "#66c2a5", MUT = "#fc8d62", Unknown = "grey80"),
  Condition = c(Control = "#8da0cb", LPS = "#e78ac3", Unknown = "grey90")
)

###########################
## 3️⃣ Nicer heatmap (publication-quality)
###########################
hm_colors <- colorRampPalette(c("#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026"))(200)

pheatmap(
  expr_collapsed,
  cluster_rows = TRUE,
  cluster_cols = TRUE,
  color = hm_colors,
  annotation_col = annotation_col,
  annotation_colors = ann_colors,
  show_rownames = TRUE,
  show_colnames = TRUE,
  fontsize_row = 7,
  fontsize_col = 9,
  angle_col = 45,
  cellheight = 10,
  cellwidth  = 20,
  border_color = NA,
  main = "Heatmap of HOX/PAX/POU Gene Expression",
  treeheight_row = 10,
  treeheight_col = 15
)


In [ ]:
str(expr_collapsed)
summary(as.vector(as.matrix(expr_collapsed)))
sum(is.na(as.matrix(expr_collapsed)))
sum(is.finite(as.matrix(expr_collapsed)), na.rm=TRUE)


In [ ]:
###########################
## Load packages
###########################
suppressPackageStartupMessages({
  library(pheatmap)
  library(stringr)
})

###########################
## 1️⃣ Derive group information from column names
###########################
# Example assumption:
# Column names look like: "WT_Control_1", "WT_LPS_2", "MUT_Control_1", "MUT_LPS_3"
# Adjust the patterns below to match your real naming scheme.

colnames(expr_collapsed)

# Derive Genotype and Condition automatically
annotation_col <- data.frame(
  Genotype = ifelse(grepl("WT", colnames(expr_collapsed), ignore.case = TRUE), "WT",
                    ifelse(grepl("MUT", colnames(expr_collapsed), ignore.case = TRUE), "MUT", "Unknown")),
  Condition = ifelse(grepl("LPS", colnames(expr_collapsed), ignore.case = TRUE), "LPS",
                     ifelse(grepl("CTRL|CONTROL", colnames(expr_collapsed), ignore.case = TRUE), "Control", "Unknown"))
)

# Add row names (must match colnames of expr_collapsed)
rownames(annotation_col) <- colnames(expr_collapsed)

###########################
## 2️⃣ Define colors for annotation
###########################
ann_colors <- list(
  Genotype = c(WT = "#66c2a5", MUT = "#fc8d62", Unknown = "grey80"),
  Condition = c(Control = "#8da0cb", LPS = "#e78ac3", Unknown = "grey90")
)

###########################
## 3️⃣ Nicer heatmap (publication-quality)
###########################
hm_colors <- colorRampPalette(c("#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026"))(200)

pheatmap(
  expr_collapsed,
  cluster_rows = TRUE,
  cluster_cols = FALSE,
  color = hm_colors,
  annotation_col = annotation_col,
  annotation_colors = ann_colors,
  show_rownames = TRUE,
  show_colnames = TRUE,
  fontsize_row = 7,
  fontsize_col = 9,
  angle_col = 45,
  cellheight = 10,
  cellwidth  = 20,
  border_color = NA,
  main = "Heatmap of HOX/PAX/POU Gene Expression",
  treeheight_row = 10,
  treeheight_col = 15
)


In [ ]:
###########################
## Load packages
###########################
suppressPackageStartupMessages({
  library(pheatmap)
  library(stringr)
  library(grid)
})

###########################
## 1️⃣ Derive group information from column names
###########################
annotation_col <- data.frame(
  Genotype = ifelse(grepl("WT", colnames(expr_collapsed), ignore.case = TRUE), "WT",
                    ifelse(grepl("MUT", colnames(expr_collapsed), ignore.case = TRUE), "MUT", "Unknown")),
  Condition = ifelse(grepl("LPS", colnames(expr_collapsed), ignore.case = TRUE), "LPS",
                     ifelse(grepl("CTRL|CONTROL", colnames(expr_collapsed), ignore.case = TRUE), "Control", "Unknown"))
)

rownames(annotation_col) <- colnames(expr_collapsed)

###########################
## 2️⃣ Define annotation colors
###########################
ann_colors <- list(
  Genotype  = c(WT = "#66c2a5", MUT = "#fc8d62", Unknown = "grey80"),
  Condition = c(Control = "#8da0cb", LPS = "#e78ac3", Unknown = "grey90")
)

###########################
## 3️⃣ Heatmap colors
###########################
hm_colors <- colorRampPalette(
  c("#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026")
)(200)

###########################
## 4️⃣ Create heatmap (capture object)
###########################
ph <- pheatmap(
  expr_collapsed,
  cluster_rows = TRUE,
  cluster_cols = FALSE,
  color = hm_colors,
  annotation_col = annotation_col,
  annotation_colors = ann_colors,
  show_rownames = TRUE,
  show_colnames = TRUE,
  fontsize_row = 7,
  fontsize_col = 9,
  angle_col = 45,
  cellheight = 10,
  cellwidth  = 20,
  border_color = NA,
  main = "Heatmap of HOX / PAX / POU Gene Expression",
  treeheight_row = 10,
  treeheight_col = 15
)

###########################
## 5️⃣ Save as SVG (CORRECT WAY)
###########################
svg(
  filename = "Heatmap_HOX_PAX_POU_expression.svg",
  width = 18,   # increase if many columns
  height = 22   # increase if many genes
)

grid.newpage()
grid.draw(ph$gtable)

dev.off()


In [ ]:
###########################
## Load packages
###########################
suppressPackageStartupMessages({
  library(pheatmap)
  library(stringr)
  library(dplyr)
})

###########################
## 1️⃣ Reorder columns in expr_collapsed
## Desired final order:
##   WT_Control, WT_LPS, Sp3KO_Control, Sp3KO_LPS
###########################

desired_order <- c("WT_Control", "WT_LPS", "Sp3KO_Control", "Sp3KO_LPS")

expr_collapsed <- expr_collapsed[, intersect(desired_order, colnames(expr_collapsed)), drop = FALSE]

###########################
## 2️⃣ Filter out zero-variance genes
##    (genes that don't change across the 4 averaged groups)
###########################

row_var <- apply(expr_collapsed, 1, var)

keep_rows <- which(!is.na(row_var) & row_var > 0)

if (length(keep_rows) == 0) {
  stop("All genes are flat across conditions after collapsing. No heatmap can be drawn with row scaling.")
}

expr_collapsed_var <- expr_collapsed[keep_rows, , drop = FALSE]

###########################
## 3️⃣ Manual safe row scaling (z-score per gene)
###########################

scale_rows_safe <- function(mat) {
  t(apply(mat, 1, function(x) {
    mu  <- mean(x, na.rm = TRUE)
    sdv <- sd(x,  na.rm = TRUE)
    if (is.na(sdv) || sdv == 0) {
      # shouldn't happen now because we filtered variance>0,
      # but guard anyway:
      rep(0, length(x))
    } else {
      (x - mu) / sdv
    }
  }))
}

expr_scaled <- scale_rows_safe(expr_collapsed_var)

###########################
## 4️⃣ Build annotation table AFTER reordering
###########################

annotation_col <- data.frame(
  Genotype = dplyr::case_when(
    grepl("^WT",    colnames(expr_scaled), ignore.case = TRUE)    ~ "WT",
    grepl("^SP3KO", colnames(expr_scaled), ignore.case = TRUE)    ~ "Sp3KO",
    TRUE                                                           ~ "Unknown"
  ),
  Condition = dplyr::case_when(
    grepl("LPS",          colnames(expr_scaled), ignore.case = TRUE)    ~ "LPS",
    grepl("CTRL|CONTROL", colnames(expr_scaled), ignore.case = TRUE)    ~ "Control",
    TRUE                                                                 ~ "Unknown"
  ),
  stringsAsFactors = FALSE
)

rownames(annotation_col) <- colnames(expr_scaled)

###########################
## 5️⃣ Define colors for annotation
###########################

ann_colors <- list(
  Genotype = c(
    WT     = "#66c2a5",
    Sp3KO  = "#fc8d62",
    Unknown = "grey80"
  ),
  Condition = c(
    Control = "#8da0cb",
    LPS     = "#e78ac3",
    Unknown = "grey90"
  )
)

###########################
## 6️⃣ Heatmap palette
###########################

hm_colors <- colorRampPalette(
  c("#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026")
)(200)

###########################
## 7️⃣ Decide if we can cluster rows
##    (pheatmap::hclust() will die if there's only 1 gene)
###########################

cluster_rows_flag <- nrow(expr_scaled) >= 2

###########################
## 8️⃣ Plot heatmap using pre-scaled matrix
###########################

pheatmap(
  expr_scaled,
  cluster_rows = cluster_rows_flag,
  cluster_cols = FALSE,     # keep WT_Control → WT_LPS → Sp3KO_Control → Sp3KO_LPS
  scale = "none",           # we already z-scored rows
  color = hm_colors,
  annotation_col = annotation_col,
  annotation_colors = ann_colors,
  show_rownames = TRUE,
  show_colnames = TRUE,
  fontsize_row = 7,
  fontsize_col = 9,
  angle_col = 45,
  cellheight = 10,
  cellwidth  = 20,
  border_color = NA,
  main = "Heatmap of HOX/PAX/POU Gene Expression",
  treeheight_row = 10,
  treeheight_col = 15
)


In [ ]:
###########################
## Load packages
###########################
suppressPackageStartupMessages({
  library(pheatmap)
  library(stringr)
  library(dplyr)
})

###########################
## 1️⃣ Reorder columns in expr_collapsed
## Desired final order:
##   WT_Control, WT_LPS, Sp3KO_Control, Sp3KO_LPS
###########################

desired_order <- c("WT_Control", "WT_LPS", "Sp3KO_Control", "Sp3KO_LPS")

# Keep only the columns that exist, in that order
expr_collapsed <- expr_collapsed[, intersect(desired_order, colnames(expr_collapsed)), drop = FALSE]

###########################
## 2️⃣ Build annotation_col AFTER reordering
###########################

annotation_col <- data.frame(
  Genotype = dplyr::case_when(
    grepl("^WT",    colnames(expr_collapsed), ignore.case = TRUE)    ~ "WT",
    grepl("^SP3KO", colnames(expr_collapsed), ignore.case = TRUE)    ~ "Sp3KO",
    TRUE                                                             ~ "Unknown"
  ),
  Condition = dplyr::case_when(
    grepl("LPS",      colnames(expr_collapsed), ignore.case = TRUE)  ~ "LPS",
    grepl("CTRL|CONTROL", colnames(expr_collapsed), ignore.case = TRUE) ~ "Control",
    TRUE                                                             ~ "Unknown"
  ),
  stringsAsFactors = FALSE
)

rownames(annotation_col) <- colnames(expr_collapsed)

###########################
## 3️⃣ Define colors for annotation
###########################
ann_colors <- list(
  Genotype = c(
    WT     = "#66c2a5",
    Sp3KO  = "#fc8d62",
    Unknown = "grey80"
  ),
  Condition = c(
    Control = "#8da0cb",
    LPS     = "#e78ac3",
    Unknown = "grey90"
  )
)

###########################
## 4️⃣ Heatmap color palette
###########################
hm_colors <- colorRampPalette(
  c("#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026")
)(200)

###########################
## 5️⃣ Plot heatmap
###########################
pheatmap(
  expr_collapsed,
  cluster_rows = TRUE,      # cluster genes
  cluster_cols = FALSE,     # KEEP our manual condition order
  scale = "row",
  color = hm_colors,
  annotation_col = annotation_col,
  annotation_colors = ann_colors,
  show_rownames = TRUE,
  show_colnames = TRUE,
  fontsize_row = 7,
  fontsize_col = 9,
  angle_col = 45,
  cellheight = 10,
  cellwidth  = 20,
  border_color = NA,
  main = "Heatmap of HOX/PAX/POU Gene Expression",
  treeheight_row = 10,
  treeheight_col = 15
)


In [ ]:
###########################
## Load packages
###########################
suppressPackageStartupMessages({
  library(pheatmap)
})

###########################
## 1️⃣ Annotation for groups
###########################
annotation_col <- data.frame(
  Genotype = c("WT", "WT", "MUT", "MUT"),
  Condition = c("Control", "LPS", "Control", "LPS")
)
rownames(annotation_col) <- colnames(expr_collapsed)

###########################
## 2️⃣ Define annotation colors
###########################
ann_colors <- list(
  Genotype = c(WT = "#66c2a5", MUT = "#fc8d62"),
  Condition = c(Control = "#8da0cb", LPS = "#e78ac3")
)

###########################
## 3️⃣ Nice color palette
###########################
hm_colors <- colorRampPalette(c("#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026"))(200)

###########################
## 4️⃣ Save as JPEG
###########################
jpeg("HOX_PAX_POU_LMC_heatmap.jpg", width = 2000, height = 2000, res = 300)

pheatmap(
  expr_collapsed,
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  color = hm_colors,
  annotation_col = annotation_col,
  annotation_colors = ann_colors,
  show_rownames = TRUE,
  show_colnames = TRUE,
  fontsize_row = 7,
  fontsize_col = 10,
  angle_col = 45,
  cellheight = 10,
  cellwidth  = 25,
  border_color = NA,
  legend = TRUE,
  main = "HOX / PAX / POU Gene Expression (LMC)",
  treeheight_row = 10
)

dev.off()
cat("✅ Heatmap saved as 'HOX_PAX_POU_LMC_heatmap.jpg' in your working directory.\n")


In [ ]:
###########################
## Load packages
###########################
suppressPackageStartupMessages({
  library(pheatmap)
})

###########################
## 1️⃣ Annotation for groups
###########################
annotation_col <- data.frame(
  Genotype = c("WT", "WT", "MUT", "MUT"),
  Condition = c("Control", "LPS", "Control", "LPS")
)
rownames(annotation_col) <- colnames(expr_collapsed)

###########################
## 2️⃣ Define annotation colors
###########################
ann_colors <- list(
  Genotype = c(WT = "#66c2a5", MUT = "#fc8d62"),
  Condition = c(Control = "#8da0cb", LPS = "#e78ac3")
)

###########################
## 3️⃣ Nice color palette
###########################
hm_colors <- colorRampPalette(c("#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026"))(200)

###########################
## 4️⃣ Save as JPEG
###########################
jpeg("HOX_PAX_POU_LMC_heatmap.jpg", width = 2000, height = 2000, res = 300)

pheatmap(
  expr_collapsed,
  cluster_rows = TRUE,
  cluster_cols = FALSE,
  scale = "row",
  color = hm_colors,
  annotation_col = annotation_col,
  annotation_colors = ann_colors,
  show_rownames = TRUE,
  show_colnames = TRUE,
  fontsize_row = 7,
  fontsize_col = 10,
  angle_col = 45,
  cellheight = 10,
  cellwidth  = 25,
  border_color = NA,
  legend = TRUE,
  main = "HOX / PAX / POU Gene Expression (LMC)",
  treeheight_row = 10
)

dev.off()
cat("✅ Heatmap saved as 'HOX_PAX_POU_LMC_heatmap.jpg' in your working directory.\n")


In [ ]:
getwd()


In [ ]:
combined_data

In [ ]:
combined_data1 <- combined_data

In [ ]:
combined_data1$cluster <- NULL

In [ ]:
combined_data1

In [ ]:
tf_binding_data

In [ ]:
combined_data

In [ ]:
library(ggrepel)

In [ ]:
filteredCountData_ap1 <- data.frame(filteredCountData_ap1)

In [ ]:
filteredCountData_ap1$Gene <- rownames(filteredCountData_ap1)

In [ ]:
install.packages('randomForest')

In [ ]:
library(ComplexUpset)
library(ggplot2)
library(tidyverse)
library(patchwork)

In [ ]:
data <- ap1_ordered_mef

In [ ]:
data

In [ ]:
library(tidyverse)
library(UpSetR)

# Convert data to binary for UpSet plot (presence/absence of transcription factors)
data_binary <- data %>% 
  mutate(across(-Gene, ~ as.integer(. > 0)))

# Create the UpSet plot
upset_plot <- upset(
  data_binary,
  sets = c("SOX", "Homeobox", "STAT", "KLF", "ZNF", "SP", "IRF", "NFKB", "SP1"),
  order.by = "freq",
  mainbar.y.label = "Intersection Size",
  sets.x.label = "Set Size",
  matrix.color = "black",
  point.size = 3,
  line.size = 0
)

# Display the UpSet plot
print(upset_plot)

# Function to get genes for a specific intersection
get_genes_for_intersection <- function(data, intersection) {
  data %>%
    filter(if_all(all_of(intersection), ~ . > 0)) %>%
    filter(if_all(setdiff(names(.)[-1], intersection), ~ . == 0)) %>%
    pull(Gene)
}

# Get all intersections from the plot
all_intersections <- names(upset_plot$New_data)

# Create a table of up to 5 genes from each intersection
intersection_table <- tibble(
  Intersection = all_intersections,
  Genes = map_chr(all_intersections, ~{
    intersection <- strsplit(.x, "&")[[1]]
    genes <- get_genes_for_intersection(data_binary, intersection)
    paste(head(genes, 5), collapse = ", ")
  })
)

# Display the table
print(intersection_table)

# If you want to save the table to a CSV file:
# write.csv(intersection_table, "intersection_genes.csv", row.names = FALSE)

In [ ]:
common_genes_ap1 <- data.frame(common_genes_ap1)
write.csv(common_genes_ap1,'genes_ap1_lmc.csv')

In [ ]:
gse3@result$Description <- sub("^(\\w)", "\\U\\1", gse3@result$Description, perl = TRUE)

In [ ]:
View(as.data.frame(gse3))

In [ ]:
dotplot(gse3,showCategory=c("Innate immune response", "Response to external biotic stimulus","Embryonic skeletal system morphogenesis","","","","","",""),split=".sign")+ facet_grid(.~.sign)

In [ ]:
dotplot(gse3,showCategory=c("Embryonic limb morphogenesis","Spindle checkpoint signaling","Negative regulation of chromosome segregation","Adaptive immune response","Innate immune response","Response to external biotic stimulus","biological process involved in interspecies interaction between organisms","Plasma membrane invagination","Hemopoiesis","Embryonic skeletal joint morphogenesis","Heterotypic cell-cell adhesion"),split=".sign")+ facet_grid(.~.sign)

In [ ]:
dotplot(gse3,showCategory=c("Spindle checkpoint signaling","Peripheral nervous system neuron differentiation","Negative regulation of chromosome segregation","Innate immune response","Response to external biotic stimulus","biological process involved in interspecies interaction between organisms","Plasma membrane invagination","Hemopoiesis","Embryonic morphogenesis","Medium-chain fatty acid metabolic process"),split=".sign")+ facet_grid(.~.sign)

In [ ]:
library(ggplot2)
library(clusterProfiler)

# Assuming 'gse3' is your data frame or result object for GSEA analysis
# Convert to a data frame to manipulate
gse3_df <- as.data.frame(gse3)

# Change the name in the data frame
#gse3_df$Description <- gsub("Embryonic limb morphogenesis", "Embryonic morphogenesis", gse3_df$Description)

# Convert back to the original class if necessary, here we assume it's enrichResult
gse3@result <- gse3_df

# Now create the dotplot with the modified category names
dotplot(gse3, 
        showCategory = c("Embryonic morphogenesis",
                         "Spindle checkpoint signaling",
                         "Negative regulation of chromosome segregation",
                         "Adaptive immune response",
                         "Innate immune response",
                         "Response to external biotic stimulus",
                         "biological process involved in interspecies interaction between organisms",
                         "Plasma membrane invagination",
                         "Hemopoiesis",
                         "Embryonic skeletal joint morphogenesis",
                         "Heterotypic cell-cell adhesion"),
        split = ".sign") + 
  facet_grid(. ~ .sign)


In [ ]:
suppressPackageStartupMessages({
  library(ggplot2)
  library(clusterProfiler)
})

###########################
## Make sure gse3 has the edited @result
###########################
gse3_df <- as.data.frame(gse3)
gse3@result <- gse3_df

###########################
## Build the plot
###########################
p <- dotplot(
  gse3,
  showCategory = c(
    "Embryonic morphogenesis",
    "Spindle checkpoint signaling",
    "Negative regulation of chromosome segregation",
    "Adaptive immune response",
    "Innate immune response",
    "Response to external biotic stimulus",
    "biological process involved in interspecies interaction between organisms",
    "Plasma membrane invagination",
    "Hemopoiesis",
    "Embryonic skeletal joint morphogenesis",
    "Heterotypic cell-cell adhesion"
  ),
  split = ".sign"
) + facet_grid(. ~ .sign)

###########################
## Save as SVG
###########################
out_file <- "gse3_dotplot_split_sign_lmc_feb16_2026.svg"
ggsave(
  filename = out_file,
  plot = p,
  device = "svg",
  width = 10,
  height = 6,
  units = "in"
)

cat("Saved SVG to:", out_file, "\n")

In [ ]:
#Have to check this plot

ridgeplot(gse1) +
  labs(x = "enrichment distribution") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),  # Rotate and reduce text size
    axis.text.y = element_text(size = 8),  # Reduce text size for y-axis
    strip.text = element_text(size = 8)  # Reduce facet label size if applicable
  ) +
  coord_fixed(ratio = 0.5)  # Change aspect ratio

ridgeplot(gse2) +
  labs(x = "enrichment distribution") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),  # Rotate and reduce text size
    axis.text.y = element_text(size = 8),  # Reduce text size for y-axis
    strip.text = element_text(size = 8)  # Reduce facet label size if applicable
  ) +
  coord_fixed(ratio = 0.5)  # Change aspect ratio

ridgeplot(gse3) +
  labs(x = "enrichment distribution") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),  # Rotate and reduce text size
    axis.text.y = element_text(size = 8),  # Reduce text size for y-axis
    strip.text = element_text(size = 8)  # Reduce facet label size if applicable
  ) +
  coord_fixed(ratio = 0.5)  # Change aspect ratio


In [ ]:
gse1_results <- gse1

In [ ]:
# categorySize can be either 'pvalue' or 'geneNum'
cnetplot(gse1, categorySize="pvalue", foldChange=gene_list1, showCategory = 3)

# categorySize can be either 'pvalue' or 'geneNum'
cnetplot(gse2, categorySize="pvalue", foldChange=gene_list2, showCategory = 3)

# categorySize can be either 'pvalue' or 'geneNum'
cnetplot(gse3, categorySize="pvalue", foldChange=gene_list3, showCategory = 3)

In [ ]:
p1 <- gseaplot(gse1, geneSetID = 1, by = "runningScore", title = gse1$Description[1])
p2 <- gseaplot(gse1, geneSetID = 1, by = "preranked", title = gse1$Description[1])
#p3 <- gseaplot(gse1, geneSetID = 1, title = gse1$Description[1])
cowplot::plot_grid(p1, p2, ncol=1, labels=LETTERS[1:2])

In [ ]:
p1 <- gseaplot(gse2, geneSetID = 1, by = "runningScore", title = gse2$Description[1])
p2 <- gseaplot(gse2, geneSetID = 1, by = "preranked", title = gse2$Description[1])
cowplot::plot_grid(p1, p2, ncol=1, labels=LETTERS[1:2])

In [ ]:
gseaplot2(gse1, geneSetID = 1, title = gse1$Description[1])

In [ ]:
p1 <- gseaplot(gse3, geneSetID = 1, by = "runningScore", title = gse3$Description[1])
p2 <- gseaplot(gse3, geneSetID = 1, by = "preranked", title = gse3$Description[1])
cowplot::plot_grid(p1, p2, ncol=1, labels=LETTERS[1:2])

In [ ]:
gseaplot2(gse2, geneSetID = 1, title = gse2$Description[1])

In [ ]:
gseaplot2(gse3, geneSetID = 1, title = gse3$Description[1])

In [ ]:
enriched_terms_df_gse1 <- gse1@result

# View the first few rows of the data frame
head(enriched_terms_df_gse1)

In [ ]:
enriched_terms_df_gse2 <- gse2@result

# View the first few rows of the data frame
head(enriched_terms_df_gse2)

In [ ]:
#browseKEGG(kk, 'hsa04110')

In [ ]:
gene_list_I <- na.omit(rownames(res2_I_up))  
gene_list_II <- na.omit(rownames(res2_II_up))  

# Make a list of the gene lists
gene_lists <- list(gene_list_I, gene_list_II)

# Run the venn diagram function
venn.plot <- venn.diagram(
    x = gene_lists,
    filename = NULL,
    category.names = c("", ""),  # Set as empty strings
    output = TRUE,
    fill = c("red", "blue"),
    alpha = 0.5,
    fontface = "bold",
    fontsize = 20,
    cat.fontface = "bold",
    cat.fontsize = 20,
    lty = "solid",
    lwd = 2,
    label.col = "NA",  # Set this to "black" to show numbers
    category.pos = NULL  # Set to NULL to avoid positioning non-existing labels
)
# Plot the diagram
grid.draw(venn.plot)

In [ ]:
gene_list_I <- na.omit(rownames(res2_I_up))  
gene_list_II <- na.omit(rownames(res2_II_up))  

# Make a list of the gene lists
gene_lists <- list(gene_list_I, gene_list_II)

# Run the venn diagram function
venn.plot <- venn.diagram(
    x = gene_lists,
    filename = NULL,
    category.names = c("list1", "list2"),  # Set as empty strings
    output = TRUE,
    fill = c("red", "blue"),
    alpha = 0.5,
    fontface = "bold",
    fontsize = 20,
    cat.fontface = "bold",
    cat.fontsize = 20,
    lty = "solid",
    lwd = 2,
    label.col = "black",  # Set this to "black" to show numbers
    category.pos = NULL  # Set to NULL to avoid positioning non-existing labels
)
# Plot the diagram
grid.draw(venn.plot)

In [ ]:
gene_list_I <- na.omit(rownames(res2_I_down))  
gene_list_II <- na.omit(rownames(res2_II_down))  

# Make a list of the gene lists
gene_lists <- list(gene_list_I, gene_list_II)

# Run the venn diagram function
venn.plot <- venn.diagram(
    x = gene_lists,
    filename = NULL,
    category.names = c("", ""),  # Set as empty strings
    output = TRUE,
    fill = c("red", "blue"),
    alpha = 0.5,
    fontface = "bold",
    fontsize = 20,
    cat.fontface = "bold",
    cat.fontsize = 20,
    lty = "solid",
    lwd = 2,
    label.col = "black",  # Set this to "black" to show numbers
    category.pos = NULL  # Set to NULL to avoid positioning non-existing labels
)
# Plot the diagram
grid.draw(venn.plot)

In [ ]:
gene_list_I <- na.omit(rownames(res2_I_down))  
gene_list_II <- na.omit(rownames(res2_II_down))  

# Make a list of the gene lists
gene_lists <- list(gene_list_I, gene_list_II)

# Run the venn diagram function
venn.plot <- venn.diagram(
    x = gene_lists,
    filename = NULL,
    category.names = c("list1", "list2"),  # Set as empty strings
    output = TRUE,
    fill = c("red", "blue"),
    alpha = 0.5,
    fontface = "bold",
    fontsize = 20,
    cat.fontface = "bold",
    cat.fontsize = 20,
    lty = "solid",
    lwd = 2,
    label.col = "NA",  # Set this to "black" to show numbers
    category.pos = NULL  # Set to NULL to avoid positioning non-existing labels
)
# Plot the diagram
grid.draw(venn.plot)

In [ ]:
#https://yulab-smu.top/biomedical-knowledge-mining-book/clusterprofiler-go.html

In [ ]:

rownames(res2_III) <- toupper(rownames(res2_III))


In [ ]:
filtered_data_III <- res2_III[row.names(res2_III) %in% nf_kb, ]

In [ ]:
chk <- data.frame(rownames(res2_III))
nf_kb <- data.frame(nf_kb)
colnames(chk)[1] <- "Gene_name"
colnames(nf_kb)[1] <- "Gene_name"
chk2 <- merge(nf_kb,chk,by = "Gene_name")

In [ ]:
filtered_data_III <- res2_III[row.names(res2_III) %in% chk2, ]

In [ ]:
filtered_data_III

In [ ]:
write.csv(rownames(res2_III),"chk2.csv")

In [ ]:
write.csv(nf_kb,"chk3.csv")

In [ ]:
chk2

In [ ]:

nf_kb_genes <- intersect(rownames(res2_III), nf_kb$Gene_name)
filteredresdata <- res2_III[nf_kb_genes, ]

filtered_rows <- grep("^(?!GM|MIR|ENSMUS|RPL|RIK).*$", rownames(filteredresdata), perl = TRUE)


In [ ]:
filteredresdata

In [ ]:
# Sort the filtered genes by the absolute value of log2FoldChange in descending order
sorted_genes <- filteredresdata[order(-abs(filteredresdata$log2FoldChange)), ]

# Select the top 50 genes based on this sorting
top_50_genes <- head(sorted_genes, 50)

In [ ]:
top_50_genes

In [ ]:
# Extract log2FoldChange values for the top 50 genes
log2fc_values_top_50 <- top_50_genes$log2FoldChange

# Convert to a matrix for pheatmap
log2fc_matrix_top_50 <- matrix(log2fc_values_top_50, nrow = length(log2fc_values_top_50), ncol = 1, 
                               dimnames = list(rownames(top_50_genes), "Log2FoldChange"))
genes_to_keep <- !grepl("^(Gm|Mir|ENSMUS|Rpl.*|.*Rik|mt-|Malat1)", rownames(log2fc_matrix_top_50))

# Filter both the log2FoldChange values and update row names accordingly
log2fc_values_filtered <- log2fc_values_top_50[genes_to_keep]
row_names_filtered <- rownames(log2fc_matrix_top_50)[genes_to_keep]

# Update the matrix for pheatmap using filtered genes
log2fc_matrix_filtered <- matrix(log2fc_values_filtered, nrow = length(log2fc_values_filtered), ncol = 1, 
                                 dimnames = list(row_names_filtered, "Log2FoldChange"))



color_palette_fc <- colorRampPalette(c("blue", "white", "red"))(100)
# Plot the heatmap with a slimmer appearance by adjusting cellwidth and cellheight
pheatmap(log2fc_matrix_filtered,
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         show_rownames = TRUE,
         color = color_palette_fc,
         cellwidth = 50)  # Adjust the cell height if necessary

In [ ]:
# Assuming countdata1 is your data frame or matrix
rownames(countData1) <- toupper(rownames(countData1))

In [ ]:
countData1$Gene_Name <- toupper(countData1$Gene_Name)

In [ ]:
length(top_50_genes)

In [ ]:
countData3 <- countData1


In [ ]:
countData3$Gene_Name <- toupper(countData3$Gene_Name)
rownames(countData3) <- countData3$Gene_Name

In [ ]:
top_50_expression_data <- countData3[rownames(top_50_genes), ]

In [ ]:
top_50_expression_data$Gene_Name <- NULL

In [ ]:
# Create the heatmap with the filtered dataframe
pheatmap(top_50_expression_data, 
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         scale = "row",
         annotation_col = as.data.frame(manual_annotation), 
         show_rownames = TRUE)

In [ ]:
library(readxl)
lmc_ipa <- read_xlsx('IPA/LMC_MEF_IPA_files/lmc_ipa_upstreamregulators.xlsx')
lmc_ipa <- lmc_ipa[3:nrow(lmc_ipa),1:17]

In [ ]:
annotation_col = as.data.frame(manual_annotation)

In [ ]:
annotation_col

In [ ]:
df <- lmc_ipa

In [ ]:
# Load required libraries
library(dplyr)
library(pheatmap)

# Assuming your dataframe is called 'df'
# Step 1: Calculate LPS response difference between WT and mutant
df_differences <- df %>%
  mutate(across(2:17, as.numeric)) %>%  # Ensure all expression columns are numeric
  rowwise() %>%
  mutate(
    wt_control = mean(c_across(2:5)),
    wt_lps = mean(c_across(6:9)),
    mut_control = mean(c_across(10:13)),
    mut_lps = mean(c_across(14:17)),
    wt_lps_response = wt_lps - wt_control,
    mut_lps_response = mut_lps - mut_control,
    lps_response_difference = abs(wt_lps_response - mut_lps_response)
  )

# Step 2: Sort dataframe based on LPS response difference
df_sorted <- df_differences %>%
  arrange(desc(lps_response_difference))

# Step 3: Select top 150 rows
df_top150 <- df_sorted %>%
  select(1:17) %>%
  head(100)

# Step 4: Convert expression values to integers
df_top150[, 2:17] <- lapply(df_top150[, 2:17], function(x) as.integer(round(x)))

# Prepare data for heatmap
heatmap_data <- as.matrix(df_top150[, -1])
rownames(heatmap_data) <- df_top150[[1]]

# Define the color scheme: purple for low, white for middle, subdued green for high
color_scheme <- colorRampPalette(c("white", "#8E44AD", "#2ECC71"))(100)

# Set the size of the plot
options(repr.plot.width=10, repr.plot.height=30)

# Create heatmap
pheatmap(heatmap_data,
         scale = "none",  # No scaling of rows
         cluster_rows = FALSE,
         cluster_cols = FALSE,
         show_rownames = TRUE,
         show_colnames = TRUE,
         color = color_scheme,
         main = "Top 150 Genes with Differential LPS Response: WT vs Mutant",
         fontsize_row = 6,  # Reduced font size for row names
         fontsize_col = 8,  # Slightly larger font size for column names
         na_col = "grey",
         cellheight = 8,  # Increased cell height for better visibility
         annotation_col = as.data.frame(manual_annotation))

In [ ]:
# Load required libraries
library(dplyr)
library(pheatmap)
library(repr)

heatmap_data <- as.matrix(df_top150[, -1])
rownames(heatmap_data) <- df_top150[[1]]

# Define the color scheme: purple for low, white for middle, subdued green for high
color_scheme <- colorRampPalette(c("#8E44AD", "white", "#2ECC71"))(100)

# Set the size of the plot
options(repr.plot.width=10, repr.plot.height=30)

# Create heatmap
pheatmap(heatmap_data,
         scale = "none",  # No scaling of rows
         cluster_rows = FALSE,
         cluster_cols = FALSE,
         show_rownames = TRUE,
         show_colnames = TRUE,
         color = color_scheme,
         main = "Top 150 Genes with Differential LPS Response: WT vs Mutant",
         fontsize_row = 8,  # Reduced font size for row names
         fontsize_col = 11,  # Slightly larger font size for column names
         na_col = "grey",
         cellheight = 7,  # Increased cell height for better visibility
         annotation_col = as.data.frame(manual_annotation))

In [ ]:
gene_data <- read.csv("gene_list_MEF_LMC_comparison_heatmap.csv",header = FALSE)
colnames(gene_data)[1] <- "Gene_name"

# Display the first few rows to understand its structure
head(gene_data)

In [ ]:
gene_data

In [ ]:
gene_data$Gene_name <- toupper(gene_data$Gene_name)

In [ ]:
rownames(countData) <- toupper(rownames(countData))
cellcycle_genes <- intersect(rownames(countData), gene_data$Gene_name)
filteredCountData_cc <- countData[cellcycle_genes, ]

In [ ]:
filteredCountData_cc

In [ ]:
# Create the heatmap with the filtered dataframe
pheatmap(filteredCountData_cc, 
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         scale = "row",
         annotation_col = as.data.frame(manual_annotation), 
         show_rownames = TRUE)

In [ ]:
library(pheatmap)
library(tidyverse)

# Assuming filteredCountData_cc is your matrix
# Select only the columns we want to focus on
selected_data <- filteredCountData_cc[, c(1:4, 9:12)]

# Create a new annotation dataframe for the selected columns
manual_annotation <- data.frame(
  Condition = c(rep("Wild-type", 4), rep("Mutant", 4))
)
rownames(manual_annotation) <- colnames(selected_data)

annotation_colors <- list(
  Condition = c("Wild-type" = "blue", "Mutant" = "red")
)

# Create the heatmap with the selected data
pheatmap(selected_data, 
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         scale = "row",
         annotation_col = manual_annotation, 
         annotation_colors = annotation_colors, 
         show_rownames = TRUE,
         main = "Heatmap of Gene Expression (Wild-type vs Mutant)",
         fontsize_row = 8,
         fontsize_col = 8, 
         cellheight = 10)

In [ ]:
library(ggplot2)

# Convert the data to long format for ggplot2
expression_long <- reshape2::melt(filteredCountData_cc)

# Create a violin plot
ggplot(expression_long, aes(x = Var2, y = value, fill = Var1)) + 
  geom_violin() + 
  theme(axis.text.x = element_text(angle = 90, hjust = 1)) +
  labs(x = "Samples", y = "Expression Value", title = "Expression Data Violin Plot")

In [ ]:
head(filteredCountData_cc)

In [ ]:
filteredCountData_cc$Gene <- rownames(filteredCountData_cc)
filteredCountData_cc <- as.data.frame(filteredCountData_cc)

In [ ]:
filteredCountData_cc 

In [ ]:
# Melt the data for ggplot
melted_data <- melt(filteredCountData_cc, id.vars = "Gene", variable.name = "Sample", value.name = "Expression")


In [ ]:
chk_mef_lmc <- read.csv('gene_list_MEF_LMC_comparison_heatmap.csv',header = FALSE)

In [ ]:
gene_list3_df <- data.frame(log2FoldChange = gene_list3)
gene_list3_df$gene <- rownames(gene_list3_df)

# Display the head of gene_list3_df
print(head(gene_list3_df))

# Load the chk_mef_lmc CSV file
chk_mef_lmc <- read.csv('gene_list_MEF_LMC_comparison_heatmap.csv', header = FALSE, stringsAsFactors = FALSE)
colnames(chk_mef_lmc) <- c("gene")

# Display chk_mef_lmc to verify its contents
print(head(chk_mef_lmc))

# Subset gene_list3_df to include only those rows where the gene names match the column values in chk_mef_lmc
matched_genes <- gene_list3_df[gene_list3_df$gene %in% chk_mef_lmc$gene, , drop = FALSE]

# Display the result
print(head(matched_genes))


In [ ]:
matched_genes_vector <- setNames(matched_genes$log2FoldChange, matched_genes$gene)

# Load required packages
library(clusterProfiler)

# Example organism database (use the appropriate one for your organism)
# BiocManager::install("org.Hs.eg.db")  # Uncomment if not installed
# library(org.Hs.eg.db)

# Perform Gene Set Enrichment Analysis
chk_cc <- gseGO(geneList=matched_genes_vector, 
                ont ="BP", 
                keyType = "SYMBOL", 
                minGSSize = 2, 
                maxGSSize = 1000, 
                pvalueCutoff = 0.6, 
                verbose = TRUE, 
                OrgDb = organism,  # Replace with the appropriate OrgDb for your organism
                pAdjustMethod = "BH")

# Display the results
print(chk_cc)

In [ ]:
require(DOSE)
dotplot(chk_cc, showCategory=9, split=".sign") + facet_grid(.~.sign)

In [ ]:
csv1 <- read.csv('filter_MEF_ordered.csv',header = TRUE)

In [ ]:
gene_name_mef_comparison <- csv1$X

In [ ]:
# Remove columns that contain 'LPS' (case-insensitive) in their names
columns_without_LPS <- !grepl("LPS", colnames(countData), ignore.case = TRUE)

# Select only those columns that do not contain 'LPS'
countData_onlycontrol <- countData[, columns_without_LPS]


In [ ]:
countData_onlycontrol

In [ ]:
# Define the gene names for comparison from csv1$X
gene_name_mef_comparison <- csv1$X

# Find the intersection of rownames and gene names in csv1$X
chk <- intersect(rownames(countData_onlycontrol), gene_name_mef_comparison)

# Filter the data to keep only the genes in the intersection
filteredCountData <- countData_onlycontrol[chk, ]

# Reorder the filtered data based on the order in csv1$X
ordered_filteredCountData <- filteredCountData[match(gene_name_mef_comparison, rownames(filteredCountData)), ]

# Ensure that the filtered data isn't empty
if (nrow(ordered_filteredCountData) == 0) {
  stop("The filtered matrix is empty. Please check your data.")
}

# Generate heatmap if the matrix is valid
heatmap_result <- pheatmap(ordered_filteredCountData, 
                           scale = "row",
                           cluster_cols = FALSE,
                           cluster_rows = FALSE,  # Do not cluster rows to keep the order same as csv1$X
                           show_rownames = TRUE,
                           show_colnames = TRUE,
                           main = "Heatmap of DE genes in LMC (Wildtype versus Sp3 knockout, no LPS)",
                           fontsize_row = 5,   # Adjust row label font size
                           fontsize_col = 8,   # Adjust column label font size
                           width = 20,         # Width of the plot in inches
                           height = 40)        # Height of the plot in inches
    # Height of the plot in inches


In [ ]:
# Load necessary libraries
library(pheatmap)
library(gridExtra)

# Part 1: Generate heatmap for `csv1` and extract clustering information

# Convert the relevant columns of `csv1` to a matrix (excluding the gene names in column `X`)
csv1_matrix <- as.matrix(csv1[, -1])

# Set the rownames of the matrix to the gene names in the first column of `csv1`
rownames(csv1_matrix) <- csv1$X

# Define column annotations for the wild-type and knockout in MEF
annotation_col_mef <- data.frame(
  Condition = factor(c(rep("Wild-Type", 5), rep("Sp3 Knockout", 3)))
)
rownames(annotation_col_mef) <- colnames(csv1_matrix)

# Save heatmap for csv1 data as a JPEG with larger dimensions
jpeg("heatmap_csv1_chk.jpeg", width = 4000, height = 10000, res = 300)  # Increase width and height
heatmap_csv1 <- pheatmap(csv1_matrix, 
                         scale = "row",           # Scale by rows
                         cluster_cols = FALSE,    # Do not cluster columns
                         cluster_rows = TRUE,     # Cluster rows
                         show_rownames = TRUE, 
                         show_colnames = FALSE,
                         annotation_col = annotation_col_mef,  # Add column annotations for MEF
                         main = "Heatmap of DE genes in MEF",
                         fontsize_row = 2,        # Adjust row label font size
                         fontsize_col = 8,        # Adjust column label font size
                         width = 20,
                         height = 100, cellheight = 2)
dev.off()

# Extract the order of the rows after clustering
clustered_order <- heatmap_csv1$tree_row$order
clustered_genes <- rownames(csv1_matrix)[clustered_order]

# Part 2: Generate and save the heatmap for `countData_onlycontrol` using the same order as csv1's clustering

# Define the gene names for comparison from csv1$X
gene_name_mef_comparison <- csv1$X

# Find the intersection of rownames and gene names in csv1$X
chk <- intersect(rownames(countData_onlycontrol), gene_name_mef_comparison)

# Filter the data to keep only the genes in the intersection
filteredCountData <- countData_onlycontrol[chk, ]

# Reorder the filtered data based on the order in csv1 after clustering
ordered_filteredCountData <- filteredCountData[match(clustered_genes, rownames(filteredCountData)), ]

# Define column annotations for the wild-type and knockout in LMC
annotation_col_lmc <- data.frame(
  Condition = factor(c(rep("Wild-Type", 4), rep("Sp3 Knockout", 4)))
)
rownames(annotation_col_lmc) <- colnames(ordered_filteredCountData)

# Save heatmap for `countData_onlycontrol` as a JPEG with larger dimensions
jpeg("heatmap_control.jpeg", width = 4000, height = 10000, res = 300)  # Increase width and height
heatmap_control <- pheatmap(ordered_filteredCountData, 
                            scale = "row",
                            cluster_cols = FALSE,
                            cluster_rows = FALSE,  # Do not cluster rows to keep the same order as csv1's clustering
                            show_rownames = TRUE,
                            show_colnames = FALSE,
                            annotation_col = annotation_col_lmc,  # Add column annotations for LMC
                            main = "Heatmap of DE genes in LMC",
                            fontsize_row = 2,      # Adjust row label font size
                            fontsize_col = 8,      # Adjust column label font size
                            width = 20,            # Width of the plot in inches
                            height = 40, cellheight = 2)  # Height of the plot in inches
dev.off()

# Part 4: Save heatmaps for only csv2 genes (with slimmer rows)

# Filter `csv1_matrix` and `ordered_filteredCountData` to keep only the genes from csv2$V1
csv2_genes_in_csv1 <- intersect(rownames(csv1_matrix), csv2$V1)
csv2_genes_in_control <- intersect(rownames(ordered_filteredCountData), csv2$V1)

# Filter both datasets
filtered_csv1_for_csv2 <- csv1_matrix[csv2_genes_in_csv1, ]
filtered_control_for_csv2 <- ordered_filteredCountData[csv2_genes_in_control, ]

# Ensure the data is not empty
if (nrow(filtered_csv1_for_csv2) == 0 || nrow(filtered_control_for_csv2) == 0) {
  stop("No common genes between csv1/csv2 or countData_onlycontrol/csv2")
}

# Save heatmap for `csv1` with only `csv2$V1` genes as a JPEG with larger dimensions
jpeg("heatmap_csv1_csv2.jpeg", width = 4000, height = 4000, res = 300)  # Increase width and height
heatmap_csv1_csv2 <- pheatmap(filtered_csv1_for_csv2, 
                              scale = "row",           # Scale by rows
                              cluster_cols = FALSE,    # Do not cluster columns
                              cluster_rows = FALSE,     # Cluster rows
                              annotation_col = annotation_col_mef,  # Add column annotations for MEF
                              show_rownames = TRUE, 
                              show_colnames = FALSE,
                              main = "Heatmap of DE genes in MEF (focussed set of genes)",
                              fontsize_row = 4,        # Smaller font size for slimmer rows
                              fontsize_col = 8,        # Column label size
                              width = 20,
                              height = 20,
                              cellheight = 6,          # Slimmer row height
                              cellwidth = 20)          # Fixed cell width
dev.off()

# Save heatmap for `countData_onlycontrol` with only `csv2$V1` genes as a JPEG with larger dimensions
jpeg("heatmap_control_csv2.jpeg", width = 4000, height = 4000, res = 300)  # Increase width and height
heatmap_control_csv2 <- pheatmap(filtered_control_for_csv2, 
                                 scale = "row",
                                 cluster_cols = FALSE,
                                 cluster_rows = FALSE,  # Do not cluster rows
                                 annotation_col = annotation_col_lmc,  # Add column annotations for LMC
                                 show_rownames = TRUE,
                                 show_colnames = FALSE,
                                 main = "Heatmap of DE genes in LMC (focussed set of genes)",
                                 fontsize_row = 4,      # Smaller font size for slimmer rows
                                 fontsize_col = 8,      # Column label size
                                 width = 20,            # Width of the plot in inches
                                 height = 20,
                                 cellheight = 6,        # Slimmer row height
                                 cellwidth = 20)        # Fixed cell width
dev.off()

# All the heatmaps are now saved as JPEG files



In [ ]:
head(rownames(countData2))
head(colnames(countData2))  #

In [ ]:
write.csv(countData2,'LMC_May13_2025_analysis_counts.csv')

In [ ]:
countData2

In [ ]:
# Load necessary libraries
library(pheatmap)
library(grid)

# Part 1: Generate heatmap for `csv1` and extract clustering information

# Convert the relevant columns of `csv1` to a matrix (excluding the gene names in column `X`)
csv1_matrix <- as.matrix(csv1[, -1])

# Set the rownames of the matrix to the gene names in the first column of `csv1`
rownames(csv1_matrix) <- csv1$X

# Define column annotations for the wild-type and knockout in MEF
annotation_col_mef <- data.frame(
  Condition = factor(c(rep("Wild-Type", 5), rep("Sp3 Knockout", 3)))
)
rownames(annotation_col_mef) <- colnames(csv1_matrix)

# Generate the heatmap (without saving initially)
heatmap_csv1 <- pheatmap(csv1_matrix, 
                         scale = "row",           # Scale by rows
                         cluster_cols = FALSE,    # Do not cluster columns
                         cluster_rows = TRUE,     # Cluster rows
                         show_rownames = TRUE, 
                         show_colnames = FALSE,
                         annotation_col = annotation_col_mef,  # Add column annotations for MEF
                         main = "Heatmap of DE genes in MEF",
                         fontsize_row = 2,        # Adjust row label font size
                         fontsize_col = 8,        # Adjust column label font size
                         width = 20,
                         height = 100, cellheight = 2)

# Now save the heatmap to a PNG file
png("heatmap_csv1_chk.png", width = 4000, height = 10000, res = 300)  # Use PNG instead of JPEG
grid.draw(heatmap_csv1$gtable)  # Draw the pheatmap object explicitly
dev.off()  # Close the PNG device to save the image


In [ ]:
csv2 <- read.csv('gene_list_MEF_LMC_comparison_heatmap.csv',header = FALSE)

In [ ]:
csv2$V1 <- toupper(csv2$V1)

In [ ]:
# Load necessary libraries
library(pheatmap)
library(gridExtra)

# Part 1: Generate heatmap for `csv1` and extract clustering information

# Convert the relevant columns of `csv1` to a matrix (excluding the gene names in column `X`)
csv1_matrix <- as.matrix(csv1[, -1])

# Set the rownames of the matrix to the gene names in the first column of `csv1`
rownames(csv1_matrix) <- csv1$X

# Generate heatmap for csv1 data
heatmap_csv1 <- pheatmap(csv1_matrix, 
                         scale = "row",           # Scale by rows
                         cluster_cols = FALSE,    # Do not cluster columns
                         cluster_rows = TRUE,     # Cluster rows
                         show_rownames = TRUE, 
                         show_colnames = TRUE,
                         main = "Heatmap of DE genes in MEF",
                         fontsize_row = 5,        # Adjust row label font size
                         fontsize_col = 8,        # Adjust column label font size
                         width = 20,
                         height = 40)

# Extract the order of the rows after clustering
clustered_order <- heatmap_csv1$tree_row$order
clustered_genes <- rownames(csv1_matrix)[clustered_order]

# Part 2: Generate the heatmap for `countData_onlycontrol` using the same order as csv1's clustering

# Define the gene names for comparison from csv1$X
gene_name_mef_comparison <- csv1$X

# Find the intersection of rownames and gene names in csv1$X
chk <- intersect(rownames(countData_onlycontrol), gene_name_mef_comparison)

# Filter the data to keep only the genes in the intersection
filteredCountData <- countData_onlycontrol[chk, ]

# Reorder the filtered data based on the order in csv1 after clustering
ordered_filteredCountData <- filteredCountData[match(clustered_genes, rownames(filteredCountData)), ]

# Ensure that the filtered data isn't empty
if (nrow(ordered_filteredCountData) == 0) {
  stop("The filtered matrix is empty. Please check your data.")
}

# Generate heatmap for countData_onlycontrol data with the same order as csv1's clustering
heatmap_control <- pheatmap(ordered_filteredCountData, 
                            scale = "row",
                            cluster_cols = FALSE,
                            cluster_rows = FALSE,  # Do not cluster rows to keep the same order as csv1's clustering
                            show_rownames = TRUE,
                            show_colnames = TRUE,
                            main = "Heatmap of DE genes in LMC",
                            fontsize_row = 5,      # Adjust row label font size
                            fontsize_col = 8,      # Adjust column label font size
                            width = 20,            # Width of the plot in inches
                            height = 40)           # Height of the plot in inches

# Part 3: Combine the two heatmaps side by side

# Use grid.arrange to arrange both heatmaps side by side
grid.arrange(heatmap_csv1[[4]], heatmap_control[[4]], ncol = 2)

# Part 4: Generate heatmaps for only csv2 genes (with slimmer rows)

# Filter `csv1_matrix` and `ordered_filteredCountData` to keep only the genes from csv2$V1
csv2_genes_in_csv1 <- intersect(rownames(csv1_matrix), csv2$V1)
csv2_genes_in_control <- intersect(rownames(ordered_filteredCountData), csv2$V1)

# Filter both datasets
filtered_csv1_for_csv2 <- csv1_matrix[csv2_genes_in_csv1, ]
filtered_control_for_csv2 <- ordered_filteredCountData[csv2_genes_in_control, ]

# Ensure the data is not empty
if (nrow(filtered_csv1_for_csv2) == 0 || nrow(filtered_control_for_csv2) == 0) {
  stop("No common genes between csv1/csv2 or countData_onlycontrol/csv2")
}

# Generate heatmap for `csv1` with only `csv2$V1` genes (with slimmer row height)
heatmap_csv1_csv2 <- pheatmap(filtered_csv1_for_csv2, 
                              scale = "row",           # Scale by rows
                              cluster_cols = FALSE,    # Do not cluster columns
                              cluster_rows = FALSE,     # Cluster rows
                              show_rownames = TRUE, 
                              show_colnames = TRUE,
                              main = "Heatmap of DE genes in MEF (focussed set of genes)",
                              fontsize_row = 4,        # Smaller font size for slimmer rows
                              fontsize_col = 8,        # Column label size
                              width = 20,
                              height = 20,cellheight = 6,cellwidth = 20
                             )             # Slimmer height for rows

# Generate heatmap for `countData_onlycontrol` with only `csv2$V1` genes (with slimmer row height)
heatmap_control_csv2 <- pheatmap(filtered_control_for_csv2, 
                                 scale = "row",
                                 cluster_cols = FALSE,
                                 cluster_rows = FALSE,  # Cluster rows
                                 show_rownames = TRUE,
                                 show_colnames = TRUE,
                                 main = "Heatmap of DE genes in LMC (focussed set of genes)",
                                 fontsize_row = 4,      # Smaller font size for slimmer rows
                                 fontsize_col = 8,      # Column label size
                                 width = 20,            # Width of the plot in inches
                                 height = 20,cellheight = 6,cellwidth = 20)           # Slimmer height for rows

# Part 5: Combine the heatmaps for csv2 genes side by side (slimmer rows)

# Use grid.arrange to arrange both heatmaps side by side for csv2 genes
grid.arrange(heatmap_csv1_csv2[[4]], heatmap_control_csv2[[4]], ncol = 2)



In [ ]:
# Check for non-numeric columns in your filtered data and handle them
filtered_countdata_matrix <- as.matrix(filtered_countdata)

# Convert all data to numeric (if there are any non-numeric values, they will be converted to NAs)
filtered_countdata_matrix <- apply(filtered_countdata_matrix, 2, function(x) as.numeric(as.character(x)))

# Remove rows with all NAs (to ensure the matrix has meaningful data)
filtered_countdata_matrix <- filtered_countdata_matrix[rowSums(is.na(filtered_countdata_matrix)) != ncol(filtered_countdata_matrix), ]

# Define the pattern to remove unwanted genes (as before)
pattern <- "^(Gm|MIR|ENSMUS|Rpl)|Rik$|^[A-Z]+$|\\."

# Use grep to find rows matching the pattern and invert the selection
rows_to_keep <- grep(pattern, rownames(filtered_countdata_matrix), invert = TRUE, ignore.case = TRUE)

# Further filter to keep only the rows that match the criteria
filtered_matrix <- filtered_countdata_matrix[rows_to_keep, ]

# Ensure that the rownames are intact after converting to numeric
rownames(filtered_matrix) <- rownames(filtered_countdata_matrix)[rows_to_keep]

# Check if the matrix is empty or contains only NA values after filtering
if (nrow(filtered_matrix) == 0 || all(is.na(filtered_matrix))) {
  stop("The filtered matrix is empty or contains only NA values. Please check the filtering steps.")
}

# Generate the heatmap only if the matrix is valid (non-empty, no all-NA rows)
heatmap_result <- pheatmap(filtered_matrix, 
                           scale = "row",
                           cluster_cols = FALSE, # Do not cluster columns
                           show_rownames = TRUE,
                           show_colnames = TRUE,
                           main = "Heatmap of DE genes in LMC (Wildtype versus Sp3 knockout, no LPS)",
                           fontsize_row = 5,   # Adjust row label font size
                           fontsize_col = 8,   # Adjust column label font size
                           width = 20,         # Width of the plot in inches
                           height = 40)        # Height of the plot in inches



In [ ]:
chk

In [ ]:
sessionInfo()

In [ ]:
gene_data <- read.csv("gene_list_MEF_LMC_comparison_heatmap.csv",header = FALSE)
colnames(gene_data)[1] <- "Gene_name"

# Display the first few rows to understand its structure
head(gene_data)